In [ ]:
# SH01 integrated editor — paste this entire file into ONE Kaggle Python cell.
# Enable GPU + Internet in Kaggle. Models download on the first run; no paid API key.
SOURCE_PATH = '/kaggle/input/datasets/vivanjaiswal/video1/military_seth_raw_cut1.mp4'
BRIEF = ('Create a polished, informative vertical short. Select a coherent story and a strong opening; '
         'preserve the speaker’s meaning and voice. Add brief grounded narration in separate intro, bridge '
         'and outro sections. Use relevant supplementary images or video, readable motion graphics, '
         'purposeful camera moves and transitions, restrained SFX and a quiet music bed.')
TARGET_SECONDS = 35
NARRATION = 'intro_bridge_outro'   # 'intro_outro', 'intro', 'auto', or 'off'
REQUIRED_FEATURES = ['voiceover','sfx','music','camera_motion','transitions','graphics','supplementary_media','captions']
REFINEMENT_ROUNDS = 2
OUTPUT_SIZE = (1080, 1920)       # Use (720,1280) for a faster draft.
LOCAL_ASSET_LIBRARY = ''        # Optional annotated media directory; uses Commons otherwise.
MUSIC_PATH = ''                 # Optional local track; empty => original procedural ambient bed.
PRIVATE_ENV_FILE = '/kaggle/working/sh01-fastapi-server/.env'  # Read if present; never overwrite.
TEXT_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
VISION_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VOICE_NAME = 'en_US-ljspeech-high'  # English public voice; change for another language.
ASR_DEVICE = 'cpu'             # 'cuda' is optional; director and vision use the GPU either way.
KEEP_LOCAL_API = False         # True leaves the loopback API running after this job.
TIMEOUT_SECONDS = 7200

# -------------------- automated setup and execution --------------------
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request, venv, zipfile
from pathlib import Path

# Embedded, inspectable Python source from the matching repository patch.
BUNDLE = (
'P)h>@6aWAK2mk;8VqLIRB-*_W0049{000dD003cWUt@1=aBp*EE^v9pTWxdOxDo#D8UF{C9Z$#`QC6JXwbfLeDz<XbMSiL5T<_|7I4}uHI8!7K'
'BxS3v|9f`<kOW^O+c#~h4>3tB7K_EcJiBB})BH%%Y<w{_c${Z6HAv(ee$L5lNS$BeTZ2UzU2u}II5L8m8{?_r#j7Mv89wTprn$ct@OWjoZjfg='
'r><)-80Q(VisDSnO!xQp_ll?KCh}NZ-Y-eI4B2gUzan0F6UK`Lix%a5oR*iImX{e@RX2Igd?ivsGGtxIa{;$%B-<pwwip;k8x&BC1fUL*uyW{P'
'I!DFup7DtiU;Mqgx|lq8G{K$KXZ1Nv6WA{3QO*CzLONFud&Uo!=V!%Txgue%)F^dn1iPih+++E4!g&nK@;q`i&9^5rUQC?$EV~Fv2TkfN>57z_'
'o2n(|^w*rGSx+q~7C;6)!7ApAgcothyp7_xqCO)on1`FfXeqF`sgZy_;+%VxKGKBTN3I;qNP5>Z)+A&;$*8+zi)A$d)sykfaDUH$pT^Q73GrCA'
'DL9@^PNrjbJ~=y?%-z}f@#p<LDo)3k3Id_Np<(>qNH|%n$k2#lFyS@jhGXO@4f4>qU((1>y2)bLDcVMgDmO?NM^p;YTFDp{*4&@P-koKm&}g#2'
'dsNtNe?haEcxIW}Nl14yVT!oI38k%B38w*j7@3D+%Vy8W$RZtGo8WvuGM~o!Ihi+N+40mpK0m!Uznr)~oPTgXPtQ((rM4pY>B*P*_`_`CUQJIY'
'=U=Ye%gOQi*~ul)t;mD*Z4aH@ic^%BMr+H|cy5^rqM7dorfq`^xa@p#H95PQo}V>%zx!@5*u}ncE;4`nhoe6Z)B^i^XmaV$@R^rgrx`d!bp<|r'
'a|5eAJ}ELcgd^!VA@`PecMDK@<O>S~ekska3$iPCgnZzcJkR|-p9Y2-#rKwNIDY{n=gKHKg?JZ<=+lR%0bW?<!OzaY$~o|_4n7SJPKO7V|G@J@'
'n#D>`*g9Kj(|2^Ug}JUTX@AGF>4giX;(ng|j2bd+T%0Aq0K&6Iel?j*zMqdz-OH=<`S|-u_at2y4q<>n1V?eeEps|VfsB6}XW#=uKD<x@(uah4'
'8TCO|`Yz-k3&6{=l15Dt`ywo*^%!bF?9B2M4ArNjZ_!YEnh_R`I(Ce1iJZMS3fKbDjc?YM6mp{`@CI{9AMKyOu6(wD7$-N@D@C(kA=~@GTTM2W'
'rr_R(ED{mS_{jkg9n`O`QjC$zx3~<ja4A&UIBQT`LyNcp@Li|!UCk!Mt|EGG0coE`D4J#Wj6j<(e5_V{GBqNVZTsnx9DVzp-CxoNr5F2p`0nVY'
'A^MuZGH@|$c$(Ni{ZZX+A3?UEIt+vjaNAS2EHZ#&pJjBF!cNt>;a))3JWKCcwzN#ww4bCf!ZQsaQbKvkK-Lb4AVJC<4_j$X88E20wBuU{@=D+;'
'$>1*`n=mG5)Q@JRLUSlY2G6qer=ZW`7AVwJe+8-zT|ZwX7G93bUs8zD_7m<@(HJxYLpv)o3;xJGVI?~1<Bbc7vnm2Xq=vKD&6XShNn;#;A`ADE'
'Ji`<0b&YdW;I%1j6Y@s~QAax2&PpmqdWtD+H;?WhV#2mNW+@$ajMQwdYihX|#jM`q-n_?n3mJ8^C3|VqwiX4X1z+);7LiWCi83a@Dd02BYGlrb'
'L*o&I1k7v-B+~**8_IM6DFPz&aU-JxNIeHBgyV8o>O^b|PhMN3k0g}1vE~%0OIIypNe)$zsXK}Cgig!n@jYhCZYyltowPELT3DGO*R^}ksobuq'
'v`o5kl7MP$?LMC}-c@0_L-k=C-{pyr%r{^eFj0XgVU=3yKn=k9@JF7;9gE;BjNgsAykXcnoywg7#O7fJ+ALYgCt9jvkUL-xkw+~#Cb5j3jX+Xx'
'V&H-UimFf_*g+TNV_`u<bs!L+lv@rfVG2iuLFq=?nDVP9$y_qNHkBq#1V)u#G)CP#K^*V9Bol-sPwlx~OA?YnJIVv@rUJuRP0-=xSsVnwTmUKw'
'rvS!x!bdqn^2MBDRIpIf9VjEj)Noyo98!HEbrqVF)v)AJWUu-pIM;SGgei|Enq&yL46&Dt0OnI8vP0b+fGcNJKy?Eav2+QVqe<*7Wn|ShZX17A'
'3aMD#RF{?PD1+GNu)e0;N)jhlu}6_0H4Kn(0LvD_<e{M><mS1wKy{z^zL=*}gqQft^<fg!aAAlutoA&gr5@qF-LeY$Kw-1B2)85WHwApo4DfJA'
'a_CyFpftG)5M1j|Ze9fjaC4PhxIyV9);He>lmw{(0(Ns3O47cfm6Vzh%IK}bq*6t8v0^>yYkFywx5Ji86bnn^9+}{%nDD@=;mHDaClFlugiEg<'
'$>OQuEW2fGQ#Se^1*8^!*;Cr9XP4&_HZAjso;PYK+AMQh=MfZ|P)=PcqP!OYb%a0(Q#xSm!C(NUQo44jo}SV0_mS@EFF#}Hxhryq6;GOw%v+*a'
'ztC@b2FW0_B`^fldd~<c!Pe4NM4@zQn&vV1G-&O1qxv+(;*^U>lr5`-3@bq$aLXb6EoBE0c@&jHwbfQO(}rrkSa@6Y(5M2pHmS^omF3n)7ZeAD'
'e12^#;qd}eFwPzH2l|N87YwK?eb57}q*C8d^$_%7nE;XL^-Wt&H|HS?1?FQ_)xWv;@?kbT7Ul4V@#W-A&v^63?sHkqzd3wkKgp)iVO0!bA2&UG'
'l<Ih@ZL932xy3=lg2kmGY?Wc0F2M3AZ`oI8mH3JvT!!DOHbEc(*HVPP%2pX<EHHdi+acnLN<OJ{w`zS^JIwvTB?)sywOaK$U{TrawSXT1+%>|d'
';2R+$;OttdUjVjR@1*3Z+;&Tgvk<GFM<H7=w3Bj7%`BW<LqvtMl$Jg+tZw_qpo87;KK}C_f=Jo)kvtSQ7<F_h?0SCMeq{6kOaa@aDtQXz(h8UW'
'j7mZTI>k*FR4SZL^HHP1Pfv}cq=JUtSg+LvM}vXBdSk)SNMkLF@}t2~54%g?Z4jBudbD6rMB*Zp6%|ug6;`4Bf?9jIvP+V7ZLw>AmAH!amQ>$W'
';BD(8$m&iUg-avxw^3TRf-C->?;1RI25woYyc|7kE8KUNUFtBlm0sd>T1C=U{0>+6ZjAFh$q>%J-Y~0!s}*-Io2^;tV}0cvSyK#eJAek+*1EA*'
'Cdg*HrOcqOm{O(Ba*}$4(*|8je3jHF7Tc1a#aK}~i_B1sJ-rx%f+=>t3<2ytp1B{#v)PC7@n=+C1~s>7_rv%eQj#s26~ZjvmV@wjL({euzM^#!'
'0ECO;h(U2CA)LZ#c|_JQj*(?(Ji;EHS`8HZ$qMR!Y>OSxtY&!I#<#+^h7Gocb_2L)jpVCZs|?qHs;yx`Y}M$d*;)2=rgi+)is1r=l?+}~y{Y~|'
'^QEuVDCVoF<y9vp`mF)7-d48jU{Zvf?Mkj8OKf*)?{HgX)z<)%TOr|dX9~CA+0WrLLDikhDm4nCqO(&28PflD;0zsEfDDc5`uV8&x`_6-mqswy'
'xfHz~pf#)@hwHea-L%!zB_~`L#5XPD?dsOn0%Cp1+H0pWBI-0+W48#1TZXHMrxVl@ZzHl*8Bd>5?-&0jRNO|pYCu&w&%Ts>Uv>Y#$k$F-{m_R5'
'T*x=H#tsNzKIi>d5Z|(pWlVmZQ8~Zn_&SmGHJrYqH*z2Iy8sXYX1#Pdc&lyVuWxHbzVKEk);aYuae(;tbIXSqpFX;$=O>fd&MymetTH=wzsz@&'
'&t~J(v3oh0|2UbqiS6y8V%~JMH~l*$|1V2I(iDQd$~>_0qmnITx18`zkzL^&*d3Ff>6DKWNN-`<A@0TGr^)QH{vifBTEZU4E!lL<nEo{WF#fqc'
'C6Cf1B<Zqa&X=>xi`n?{lRG{>o?Kqm=LMW%K8?Dx`Efp(T%Lb9J83MZ*IQ-)fCx@QM1=+bSy7VWK{*lO@{>U)g@eQo0NaR74meW74#|BMJD9L}'
'9JYv+Yxs(6&C;BNPAN%~9Q?+?eWE|CU~vcBhcmh+ZO$am;uVPMIaGY<=B(eg*|CTs1$ce;Ue>}|9ei(*I^L3ccOihH)WP0u2$S100FQbU!(8a3'
'!9#0!m8Y<tpL=(W9;FzKg%~hZSY)9MPBAsNh*=tkh+rL`lK^D8g(L!}5=!;A$ip~V6ldZs^Qcn*2tB^ooWETsDP55WwI&?NQR%Bq2Vpv79?O*e'
'8SEqxpBi>(8L%b<QD2ZcVEt|JK?gpK(4Kh40^((D&Ep6f4<{??AbISBWCOfAtS*884(qLAVX6yGk|p!ngd}1sp<@qta^isf3HEKuhX8Vd1E@HX'
'Z!xwlTG+Pqh^~F0Oly77B#*qM!#Y?J9}j6wJ58-=OfmB)ejM3hlVQf3HGw0FGDkv#7EwmMrKljCU%@6>w%KNu-~ekHA+Q1pkFuoWq~ug3SinLc'
'#iXqh*1G2M7=%_KHknY%s(y_x{-z-uwwou_wU1w@ly@Nwvg(er#Zuggm$ITAN!Ju2NNvdljaLv`Fml6sd<A>{7hsNu!_DD6^RwmQk|G~KAmB1A'
'B3a!gO!;9J=H%|A_T%(qat?PVlVcPCqgAbiB2WP4%dH&#em?)~o=oRBz5Hx9Etxz$1Bc{Cv0H(V9R7bnpsHcm?CYz}A+`$(8Ea}2!DHC`-1)g#'
'Wr%WBh#EG(kA4XUS1n$ArM+X$bEkt&QPrx4k9Z$JIM`jbB2~R^E#`g2GPd~UhW3Lz3|IIX$>OF8FYfig`P0qg(bGTstK_{|3HZ9NHMK8Er|K(B'
'{WY)RS$>of$Jji!@V^Oc!I;~e$u<s)Iu`#6Kpd`VQI+HWK@hKKko#e^ktUte2a@+$3aW#w4KdrzVlmY&K4}#M;%!7~)gQtvak8(|(fiI%r;w50'
'90;gbcR~ICG+4NFPrdTZHVGY>BBvhK&uvuhdw=ggP)h>@6aWAK2mk;8VqLP&f)Q^90090D000#L003cWUt@1=aBp*EUt@1=ZEtRKE^v8`SN~7k'
'HWdFIiT~k=O>#1h+`-sXT}vy<Dz$3Wq0(-e5Iy-4-^FVZd$z-s0ss5nvz<%s=tL1p@$>us<@4OVy}h^M=3rrjfIsCIS&Kq)I8$}48Vg1^BgHkZ'
'U?p7%+Nf*M@Fo|)n*zjFUCCTJz^`bmu-Wd;?#_kaCNH5Dju+f<kQQ>}mUs{#u4N%megutjq)#mXe-}J=Em|?6LTh-R!N*cSrE*^F?p#Y7F0Z9n'
'2?N&2sykY6$e62ID=P|E7+#|v@LpUaBX^FYE5u=gSO2ua-0bc+HSnnrHaAi`NfG0XY!(&K*Uo^=mGRN-?d|REER3pQHe0l=HDWdciMEZyj9^II'
'YoRnmYU%1~eJ~<Qx2CG(Jk#7*(a9stpGkOkrw|L!6_-td*>6QU-Q5BF!Gg4=@mjOCLC3iki4mD?=ZRs@E<ZoN+JDYQzyLqinJAs686Kog(q6l-'
'^sG>;rc%6^q4@+OS0nf$o1(La0iPSOpj<+0`MeUC&9<I5Tvj7cm|3%u7=tr6KFug9w7^f}svF2lk$=IeC?l7$;g#<XQhN$o>6Wrw1bt|o8ikso'
'5xKCI8;J^d13b@#cHVO5E1&_bWaF}+?X3{nTlXOyl*&@7Dd3#S@f1~3H#pN;DwFpbJXbBS$Tu(-Id3WRq>U7P`tT08*4Q)Xplsmu{LQ+dGdMTO'
'sa#bDD2xTaj)qCfVwMEF{`m3yB3oxL{E?Rm$Hc-3TvDF1rEm%Bf@kRnda5T}fn4}nz{v?PS@We}ASzUUxy>02ER+FXZt^%<i)ExA)JEx*RyNjA'
'TZi88ZIWV%{H-T-y)qE?0jS2wrWHLkSUcI2+OXK&aN{gNNW#Wrwyn5R!%FA0oif-Dq@D}gYfHwD3Z`Z~HnGz;xp-Bnm1j$WKxVNgYvEJstuc>e'
'@sfI?;qkiF)N9$Q1I0|4CEHX#wA5?C4lrH|HuQVdh~b^))~=K(y1Vbu=g|R6nvlQ<j*ov%2Vsh@G#~H>MgPheWge{>mL=w8i#3AFAor0HJYB;Q'
'xiwqkr^VFXP^vbO0|&6JoNxC*e`@SviY1DKCib9=cy)W91_E)}Nbug+jb^0w4<%iSAmFEi$jo>0<?a1ddO5l3MQC_Yq#qpd)pY|M4E4NtkE|81'
'=LH8U<TN5N33zED4&3!F66msFVd7Gpg@yvp`7Y2dN-XJnq_YA45Ab_~ShV1^tRy#>K!TEPwT{9DUa#E2D#jwAci1QoW*rwp&0SvNYWSR-olWk}'
'&fw)=$&2a1#cBEi|Ao6B)4TI_jye%tYgChB97E4Xq~y)xP*6)BWuIUogq&=+wFWeL2&gc9Ue;q%1GnK~yb|-8#KTD~m;IsE$L#)lx_lcvAKLO4'
'<G;1=>cdGlg9Eyuzt317xt0G?<dYPh!;{HRhm)gY|Dn#Tv5k7pl`LH8S%>If3b`zuC#UcXj*cgf+Nbq-^2kPi&>eXHsNy)q87;%6NNyS8mvA##'
'uqn&KBXP#|>7hZxc<Cn>FRw5lW>sr)fsAViWur}d%t==vM;NbuoC7%w6n+N}G&c1NZ>TuG4an)X=3y4t6hCW+0;Ny)Kzm%oEDivhZcM_Kh3?L)'
'8&3D9s0Cl2M$X%T+o#4!>sY$ao&wwVq%(YBNcYfxDjA#77d1J)v{A(wBJL93Q;3gg2U-!OW376i4sYJX!)s^)n(bP;lPBFkuOFt!WJxv_U1MQs'
'p1X_KnSMWy$FcI8yK%f4b1gGn>S38n!3$yR$u0YYYw_T8i7!KJiHp~h!*}$$e}&O2Gd{{DkaX&oh;;2dLo4H#++eIPF+!)Bu`MTu2MdnR2`vv$'
'9lq*Pc0aI&tBKcSjOQWk-ELvHd=Lte@V7<!h@l_Ca5Rmikf-hD^krCD6K|(p(>{ZRtc1JK!oO3J4#&O$ZT_vs=F-(%KPYggojS{EPIL0>hzPzw'
'Y)U|JG#U9*I${X-8a?gz+_%(88QHKU?+EDb2#W4DRvj|oZ2+F>;n9=H{m3DwYTZe4_{->clBSzYkQt5-{2AtNhIx0b?Ou3sh5IIntd!gC8WX|<'
'y+_z!-w*J{tA!73L+1U%1)Dqmw=<-C3LUa|*~_TzD)eHm3WNyOHeVVXbqIH9x;Ya6`&5aD_#aS90|XQR000O8003fL>#Mwj`Xm4VfPer16aWAK'
'VQF7uZ*6dIb7fy;WNCD7axQRr)jesG+cuWpQ<eXKakh$*o6SCACoVbkuH!fp?<BF;?PO<5qhe7MBr#2i8XkR2Z~ymw2LJ**)N<@E+f|7r5_ouc'
'csCxPM~@zTpV!-kT!^@?@=aW&iI9u5%8QARSLs4-6Df{gh;3XH@bZejEM%Fkw)iP7OIelpsesn<I{5O7FTeP$l%=TFQplUMtkUgDYye~|<aV*k'
'(`|L~&sx^f7cVc~i|f3&l!b`53z6iTU0&kQfp~o-i(9cM;$`*a7wfnbDU6a|Z^a@lWCBfYpNm*5<1_=zap*G5w<3wRVk@Dm$l-t2MOsP1<2v1L'
'5sjp-0KrF(9)0=6vdA|gik5X%7cz=O3S$=);IYjsm}kB%aVGj{dAm*0+<c$MrTpfJ`T9qhZ_SsyG#?9TK9=jcN;5U=F0R&DI@e>qgReLky^GU`'
'=^&#rS<6kVTfIa|zn4I|vhv0EfWd`aZR8fdyi$Y%tlwqv))#**;@vt;_{+ue2B^8Mx&!R;EKP2O!M4n^D;e?Xr*Wp<cl)J<?t@QY38+uBgW<Z3'
'|5?XbTA5MOZMlO5Y<l?SQ33oRi!Ovsjx*B$Ht{Y3rdiPXeEG#fE=3t%$q^FO7gups%kkvPF9iI-Lcfzm2|U=aatutKCUKVCiZX|xD_Iovjt0mi'
'z+uMC;*uwqNnoKROfev?L8#ScCqi1o;5x0=QCTmS>CMOs;Dt9fZ31GysFF9;2pJ?;)SF#7A_Q^HfKC;j`ydo0a$sC0X&U|*XQdqD!z5n-$A(^A'
'EsvksU<(OUOL)HHh6YKzi{~j&I+f+9U^dZoC?qiXR^0J7{PVILXb`dVdJ~~K1oR*(^SVf68E_-u)FK99L=m0md8VNU77WuU2hl5o6;N$^HS%7*'
'iC(?^<=xwh*U`(1i`O5bpWl4{{^k4s@y27{LS`p#WHyWjzi!6226*@S@2`Kph+e*X6aDRV2W)y1&*R%+xct5!@SDHC{Ql+t3_z}dV}V81gP?VT'
'Uf93=dhzb(mlr=pFJHZSeR0tP|2n-)No5WWU|Kj6F7iY!>H=1!T;5oHS)>E^z1_1SE+9l%Qw7(cKh`6l;Z7FbnC4*?{whxsi5iwa4KMnK4;Rrd'
'Z-03GGtAGK;HECqL^F*ZS%c@7b&(AN{^hreH?N|1FF*X$q8u0k&^HkqBMrD8#_w+~-n{)a`sqiFA`Z8|?QidYR<tmr;PE#3LS_b&drLKel%w7$'
'N2IeX&GZ4*udL$D4h$IRKbfq>T*g)TT<q#?Qq^RHPFC@*guym(x-~k?S2ixm;&f9siU|~Woq;(>>H<`01xji<^Wi^!yM);x4xoBK^A(j`FiO6-'
'l(%6PZ{~|wTu;RH)C07N%A09q@h@E(u*3@3)ie~x`sMWaY&P{kkHf<kBKYPh(lnFXk$OKCFG6wt#D?5p9>lvHbTw_CJ2Po|sHeI)qB3?cEo4y2'
'xB&hwylL>4f1mxE=Zmr`s3|V*$iq*b?*aD&@FLxf#(y5ChBg+6{_c)G<|YlVEYodTuE{8BS?~p$NEUU`=Al>Mrc?#;1~DKxZm^)`Hr}DH1GX&J'
'yoIX<b}SRCLiu~jiA!LdhKJg=w~&C9L^ng*-)Nf`8!+kUKaj!i>n67==w`YAhbfb-=oN2oM`dscR#5yU6dr3ek9f3121UpLdRbA)PEKhr56rlf'
';&-&9uZtotM&1v~D-@tRz}b=+d=m^6#3bE;R4E4*?0lK!K$^IaVh6?){sk3%OJ<)v5L9>uQ)BG1O@yCQY>vbPKT%DC$ALL7#0%4&L<6u-^6i3b'
'm&;wI7qt`LiA`!2Jpf3iC(a2Mf9j#)M*O{*-+40r45VYKNt_wFQRzYxkenoV@*F0&1l<gdbO|02u*3p1WwAlF#jpY3pi#Kh)I&@_Oj-=%vUWg>'
'3w*P?9WA8Xg&&F<9UzC{JBy+0S+k_bMEP>*Su-W*#!{6euR?r<vs=lngpQ5k1qitO^Lzn(P{he4gZWDQN{SX&c9SlyW8@MLpB0Lej$Rgl_#r25'
'*vMifPYMaopc4W#K~vD^V6`g!OJ(7o6nROs4nf_64>`JK6^<Q=EJ%<>Spy3xAT~dKhb>4Q15JO;x6)OVW(*3y)RzY2i#AAR)*}&|x0Gei&|w!K'
'_t9JnEy{Z1$j@<0EcP)q1KXg2#lcKipsm_q7qYTAxG~KPL~|_%Md4sP)G%;AlX5FPk+@!@xd=n_&SVL4ByA@B7>2OvOi-g0YW`511*h%7Ib@1f'
'90!(t^6`~;3D#f<k<c6%UZ4buv=slWa}*3x=td*3m}tKP<8E|NG9VEJIY9#zmBFUo5cwQgg%<{$rnwm*<Yk#oCY$(Xbmo)OK##^JC+d-cJs!`d'
'6ZLuKk`kk;8=pWSIpwzAkf4wBIBg>j6vKgt?ixGtQ55`Eb7G%D#+UA6jPYOwp74S{a?s=CYE0S3m>CD75nse_IypO^ja@h~067g1DVhj!&8dLU'
'2O`ick1L|b^mNuZXm0npAiPIfz<dCBB#?OL!BeIvVmdxKdo~_-=s=U8aOF5g->og;BpWuJv~Oc28%Ai+LvNkumtNmAJmRh>iP8D$f+vo^sf96y'
'z>9G16Ivnuk0d%FS0wbReAhofmS4*v%H*;d8X$&+CGvz`2}Vvvi~M@q4{QgcpyPO($<T|F1cn}h3QSQVNEV++^IOD{15mW{jviGRPL~+00VOw+'
'yLb|uFYk}<=JYSD>ev?2ABud3eiVpxTEQBwVo*-BR@Z9@EPkWj7*VI3n~$yyCVf!3FY<bxfs^N`eso<{?17<j;w3{4T^U7W1fs<W^b$Knit*8~'
'IEn?#E~oP(A%N@6@jE9wfG>ewix{vHS-ONq8(W9uNmB1pQVP?SfP=GY#V-z)P8_qEh!wkFJrWN<_#Fzm7M31v2L}w$uciA2d$WCWWew#_z4NUh'
'j#T&~1D3t>(;nGF@4?KPr`cerUtny7(0ee|O`34SHuMgegavSN{$$oMTlH?2k(5DX9&vcv*bn3R5=>4>C`Dhk%$iohtk=&blXF(vO_(3w=;Tp+'
'3m%#;z8=q7lY^GUt*u&3C^0gRjHY!JEI2$*+oVPXJD^IR_o>gLuY;%E)xrfO#K2OBb6aOV8Pb_9rSvk{DP4vb(*OOo7VOtKtZY%Q<esA)cWFXd'
'Z7{j&BfP%}gVSgJZ5TXx=3l3aY8?j8p7?8-uGUo;oPT|&x~x$r&cTCTrZCi-EUPgDtz_F<fL0(tOV)ZJ7@&UmhwDY3f;)&Ns)*u@aOx#7Wp2wx'
'X<CfDycu9ycoUIwEFE{~?KZm==r~<bXuzxw3jni2FEJ1pZDSxryC6x-`Q+<0ghD!w0PwfAo<+q*t|sfNyEth4+Suq>mpSgm|BAbn)v&9Oah9sZ'
'i_?3HK`Ct=isPq&D~lwEgiKc(R>#x}6wT-?c-kT$nud6_0?)OCNqcM;JLwz4<_%>vi|fq?<r!Rq*(iZEVzBA{Ri1%=7TP)i00jKl56;g)gtc8&'
'c{aToUR$!Q*;h`IrL{Mr;!TQg(J{1QP|VMd##Tzbar`-**iOo5wWR`%C8!h-wKAkQt4Y%Q1lCWe)B*YQYzZ}RaUB=9IfZ!}%toLnk@v0l05}*='
'tJ@|^la?hQ0@?vX__CGg`>gSmQg=MnF~pNSZoOI?Zey_qW2ZS`>AlzKle>q2+-vFMug`TX23;*|?t9WS?O<)do*`eWC25qy0~7Er02Bx^Es26-'
'GD7pZ!puSeI`AsSBq_=?ENy8}KLpvKu#Hgc2~>KQL%$xL$eS(gStTikoyHT>{Y?5si4F4CT^Z=FEv7L!7|d8v^Ly?No+-P)4jeB93C_KV{&4T}'
'hbXU|=f1}e?(O70x11O0VkNsB_<7Uwo-(~dDI*`&XJlE89(0cz5)9Mwmy|OoR(es<VIsUEFoHTJWE0RG=Gc51Tj+QubLAnaTeHw?7ipbicz86h'
'DCkwTGGgQ5Nf}9PW`%q3z~E&9v211Wax-WX5ukN5;YAqk4=$Wt2GVN=jbHO7zEKJRK|R+y#v+Gyl#XPNW87rfRG{Dt<!)y<ZaFw_XU|SsX01gO'
'73Yerln~97Yh2KR^Cmc=kt0qb16s%cdW7lZ+39S&2Z0j`Ac8Yr2N=gD2p~7J?OF~Xw@6ET6c?EL&%u9AlS^=Gs+{l}!U!{oXvC~B{zPJp(9-VQ'
'w%*J+?OKu)220?DILk(b_m2h1#UB5iGitVtKVVJOL<WX4lpcdzx3TQ;X9nm?%4Si6KPdDJ<2hy$RmP0{Kn>re_t~k#0-XCxxVB+Xn;twt)!|bn'
'`KyukkDsPJd}peFwc;nOr+T;OxDD?iGD{3?&b1W9kLeb4gSCqapl)^A{#Yk^pQ{=rO3m-$TePfWo#}tImdT|sc}=DNw0`vo=_IeweM`4XAe>H;'
'5-?c1Ek-_6u@e}fGU_=k2>^PAc9_hsqh%_y1)Il4?N6G|r7De)xMfK@`}lwRA#It%W8OM%Ubs|)cnR~Ns-Om%tM9bs;S7Rawz5jl%=$JA+qe%C'
'^U*=u@xxScgXegCc5@TiP0YvU^}Q=Zij0TY$*wfvvgo(uFu02{RbOZRX<yVcWH&WRw$L#8C@bo1OBZZmF?{2P;B2O&K>}HB@Yo;qQM~Iejt4?;'
'>u=OzECZy8>}b4R)-;VRAET4ev7~~?O|w2lS60vbK{7ABJM48oX^3^FKi&I!6Bi!#4P=4_g^cDH!)0=dR-82z&8GJa@4(rWAd2Nw>xi<(-2{Y#'
'QV(mlnXHGwU@$;JcU3H=XQfNMJn*<CI3NQ6bITvRiFu&oPH&>Z$(BHuNe<Bw?Z=$@;_Ns$fBG1gm;sEBgVS&5JNE{*5Y2}4M1QhMVYo)K6H6}('
'SQx+u+;_S3#baM>JF8l_CXRkf6ULwr>(0bmpvcQN$Czb?k<jt_{pVjMT7y&NsMw`gImXJn()Bha)u<xS*Qy>B)iZ6)6~~maTEPk}0`Z<`3AEl7'
'c`31xvL99<4q1hT&R&zl3SJ>%xr@_<nzMNM?hST_ps?O>hSwy^Ow}!xBnD=(Du&Vsh68m_R$USnvuJb4z1K^wOM0lcsetN&PzusOqS_x~D~%Z{'
'F0$u6^-z;uNmGL2RBD;higpnWl`FglvkR-I3Ko9nNI<QH>R5{5c_EIyG!sX{#@E&iMviu}0Obs;=1qtpeM}r+P6{m3&%}sojhR5q5SoXyrP6kW'
'SiMt?;Z6^DXd#iO7OK67ui`YL&;rfGl|6mWP1A4(j2E8SZU%!1#gsoysRC$}J|0<pj?cy?-;UV_Psx<V+m#$W8|xhctTd_<l~@8D{DZ_;a83o*'
'FbP#FqQ;^G=T;@T0`Wb{_C(x~jFWeGI=c^qXT|m`Y~nvJ-u^1?9+B>OG&v1U{YR`{9!;h*|B<frel&ShS@m=H5b0Ca^FKhWK;mQep8X~)Gy9um'
'y;`vjE#sx6N^!8H&~Sx49<}hItnfn5^?DtXwon5G;zNPnCPce7fhaqIe&I6Dt9S)&Kq+92i<&iej_zm`(4M0r8|GE51p{gjb5m|#lvwUQ&oPY{'
'<CcF}v9kPRf<D41I$c6oqX|f_9=U{OQW6dAIkpMK4m43Y15#2g@~w}h0`f!|E8Ka?>XP<WkWqByiDK!i7-m!5ntGaadqwqzR6&u?Rqh>z4lK6U'
's>$JZw-`+cs47O&YZ)=CHr>F-y@r53Y_}L$v)1NHjgQM_D>+?A&zbz@g<5hH!8@<I-AQz+fl}yc<3AH?i2K%-Ck==UrG$6#tnE^1hEh&K{|HMe'
'I0c^wp-erL9&biFE3lH|!vLlmU}h}L1*pyE$(=|4c#F7-C-nT}kDXjS=Tj2Ccf=op`8Q7}r!7Z#;$fsP!8&%Vl0ZR?=C>6Pd^`?V1&+q|_l{C7'
'cen@BRF_a%vKnb?<kL}%@Xlj>KIyYc5G={E1SG-nfz>GT45wBDcVXDUGxsLy!~N`GpSkk~6bG$TkfY~KPGP|KCaUsFiQ&w*r>FNmQa`V&aCG+6'
'KYQlv2<tfpWE&#j0L;}_bvYXMu&QZ|^B#?-Ub05tyYyzXxv6Ff-^|Q(HSpeoO)_k{FpxXYxdj-}c@;#vxPaa?IwRHgLoNG>rG36}E7R@h%xCA4'
'w`6Gq_KUd<c<bX%(BiT7@+MYu^~j%Mm7Awkmp8#W9IKgmL3O{NHJ_r^WCxW__y}}U*%gn-fgG`VHkDrcH-M;%Z|&#QJKSTkT-f&nTf}%!X+axM'
'DA<EXn=pb*=tPoGr#{SSCRq&E?7C~I=94u0U|1ZIgL+ew;vF(}Kl%ZhB(egj%4W^;X6tnz`P>sH?XBI6uj8~*<QtV}LPO62d%H!SBW-pnbQy1`'
'daCnUL6t|<8Wqc^jq>1HwieMUWz}u)@Q)fW;OFKmH_+QBpHfE%+F@6tBSR(KJ*E6;EMu-3WKZqPOex^b5v8k+&YV0yK*P$+_OY!8wl*1IgQyxi'
'RR3CvX@x8ub`N=t@RLPzy(z|t4aPLIa~e+MfC7M!gZ#2LKCn4>L=BrAUDi!Dz&_?OIeX8-jZa4npk=ZQ!#vj~c?T(!k1{!~@=;4-Yau4R;Zbs@'
'dL}aUlx#$phDB1$Y))4Xt=a~ethX4p?xBEtvd^cJj-zHkK0u}%07f)0)i<q$HiWWOm*KBa^A}Ov?8G-iwca?jlPYBpe(Z+-VbGga>LjE-oHV0^'
'uOXv1X-Mi+Bw+4FBj)J@hm`_VszQ5fdc3hK_2xZMxmEGNc#_3-6;H=F)?1~=8ruAk&)cSMzeO6w3B~M@76xZcDcQ{ace1XxmvVtgQA|-+${{~U'
'J{&*A@3n$jF|(ZeBC2=bh%6!|fom8wkszRhWg|ZB&#C<P+xLHqet7d9lkzyB2*?A53OD%B9f5CC5B?XW3&SM{G}v5%`5&RIZ9(Udt3;KYQGUs1'
'BDifM*6Wnw!j@@B@Vr>qe;Rb4{$L;zV^Wp~a@smrH|r!Z939*w!ZV#BX>u0e0IBT|D^W6tT&Fg~+Ct)nOIamrwTKU()dhwqfNcX4i)VR-4J%w?'
'zQ@9>T}#k~M%%!Dz7Pev_wJJ%va>OE4kgi=U2-<(61$Jw6TiMbbL*Q^vVL=_)wd3yj=QIa29~<)mdD^$Iry?0H1CEPxHQH(Fx;>KY68r86j;l<'
'hw_c{$aw<HcY_D%dzel2?un2eS3ogtu{$1!J($TYbf`Aq9Uht`fUy<;Iw$L78I7Fj9r6bdRPvo;1>7P{nu;3Q5VQ;Ea2R4N+IiT^ld8mzh-c+;'
')OKQ<-|^{TfZe7x=w@Tjmf_AGO9yysW2Fytvjcf>fmI)v-kr)Jh4UvxzDpHnV|Eat(~clvasbxKIm~Tu-nOXNyrbSp?VM=%N_@amYZ8kvN?A}d'
';<!9Ros9=1(1*ZOR=(tm68J!ZPNHs_y%2`?UwG38Zmz&2(toj#Y`|0C@3PbYz`2O)3{OAPzN?_aQdnP`mNqyTvXG$EB#ElI)S<$(ETV(ED0pB`'
'RSjjgs8QDGJmno#oReLjYj3v?(%iH$D|c^XQP~bt;d{v%*S|uY<H~9PIBd$%81>l|DThm~n9{~w!{Z4W?4jz%R2k@=Y|BjP6rikEu~n32h8pj+'
'zeZfTar^<Yb%0GaeOl(Y1v0cMHu|DCb&S}j5^`CVVQ~7yw_TwSmAe9<GVdmMaePwa-J)-%j~db)Hv3WxjTg&M^hj95MWy`Ku{h^^u>YyYf6eOA'
'q0f+qe1Pz;H@oUqli?a<JHOsXT<w>W7Pe`8re+&ye8M68c+=cDI8RMEJ-3caJ}v2Nf`3KlY;`tFIgUo~K?~7QWUN8r!-@KAulW5fX(il-f+%8('
'2{x*8vBY2FGqj>Tp@<4ucuA?Enm(CR;_-G>1G#Lz=_96c8{=)9-C`okBIo(Jk9b)*5mgPRLo2nJH-vO6R#FICr~+pG**Q<as+er;Gqm>)^_GD+'
'QryMwtK!ybX>HQ%!O$rLuyP=}5<nAERRpMV04oTRH;LR;;x+w)W5syvE%{8_%|p|1a@nH28BEa~ATgd!&N;2D@UVbPEeN)AspLhWXzz4UlE8sr'
'lGwA}EhrryHbv`l6L0Z!z7lk`8CGx2uJUu0c&5j6bC;2HQurHL#kjQ-fa$PM(K&URK4sKDHt)%*($0E*CElq+(xTkqh6$h+XZflIB-t~xgM|+t'
'E<}#ov&aFU7q-W$0+6SwLDD(N=5jO83u&pj!-TnwwbULB7)>nSCm#-v)+ur?mq@BBYY(db3WFhgx;O8gE?Z8+)bdc;+kSmI{SCP|PDs)+sZBw('
'25ehZXNnz^qj|kp$>vsz&^258TQ_nI+QWMOIC%23cXYRzMeo*&<HIrT8>YF;h~p;VE$b}voWhN!+RKU#xk9Yxus_A4YQu(VGb;%(>0aNd@iCo>'
'F*}G=k*>>hT%-uovyJ_MbCX$f)~h@IVes1B4b>WoJ$M}gABP$m!=cCULic%_UkT+RgLVL7ff*;qK&{LPu8Y=Icu$VnRr(i~gfw>-vQh~Sqc5T`'
'EK2=RMY~r=hyT^wSJL#tLVt>s0d-Bv24}4{_vF<~gibk=-sN_i3QynHMS6+^!M(?8#^(b^)<tC0R^=1ya7Uf6w^R>0uLC28_SI(lQ0-<Qgl8^$'
'Q52@j0?`RN@w6LByXT&N=c!<y4HK21Lj=v)eviEqZ=$@E9&7DQtS?l;YyuB&g7oq|M$Vg<kf}+--y0kFOT0S)64hX>vm6<a?|j3{%=qReAO`sE'
'7%dT_(p8z3U1VQgF)V-S-p4^D>%b}zH*htixoQzFro;#2jM8C{Y8@mp%|;{RQI1a-gdM?R>lYoCi*KUVun)wzOYefc7S5~P5#W2FfvCxqcpxZ2'
'Peh8oF`9ZyPz1^iqCHd1dS*A?4o=K{BFz>n-$O!3IBC(?$Ft)^j3esum`C=}JmP%-{Axt+j?Tu99fA>4)^kjX!svas{KVgd%|$=o&8-3<eSRdg'
'M8R@LIOquPJSW=pCcxZQkm*h4c9qSZ&G##5i0gzb&yIq#^T#y36PrkWYbioafo3$`Sp(y?H>h!Tb4F^`W8QvO5jRoLcgnLIWr~HlpiF5b>wUBK'
'THPX0x2&>hGk$SqU(tj$H8-lPOCN^vCezXpZ0o?Jdu(b3;X{>HSbww&=)*m=-BI$$jA<=5YXyv69$aw>o|>!{eLeF2vAxiOkmQ(4137JO_SNTE'
'N@*3S7#TBGW+P)HDaHfV#ioiFSXXmS<%plS$jw}xb!Kh+QOn#4k<_(h4S}bpv*AdcZ^?^Hh-CBq6V1q~(Vtxxab%?=|FQjnFXoH8J1U&P;3NHO'
'Dq7c9wwa5@#NQwU5e}b0u$Qb+fi2vMO=>w%tz%uBW_ma0o%HefMS}sA{<I20b1a(&zDQ89<H=2*ua`Q(G;Lm2CkfV!X5h1&U~(bDJz}~m#%~{_'
'*5!U&z)MAE;^N1@y927SQFSSmV1-Uvoa#vRrB%qKfp}$E=Sho`F-hny$%Q>lY~H#zKvkIl@1NPc?`B7sOK?B?<0PTsE?>BcI3!(Vi-#K`_FEJC'
'(n2x}j(KtJGu(GUz^qjaYfsT_ih)9>e@7s7-vfpEV{$<D<&FDOJ{{j%3F8j<&`p}~MTq7$O|1FMkmKkuq{#)HePa$n&SmLdmc+N0M|=)a>Z7un'
'b@Ga8QuGv>i#Q`-ZWh_{=*;p}^wY*)AsDu!!lz<FsyH}Vdh|8AGd(A#=ZkyqIX6=k34KKfL%OmlRF8Nu9c6ib1d;w@GuR1X=Uek4Qii6%xWgg#'
'1*t~M`fcN2_j@oG)qv@T&^JweUrPnL8#JNk^8>uP2lp7vF*x^L7rS^&7ZPp2kS;0r`H)6v)vB~+&&6LYehORPKf)=(%W$L#H{y(rG*$8#>%axW'
'Bc7AAg}|p2_%_?tEn_`&Fn3urcZ#sf*p2?$^%dh*5@4Y6gVs&staZ<xB%S_S;(YZoV#1e34fqZ!oGyJw@lv;x3=G+xFt~?tbLG!qGgQrLFqP|Z'
'5Sq1-EtGBC1GDESB{fj@z6q}4rx@LS9tw4<l#0^%%fM_SRTD)#c;YqBRjJ9oZV%`f{M+P(G#7-?m(L(H1`FTC=OaYu5HZ5Px`h7e!r~`hqI7Wx'
'OC@OC4g+8jisBZxsp2S&O=}$Prl-LmKLR*%poM9B1=V_K@VF1IE!M^W*RALxMtCvxG<a{eX8_FcZNN_jKW_(KDu(x%6TV)f>)9mvdux+0d7_Jb'
'Ew!c3LV?)szlLijj>I&xnq<vF&RpYVou5XyWy`U$-YN$zo|oXH#no{&{x10XX>*I0!fxzDg>+b~r+&3%CX^VpZJe=WJr^z(66r=Yy4J;u>>w1Y'
'ET7{Rkm??BW6q}a+^4RZX+9v28w)P6LeY%E=YZ%}Gde$lVVh7&<0+lf_#Gybhx3K7x_!Hs@9CesFj~aZzZIbNBb)i)BlXb^X7`rn<Dc3zV0cqJ'
'i9PE(0f7f?lS=!geYgF#^^1n6j&j>o#;q6^|J>kX5PMdheWza7Bih45%I`2Wsj3Lr9rR&G-J)BMx65SeX--V#dhZq_^0xc0-LU^u6>W3Wjhs+#'
'V!Tja7k3lU`Am^=?Vq-?9QL1;Wma3xdL-sh^=+VcCklD5f8mIcQK17~Xi~-L)?5-tAvrCI8-<*^*Rj}tJ=Yl%VQuCOdAEDGm1al_UqthSPY-ZZ'
'!YH?*qXEPYS`}An6<>RkY44i+`SoCX#KRt);XR2IxO^N#(yl$#dj7er-H+*wO3Wz+Q1?QpI|?YBr+%zRC1UJ^2G4kmfZpNfye=O|Wb~2n&n59_'
'-o1GXza0Q~ysZQBe){haR}BC9@ATDttVCU%Cq%}d<~`>&&@j?AuVT8EYu*<#wqUd{D`Q-|eP@YURXuzFfk&s4p6*c^`;ApMrp%?n!GAVB#9*k*'
'<Dum|jk!>h@*QUW9LC{eDtfH$ZHv%3NN}jp!M_7El#5^2e6*9~?Y~bmA^10JMw|l~pV6A&-+)EIzX9!NSghx|4O3v%jQ@30U$u9(zqd}ESn0b5'
'C!{q3qeiNLKf29Uc}`NBWuqQNF||&2%`8W-zio5KOSf&^tupL4jyo!;HJz_O#8ll+=Wa9lB^-{1K*F#1*KqKL_AI`^(245p=~QL!-02@W>i|U1'
'=TFqlg(a384PJ}oG1^pg_%KC`)BDg|KS5Kpr^FqdvOVj0>#RDu2En(znu`vw0?;9+Vc3mQgmdL}p#pkJO5=XaY4GhS{xpp&*Lk++SQo^QlAt^c'
'V~o-3sDC2qvUlc`3_9&n+mLIdRV=AjjBsh>7o>X4xPAAisoeOTz8!%6@rR>0jzYlM%eq^wb`-wns9;YdXnaT=Bzfk5DPEjKKd#jc7h8(o(oCP?'
'#;OnZ?U47Q3_{Cy{jS$_ZFaFOm+{UHgQs_oF>k-|@t}&{HKl26^gU+gDTczJQlHN5og9yWhMV3kKM$6h(+X|>@1TR89TKYFT8jpl=icx6#m%nr'
'{}gYX%&b&9L``<ZlpBH9&bK+J->)@0KvbcL8>R($@-hbpO5N-a5qr*;8`1T;YOB^d=%P|w&5xOxt=XoYHy|?*r^W$L_xN*O-M)#xxr7ItxLc9x'
'YyPoJ0Jto<6breds!|-vT$`ydpj=ka6!Yq>S9IcmCeQU@<DRfZaU<=V^*BiVKI;Gb(Bz~1;I@|iOi7brZ}Ic^hn8R1n#2PKRq#eM+PExXnaU;p'
'MoRNzeJ9OD`dm7lRL%X+J!-PG*RNjjowM5W(8(QrI)ZMO4g0R&@RbkZ{@N6qJFUuRzH<e~LSl)gK2)G~VimaPPJN)A*kH4JVr`$OA7(`rbgp7Q'
'H}3N-(C&>wbw)chz!IvX-}DP{;q0FM_hag^!FrzI9;}*N|8Hih(051<Tj%Vpv3+wyUJc8~<b8e7h<#%!E=qIf0gh`9)+Q3qE}%Pszb?Ma@@u>?'
'cYS(xVvYeE^L+l}+|ynLFB*NGp<T8wC@J-8^{vORP}>#Sna~}SlQFYg3H{3QE%3G`-!Osgbab2c4BhgaRVU5Oc$%Aw@Ra+ZUegzRYwiKXN}LgL'
'Nr06;m-4n8nR{@0jB#&;tF^;k<oLtc1Jj-ATG*N2@@Bk=`p`3C`-+kYu>pWnE^O;H-3Zl%(m7n+IVRg_(F?`B^kXv3^_dv=g@1Q%1ucZJj$bX8'
'&bpc+QY@_eI`wVfI9XMfp{LEL=c{~E^T1SKoCP{B6cP@U&kq&wK37Yxy>2sYSY@V>YE;Mrvlye9yu>;>(;CJ3{{T=+0|XQR000O8003fL$kU*$'
'6KntgcAfzM6aWAKVQF7uZ*6dIb7fy;Zf9w3WiD`e?Y(Jt+sL&j`aNsqKOo4>C24~cCCiS(4m(G-)rnp_9n0P69JVe4k)VW#Bp3kH&^r9@Z%<XI'
'Dilb`>CVl0>oIAI1Zu8bd#XKzVfZ$i#PjYf&5JTPIteD(VwDwXnP$r%Sze^er1#5T{PGu{FViAeWYhIL38u+BJx}tuOy+k%k*s3;S_I`~o+MCz'
'IZ5(mF~C|+g6zCV^6MCz8w9i2D$mZ7C@A8^YMxAkSspKvA`0T=G`Pv~X?LDnC-b1p<7F|)(-l_x3Y(dxc{1Tf;-FZ>^La4GapRDn0yGPQhB3rJ'
'dAEXwUstpEIFGrF<TjbCxsB484Nf?n)A*C%I-aLfn9Vd;LA9XFU}ypKmfdvcX#vlF`HLX<=9|STxd`I?;yS>mvOr1j_U*d>8eYOA^57!9PL@<X'
'KTlysIjq%X0s}9zY)&)gaxk$3iWb2t$rou+zyN>w3+VVJPs;@BOLwz$UShL(yt<V2nBPIyVHp1M7qdKD1mp2+U9R(FJPuNreqKWDWmeMS7dS@s'
'w76SN(oFxpnCP#Uad8QotAD4fcsk9KqR>zNUf}rT$2_|L2)WR|7jb!6{YV!H?nYHNtG?!m{#snZuFR{S+2krItDoz0*xiXagyOFHUM9EY4J=9Z'
'0D6`6SixvuGHNkC(%|R~3=4*>UK}m&qTn^`a(q5lb=UbEs`dc3i$s-!XMfMqWrSbo1+FQTj?*!pn`B%}E|W#9s<aVa0ytL;kjbayAM2zjBj+)~'
'sCrfuNof~*6&J~Aa<NF3>WKs1^7M}g8tKG%@+m2@`E@dV8{Z|lJUgA;9$!~YeV3(^Bm+Q{YMuI&(p*jFc$$_}E||c<&ExSb%g2*>x{A~>8YfU>'
'jEAS*&MKRylRMQG{~E*QE><O!S>jek02nl6YA4IncrMlI&HZByhqcu0rORRkXQCQFq#xzUa+>70TL`dlh=vMsdlyOhmY%i4s_C$UWxqRlbNpd^'
'{Qc+S_opWx-k(CP`>?lI9fna5_HL5%1$|j$*ZlWN{vPt*_&TM(Wx<aR;LGNhzc_h!^vB~`tAAfzP^rJeNvAKX<%Q~IMPDw`8T~z9tgxA*Z{M8M'
'2X_<87*@|@EHV9s8Pea`Jf862;4idt`uW4hzkK-ghN?3>gz<T{F0oKN1AIeYF5+}aU*<(df0tAGlPCD+JekDnLcS(T{+r_XrN#54ypb1^Y(1yC'
'xS9C(GQOso>9VAz7a2FY$O;-nww&ML-|{jpb%TJZ)103s3;D~{;7cK2(t`hH>Or2ZFD~)x8leHd0My^<MsKJG{L4eVQElG9nYxrObAFPR{6Nj-'
'E?cYd(B!gQHw&+5p4kmIamNkJ=QL{uj<jsaVB@7UqyAEHX`S}vm%rE`aQ^ZalX+Yef(er2JkRp>r}YxiAN}YIcmS)oD7eBjnFZrzcGK<z-LHe9'
'%%w!00MS?y;_SiG8GdPpzxiwTH;e9Xrk{WFr@?RD4SsX_UoZqJbFrWjosQA$BEB6LFw99ZKEDHIfct<=wflf#?PmwUlVHDpa43K9n3nCVz@wNf'
'uiN3#$@uk$cOO5T9*^H0eLp_^@a?D9$KzLj{d{~1Lx2&s_g_feCr_SrI@Un)^|B48cmW_fAmTiN3THfWSib>w76pLmllcJK;<eB;ribe}hk;B#'
'@&g%vncvAb{17Hu0HTS<h^}zjg+fCxfHA+4Vc^hIMJT<VmvGo>rPQGHNM(>6uAJOXl2sX;P#?6v15c~QJdAa~HJd_<L*w<wlaI&rJjwI=a~Z-='
'4+Uu2<ZAd0oc}}yl+L8v9&hag_^{wt!=P`BYnqgCI>*^^N$##UO|Wn~T$i)%OW4r_r&A2WJOQjd0pvUC9`^hF!I?3`I0Z6n!TKz`rv=&sGw39N'
'-M;7kM&11vSbGz8EZE~g7@(qfz42;&4ish#RJ{%VRbcTxTPFGx0lA;TR21oNBOJny4j!3wMW;d5h0FGc5l8}HzyRg|IbW6~VRHNnlpx4*dMBj;'
'O_fWase`SP4nF`o40yc3pFV&7*gow9-+p=vjBgIS;sk*tnW~9nn=oXjhaNDqK<+xdoJSt^!fNbPg@#M)^`&58{i8?Urr821!e+-{->O>G5OK!t'
'FzvG?{3aYIAW6UhFGh!FqyAX{I5ohZI67t#5n*8IbUDk~*iEFS%FJdS93DQ4NFW$WjnfaW|2#hZ{OS1UU1wvREY)P35;y{njBmD2a2CEs0zi`z'
'wgvt!GGyD4i=)*9xWqpIjf^&+Rc;g~B$LHC>3JJ{rYl$k&4TyR)mXl^<!{F*JA;zFv;c-G1$isM`SXK$maWd?$(8XWU0#h@e$vmNIS|wU#`Tj0'
'9G7$w7iINmO$=2!OX0j#T{rFj$7&?YY}p0+TOx-oi_-`b`hXVkEE$8?lOWjYqZt8SnZ+=+aj~Aw(p!26>jbP3z@$UlxCp#O=D)2EFCAPI58(Jx'
'33Vv+L6C}adTXrUe+M-_?!N<<Co`5vkHa3jg`m`<YW|n;!Sfew9Or;0;7o3sUL<tFMQZCw1(^kMDI^2+vQ@Hdhxs|I4!|`uki-j<oWTZ7F4xN|'
'oGP5LygiQ>=hJu~MFGXf)3!mfJ5g{Rh8+h4csRZF3Xy7?8niL66u3-o`CD5yo8-}eraaAVmh&v0j_H}T=S7(=MUQ}M5d}{o<R_TlAi54)fG8Ul'
'1pl}w+(t7>-UVcQ+eBg?gainm-=f%`gU|+I`IOev!UJknMGXl|);S0#B@-5YG*1^PKG1W+^W$N#xHPXFZ8Z&~cZ3gK&r@i%t?2mv%YLsP!MIS7'
'0Vkcc2Za!HoaTfoI506rc*x`BMbduZZnl?t%4|AzQDcbQO^?B<9scq7Gf)?)6$O_7W(jZt_u;pIi@HY_@H89*;pw0H``x3HuB`jsey<;HXxM<X'
'R@t&hoWaQ%R&SxdxLg-w;!%*eJ?rmB!L$AW{__m~`7QqQa^q=+b_6gb>BELqlQLGMLG4f#2s{1;U@VA8RI!F`ngHyR2})}7@XSM_!CN&p9#GQ='
'EI30YMq97C8dXzG5X{r%I(hgCe?1I<e`}bemy9|yL}UTS2>F}f-q*#(H%(wG=W$L<ntzu_#jq^9Ku<0}e)BW|Gt#Y~f(``}^`H}c%}4X$6SL=q'
'_$8ULnh}OFOE1>g51wXj*f*jwD`VuK`@Z>VjEHI~JXeb=0FE~QD2AW&bpqV=EeJv5?23MRdZ2S@w#CHaMu-KVkJY+q&U~^h@}ZYLMnWRE$|N0I'
'fS)J(bQr)6S@gsAfxY7L+qXr1exB`7A6}&%scaiZ&VVfx(KDRSSq4add8f~K5s=0~lMhr`(qJtvNt4Vkp_s$q`Mx8MKZtJTte`CZInt?G`3b4#'
'HoEmdnx#OP5D5>E5Qx2_3V+&9A4S5Px=a=M*g6!{N4K3aiq^LefYJlO?-=iGG2x#ji(gC*cay=Nn*2SU?A6Jgo+Ay8G~T>$ZKLWxL40-*o$AEy'
'mbktgqj?C;KS^M|gs}KY2~iSRz+JRNV2m1RAPLPJ&4UP#L3o<B=I7ZW_1Q{nekLNL??qW{AfiZHBodZTmD&txjZ*t&8Hl<`Sz@chz)<qOJAQL='
'G=BTx_0e0U;eB)R33W}OR;g)WoD?iZQxDNQv3?fd2-b=&Wuc8^iTa1>R{^el!w5Rn$d%f_?7GuSZdY(j*9Gg)SnY$_z;sfYM}U_`HDN5&gIh2r'
'j=C2-c^#!S<n~x^Gl%1Kos7$j2I?t-DwK9-$D$b{3>NF649)>a@zNP-Ly-cB*uG$>GvOW)JGOBX$3^-hhyy<1^+7u*OQ;UXwtE3v(ZCgJ%7rqS'
'Cl`6V&@59Mr#DWgO&pWU3e}YcWp<S;ZOHz7{Py^dpN`&*Uw!yI{`}$3$M3g5HPzp-81)7fJXa_~Ex?1+0f+rw-!JtWtWSIc9J~_Pk%Y}yb3NGG'
'i&tq+tz|FEFZRx}@}6e0*@Keb@GTjdg)ER0l0^nKNWN`6%+^-wx{B{`yG-JWW(Z~xYMOv|sERY;Wwk$y=YG@SC%ULZg;^VKRL!nJeVTNo2T<Y9'
'vNpjqRmx!kOF2$v*SyNE+H1>FV+-bb-|K;GLJNXxy0v&h47Nhu!rBh%wh4rBj00fcbK)MURL?~HF8OJs7oN(p5XQ)Hn-Tl&1%i}FvD-8N6nKo='
'1Mhm)tGG*rb(6W#y2cd>Mb!GJW0R)3{?3YBcRlYjp<#jti}h*+$8qW)JEQH2V3VB{-!Ka8F*bAxgl0q-?`U~<CZiDBEWAY|dz>C!d&6nr=b??a'
'<mrwXs5TApt{S}DS<;Pd)yzoSh=pCUWuWUA{L(EhK?nkDl&47{x9p3!98T9cO+G?uGY}yvT~3qR2ql}z7%zaLn{)~i@MV%-T$WKjCd0Ea+O0_;'
'ykdyhEKMF?#>*w_5_O%C?FHf-xq;?2?VK4x);<Vc)vu7Zj@M3}d+)p8l*F(M*mjSk+;(+a3dq{IE$x^xhfEx5&!8qq4bYUJI9eJL>8Zku(U1UG'
'qI>(hc)sRsi-OOvVDgoQ_5YlHc>jim!IlLN6YkaKYkgk2B&;Any}-*m7;sIf=N-iBziK}6j0ZoGi{WcvN-`+6Om53|o9+g!OG0S}FUH`J62lZk'
'KwHCM5K<#yhgM2Z@_L&7)6G&NR<kJPsSW!*$XSZ8&;WKU<tZZok$Kkz-J<dIc{mPAxHsTXZ~Q>_Z-OZv8fMDE0gIknfCvi<0GaBAkPesBr(tn%'
'e(WjF$FN>Oo_lIgA<|S#pqf#oI=!J)VYVvlg65gGi^8&wdA|n1d6v!F^usDbPjvV6tFik{g@plv@_15m>lPkHEE4BHW$ENbdTXi&yLi#rjI2WI'
'=o%l_ur)s1d0weO53a7GXe^0%@s1HQBFR~J&r|9>n{DWE$xrpv^o!I+qJ-+n-l@yhY5V-0us+&y^y&9xK(=hF$zojWzaSP98!<_%iH_0K2<ozR'
'kv@4(f|s4(DdOt-{;KW5!%|^vwU^2!8OSQh+{x74P-ZT6?CU&UUaaGbgm10+!m`G3k=>F_r4^YDL)0vi!XQt`1~kj>TBt$I=47`#i;FVJy8sTw'
'D#_8R`U?I4;yTaJ?rqAdk)lUt!`LF)T^9pyp8#D7*-|YKf<fe%7JNI|eg><%NSE#Xm;G%<T%SK3y+3{Z>Ez?*lMnC5umAMz`#+CQk6!~#Ii(#w'
'=!a~?Wny2&0a~WuWZ%f;SP9Zx5GO@KLRjgAv)K#<f2|zulRCV;&x1$X<=!Mo=NejIP?0@xQ*PxK1`)=kuu50kA9SU2TjFq<K(f%=cLa=c!0%Vp'
'BOgk!VGqEY*#YS>a=M9ljg66rLTSB3yO1{7pf4{x8oef-jgUfdSM=Dz9;p)^ShCDyBe&}C$f*?+OW(QsPIWdA{fBhsyKH)&ELP=Rc;;xlaqoVo'
'PPns_PvD5tLAMp*bb|Cz`HP$p3v@!i-W<IQlm(d$0R(C`2PX@PW)owyYHlZJ9|wy%^ssRVnb-!xrO~1T2E=|0iFI$|dsnrNhEgHze|V-D!iquR'
'HxV&@xGoOBFsS&NOqNq`g9(`cKcYE4k@0E;HK`q&Wuez~%E=BQ4yLUk)e3#miB1>mg}UB4eCfr^sOd%}$aZAQ9K$rt4Lrjm3Y`KE){IX83yXC}'
'`;gJ6jJOF4L8pYJ(1OE$hP65s>O}Pp0^$f}M%abB8V36$Z3b-v8$prh9XKT^wAAK;-C)1NoNk37=}qU%nCxy!@y{_P3=P~>g}GUr;~AD9q~QpY'
'<$6I*NTk=YP_yWPs_okYMudOeQ&WTf;dDbr_r?u~C6?jGgXZv4<9{xJ%N(DhxX6O)Ji8!2CP#c#T#|iO^qjWxN3!<U6?=3|B55&+=gIK!Wq;5;'
'pvV8<GKbOxX;@suiJ#x~JrB_Y7wrI-C8?Bo0X=My`qc0D$lQ$J9!<dUI*XpzDq+uU3C~ZVhx-s$1vu>-d6u?bq9}NVt3e91K{Y>9pD6O|G5tYU'
'ugR+TVK)$9C-Nm=<+fqRDU+3{1ETm%?_dT%|C=H-;1AFC#71BWWc8KFWS+#!F{$a=*C<^Z_U5kbeV+7+^?5rFNB<}8{@CySu66d5w8l|DRh4<7'
'QX+?~(8HRpN~ShjIJz2s1OnObJA=YT(4Z3iQSi<b7rMgy*g9*pwMyF(keMFhb-=D__5xzBqk#<X%vi{)<c`{GV+m2!JF|FMVRAJ0mwC|A6^0vt'
'iKWoWAim*y#@-CD0<apm0Y}~cf-N|czql3Mv-^VxPbT)Scfx=ubqQc?IW%rFdt3^S1b-9L(#i#=_DaTYgDyW*Cxyz}{md1-6&*CiQY+QSRm$>n'
'6=T_H&By6-%%or|P}xFHg1#5@eHVe1zDLCWn2eAHBIY~rXPxi(n){`NbIaKb&+l<eCQRMHhG#e9Vv^<m%+%<cGBr--$)qF`0PosKQ`Ck{1y#*D'
'9fZ*v_e@0GEg-`qR?XQsVjXKWRR?EoV+?Lsl)t4*^-=-snNrU$3-q<)t8p&VF*y^M6mSqULz&!ay_}S5B^(v&Me}HtD?+CS%@lZh)ozD>z?Z+m'
'e+FU0d^AtBoB;`50##)w#;H{RF$z%;8K0-<Ef}8pn<9N4$UQIvXvY|)Sr>>;oBW3u0u*CZ^)C~x4G5QtYzpcxh*5LLPyAZH_uJ<-l^|+}EaP)x'
'26<1A|2Eh92~<3M4iBqlqu|gW7c;<g=np_nlH2ZqNOl`I*9Flwg#jH3$2nDb(6IC8u+hO&jYRsDe$;I`9dwql^=%G5t7L5iMKW`y>Z3~Wd8YVX'
'rSe`*rKMWQNZ<cJA+7BIaH*A4>Wb?)CAVQ==|O(?odx&`z~qddA6664ZB^KzYX0{_)yPwr%z^1zz@EVb^oFN!+vGEcf_mf`3~-pBV;E2YkXnc8'
'@}IF{sNmYUqbe@RB;vO$U8q?Q!&r(gJ;$rQ%M36tgYf=-&oC&O0HUT_K?ObKdNof*f?CZ#mOw#O*Ljhl7)BGRp0n+Wy#2kbJgm=ihu-O~$JOrB'
'3P{zb^aN)Z$vSsB4J~Ne+vd8F-FW09E~+1=Y^T<VjD~^f0HK*y9nbXh=SJaWl1z!zcokRzDSK4e>ICS~71e`o0PD4@Sw#WxSwnXG08Dd(%=gXc'
'fR4m<k{5}q4boKWv3qGUi##4z*D*tjN8%kv`)3IH89Ml)7noGVuV1Uj`|KP>KP>9tku%ggk0Qe89Z|Jz1v}#9_O$0{=b8z+**UC7weF(<H-1)M'
'*&c3E+qW(KGjmVZ%O%@a_07{};ss~*;ni4S1o%o&OVDaApKIulpwA!oBk<K#UDp<{D(uy1dR7iUG19QPaYC!XsR6@5)opj&)#drBt!>7I#_TJ6'
'SamLIF{FPh94Q8fV<Eetdv?_}z~MC}(yhks0TUi^4NSfWp=*?;H4BW#su^g)Z{T@Tn{Fzzk&PkYt?B{hVm{$+tgBHOxGc<UJ)e(7mh_6KQ9!~c'
'msV09vd>!LQ$*%GPl0iHiXthhw^hSY@Iu=?0TCK2b8SIMW+)94389{XSdJc<R3td)IKLZTrv+?Vfo{xUHk*YcSg0YO4IB3|$6y)-y;K7=b`VPE'
'cc%2L%8Ulj%u{QRSOKjK3FZVmiYV%8NdJ12ISeNV6$icN27Bc0bvM{Y8OhUvf-0v5w{?h(x@{qydggD8e7kX_hv5pCMhsH}@{xRX6-#uU6mJ51'
'1bA=^=zWYP#CSJRW>C>r={#8`vlRMx7hmx$nN;DpL@TnHn*cwZkdYjWKVY|WEP};XdAh(*J<B-HNeR~TbyKm{!+!6@fp6`ug`>}tSuq@idgRqX'
'kW=W>&Y=P37*ND`f{|IEda(j*ndIXcI>=)jjhM7sMXk00VYXVp-KwvH{R4twBz6i$QKo+8preNXDt>kVD}ESY>nG#c;Ki8_X7D3mjJLTy=p||A'
'WmDy?5d6EQ_Fla7LNI;|C_c~15@VS73*HX#u=(HJ!Mo?r8eWpo#Bjf{GTTT%3x~gLDB(Le$t?<du}_JsKKke!O{YN|ke;inG{9UtgM)z=d^ho('
'aNbu*vQkG!V3dGL!Dj$CH3j*_dy4R5ouub4n)cB-O^bB80thM}c%1N4J5r5D2tS?Lsp1EhS#~A<#r)ucV=gI@V}xLGD2^stmgt5o#e@|WOj|t)'
'Ra6eI=4si^!}cG3KOFUb^@p?04+ScSb)y|^k0dRa93O>t8&<s`_l){k-8;!@x-HvK*_bsoG#czb=la|()veY;W=l0l{m+DUk>Og8)oN)H>);yo'
'z8T9H3*3NYt^f{(Ar7Dk<`|_kpU+0DvEOUNG`Mg>>~uHc0OHj&vUmy92m6B4JI7lkv@LC(#@##_h8Q|wSvD1_@qWgM*|tXRi8HDRDSP0#QD3!K'
'B|wLyW}iT$Mwy1e%U*vQ^l&_tB&Uc`EUy5i4qZnz;2NhFi3R!lX8^x1`)9s_V^hiG<&vUGEceo(16L;a>EL;SmJsX4ZDx=*=kZwBw$|WjaL_y0'
'3hG>ZH?R{K8PAhh<IoM5M@>UK-vRFuLE}pRo(KH}AkUE#)~iRrJXw=Bz%MGx2z(8I{dI5@i0O@Okiy<LBrjqplyZI-h&O@oWXUw(4!UM*x--<X'
'qJAiY8n7{FWs+yBUZBj#&=09q1b}D)7PDfIZOoQ1cFT-K+MrBJ(zhjFN%YEeiLnHgiTVWEUlhHnHnlkBTElIW%#!r<1tCx>rZ*NIF&cGFybvm*'
'!2hS{i!rZ%cQRuP^`Ykt0g>xH;+Pe)+i`MDS`_BMRCvn<fq!=E<2@)AQz6EIjxNDb&glLW45<#=3>Y^683jfMjLRXl)Du<Bbv9owlA*r6nBDgH'
'QKg{ao{?CQ^{stB?p&)F&`GO2w3Q`rYz|mo8tS0FfGAiwZvq(t@>p{(57z9H-D=f}I#@&X#dvofV^~zt4Gq;7qkz5OL;I&uu3GzH_0@Q<IQp^p'
'dFr)xUl?`R2{y}7G)Mpn?cEN)5qRcsz-wGW33rK>_#8q@HG$`vq>|tUQK-7Qsfj+b@Kh;+#fr5uXuEfC!k~9B+Z6Q{j@9Lv05;SV2v=%KbVk_h'
'si#N;Q2mhNOH^N0ue>_2joGC-jm8}!?@^J&`Q#GSb!`&u#W({GLS^j27X$Rdu+0d=@Z@5NDre*Fa&H2l5?}yMj2(zv9T3Aj0lAVbn=#H`z?~+B'
'1+yHGwpJ`y79Bs_Abp|n^{{=Q<<mBq_To4@W-v{1JAEVDse){wxd5zNT8_tfDKOKt%6Db@B&sw=biX=q&#fJ#Dw0`^)vSgTy2_Ff=d+&lSVg38'
'-jAJDi03Wa?o)u#w#lOYI{1#Z8@&zEizV^^g+9uAw8zEX={Mg;9HVWH2Ca7i;JrM<NO3aLY-zTL*9*c7tN}>|r^|RncUV<uZMozw-~h!K(t5F$'
'7}a<!0zH|{p&Qmk!U4MhkzOQd>rOu1*pXt0By^;yhJbRF4$oaKdyinX99r+}S&yQOYp2GxU0vQz*NasfzYf_^%#{UD>2G%{Uc3Yj9x+-g9S8&!'
'av>OW47SWeV8TmpK3|}>Bd(R-HvjG0cjIqAy=6NLr`X%KNAHftr^lbZJN{G^($-r#e&k+gkCc|<(Zl3G?pp`??!%kow>lhUSK(b_Ollnefd+@^'
'WRaAY+0+;s2U0=Yk^=(<>_R#LLPHv%I}TZk4nW3m0mmF+9pIhWdCjWJRDtkSY{qZkxg?9UEQ0k?{I$d=Tw^U`HT+XPdi(anU&e2aKOcX3ck=$^'
'^z+H<@u%b0A7DHFO8$=+1t#n$i!_po9r8t{qGi4&o$;METgK1E+B2L8OG@bVmE=X+J~9S1;DFtX>J8cp57A)Tc<W#~p&`u{BC_5eh*kqqN|#*;'
'OB?+0<F^5lVZ4PxUhU8E#l@Tx6R4Ue6y<^F8`lT4kgdutnly3fXET7npT1#e>IKr4Az&;3%M=>hAw7-BE(&lct}#W=P@BzoE3&nr24{w7OU25M'
'f4qamW3GiT)e;@-<`Un(i;=Daiouz)N91ZfpIfcP&_)i(@ZDS3G~_*jd%k1QZjmgq{H|v$q%n+|X|oEJkCSqE3yP0#K!T2vzb><P2-x3b`D@^S'
';`!Tm5k38k=3wcM7-3zt@{ov7bO3zr^JeF2*DG1~d@bYNvqy<XO*36JuTuOlo=&HByao?c_yRW2s`0(8YUcrS@idMjWrr3k{EjRkjLllo9!>*J'
'RRWVum&vrPI*(wJP^HbGvdA~l3eb!na)H0P0iY>dGcJl3KLrsC<gZ1%!WdOC^c!xh+0r`qChKX8aYXd$wq4T-qoYwsmBv%@hQ)rk5wZo@e_?>g'
'ycW0dq0KxyutwRqnQuQFy%`_AI{B>8pp7l-_|a7%)=$pXGx7>B{?T&!3jJwc6N)fVly{o{SjT{{ek4RPX0$@ntvA)6kD%7l<@oSCEq%em)aQJ8'
'PGu;`!5Tn@mUd|E&elC=9>v;*U0u=SNE8`{%h_S$)srd@a@Ba6t)V-v+B6?eR3u|jd#eoC5)*zRjtFbldYG#Kw@vN>6lgj;?7~O`6)MTVIbVT%'
'h>w=+JOO8Rk`^S@k@G<1F@PUt5r$r@y5-D2d1B17V*(eF@rz_h9x*yt7O7MWlI&r~XfQxyiRcZ3QGNFJ{!I(>Y>qw}VQ~j|x$z=Y+08)71^}i~'
'f6dk$!(7%)`M0x`XfOCAoYFF>Zt%@iMd6mVnb{W8Ufg92Xb_14Gp1j`DWz@cH8kVX%DLhDXsPs~;L3Bt3pIOVFWjombp)E4ER&nDw$sL2hv)kT'
'+nj8Xx*UB!et-NI4iUtzwJ-bRTU!%EF5@CD%e<}04rcRMt>$-Qkhjo8XR(@-pS<TVkS`+{vFX}9Pu+&iR-o3TnVe3?0>y|AdAYV>HRhnlsMVk('
'0xPeI(jUf6t%A&mvXG$iCB;ya4I$mC*~_<HneqATF*R)p0jyfCYL1NV&iu8smoxXE&i9ss^yG<}O>-F=zJ~VCE%m0^m?NZk15m1uX8}B<UVS|4'
'^1u~+{MzWg{+-rFjhnbqni5`m=%Hdv_f<0L?w<`jFrr+i^g)zpF}l$xD6Z1gn4<Q_b2Tfm<aNkl{al;$Bk~=!&W^om+M^0ecPzNOuCC2x!~Cm-'
'^qn<xO&_>Gbv{sqnj?LZCcLC1V8;rV7)OCxNLyC6f|!xq>c-ViB+Xs@aaotV2&N{zNj66vbGwfFAIj6yw5zftvWzD&(8;!~@9|B(QG4L7%0MSx'
'8%d@f`uAU_Za6BgNTr~|V`RCc8Z4YUPIWu09aqpF6dn|o9s(GZ{4PK#pA|zg(mgb<k;!n{3%vZo{2KFs67^*4hedZRRcpiq6`fviib60d4MYpB'
'BlR7<e*<un%vJ70QBopLG)s~x%Fo8S!P29ZJSLhF#k#iMLPIZGyHyclU~^d!J8e=iRDKOSM-0P9Ukrhd^8EPSG4Vr}a3%5&MvRVq^j&fw^Vb2X'
'2vFjR_+)g29p_Avs4c=}TIWfXVT8Cm^|WU<D6k0Qsd3Qmyqxpomk+CWg`#dIMGwX|Bt4tFW*z95GLnobo}qN=GHm#PgBw~GX`%8GmDU~+g4Sxs'
'4EXa^6{*nTnMMB}aBjnf$oXE)x}oB3iXpFV6NcUvnw0DQ#t?+)FyO*C6Gett-&mm66<hyd8aR~#v6<~Nc6CG@An>Z8%M%YA@a4je+XQKM+p0Pj'
'IZb=5Zpz?X#XUAer$a?xf<4<;tIbn222i`{la}GA_2>dkd`9Y{Y~X3>hM5QY$V3<)bKo(ubuqtFSdj1R_5GG^m)ry}m_b`TY5<sq&(k3MDwyC+'
'KH$W3HcKdqxE4H%_4(f^bn-<mRD-n4N#)^FS@1;LL{ra)k_#$hTta7-okTBqjfhw*+Lha>9JAP<UcDX3NIdXBy;&DCB7Flzkc(mE{P1<ql;ERj'
'^FFEg{$iGlGf)l}=m7=)+*Ipn56m}}<aVW;shSiNAT6Vf6-ed$Z4=3}G4O}h8&0lBMYgpi24ueK%Cu}-LmToa_ARi+OA+1M`UrsHK8DrIyTQGn'
'v7Xk0uzMFh)Th3V^)_ByBV9a#lDv~aRoeM_UwjzDNJo=k&qL|VH`Rs@9m)h4P(8sV#f8>$r<}^!lEX!j#Z&{SikEk7cEIA*Gp(K<xPncD!naDt'
'c)jRZ84k2T=6&jx378u6!zZ^0-{6HlbADa(g^J^8_8bM?sGgqqoyOW|rO53>g|BHQi;BZk;jfTP7g)8&72D}@GG9;8)d!?65sd@&aGlRZs@o!1'
'*FPYjq4!W+Owv>}DMq0~)Nwva|6t6Z;92|X(`1yv*DO?A>b-o~04h)od2<uF@n)2biN&oGNsBZK{F`F$nW&ECQKU!f>B7T+w{0m2FV}^Mg|pP4'
'CkcO8zW(s()A4Hz))#y_{?BhuJ{`a51%J)fIo)0&0j{JEp~KZQdQIdm{G?dRiKJLn#7ZSw7p*MOu~l5cIfnt8%Gt)IKT7gKn+9t}()8oA2@spO'
'd=*SHiPOCRey*{b8E_ILQ!IcVKr`Gdh=F&sr~eHag3*C@h%32sAj62U^qaa6)`~(~fdp2?H&IUj0FebC&IAULT*u2&<WRYmGMAvK8@)7xppP!P'
'+z<QDW26CWW4#En85!>>$4+ghRy1$LXH39MLf5MgZ~m$jKq`{Lc``$jFO3RO!btM6h>VN{s&I23uNiL8Nt}d9v<)MsU(V?d%kQdVrKC=B9F^lU'
'8emk>v*2mt8TeR(^!6gY!<-0Z4ze(Dl!G6ZA&1#Op=~tS$E>v;rq^=6uTuw&3lvX_X}5!CCTfuLuSn@vij?x)Iy=o?>LO-!8{`cJjvunpLRWRJ'
';k^y#;8X-X9JVF=+u#0HqZX<q9p&o(K&kd01~{P3AATsFn4zvPYCwEZ459EzXGc3w6<YdVW_6=V<ztTm5xq5bXF(W-8zcDv+q>^Q`C;Q3$*DbU'
'u8YT#zV(o8UA6liG%e;(UygpE7CXYOXQ>)_vuJAv#^t1?-EvBUzyq6Cdq>%HRM!+<vI@ZQFg1oGJvKsH>LxrS$m7y4Y>6+7crzDbtN^jQ6LWcV'
'f@KtXJbHyKH}*IRz{jgN&8s|S_=}E)@1~w#OIjygvh70Jwgpp7eDUS82WK>(t+%#0vXgy+?bmP<22GfiVmMP2OTGR`&44Ew4W0vT+j#1khBnqN'
'k$S3%sKjd%5x#K|L~{xlJsBNEJ|-4FBLSTF29w`X>@oGV3oABg<&-|n!HnCHWf%fgN;pV+zL=xUS^<sEyL`{jq11`xQ0fHqyA*P0un(3_RpC1d'
'`dVp;ssg_bKBLPzrvpbkO`!`hOg!C_OY$>RY!Pd6iX&o$X%Vo_8MqiUQ#70W+y38h*Z&sohkp}dR3wQ~6H8+%p|R*an62zNLm!2#FdR!IvTF&8'
'%l`geoOZ=0;Xn36drJR#gCWU?Z6e!VHlsx4V?2*3W<CW776`bxgvDfrh5CYXbXDMbn@z@AyN~QbFR=JUrTkU#0B+EHc&X-#<|AUz*|Wgg0Bxp%'
'oMyptCkQ#hu?y}rg`rv_r^;EK&>9Qs7Dmd3U(e*ki6OD6RwqZahmO{FI=6If2z?EuB9FLM#U0`8IYR4!>JkFeeg~)<03iX6dYN@`*I=LEd?Dxo'
'+fDL<TyYUUY7@O}Dqu&mgbt+Im&8#9!!KnF-=I8pm#S%{X+RCjTP?9xERW3J4oBI%ovITL#QFO0wd*t%jR5WW$!tT5_C6>Nb_3*U%MV@D7tx^|'
'*iQ4}`SbbNt7HVOHnph@2bAh__-m{5nex-Gy*PVV=XMZ+oxMB7jqcqyPXB%;2=9<8rEw_P-rdlk_7IHnb{KH`m7lw58<}J(RaEnQoz5tnm2q{<'
'XK{>ypNqQMH^kcrQp&P#rqaZczW_QRmzOC=ocza{EM{Z=r9G?RYe^|4fRIeC5a$qfC@BXDMI@f5OH%Qk$EfRdRlK2gzNAr*|78%af#FvU(9kqF'
'_f=z3&n1{htx4^vNlgOhld$Fgl>V~w=6Y3qXGcyL3U@9J)wO2Hr<0o0-qeBi@C-uHQ|Yx*Yh$*v*?H6r-1&1Ia*gaG&XLPEF&3a;_RvM)wQev|'
'_%QNRcdF9Zwv(aJUc>DJt+RmSF7JIRYeJ8PlP%><&=82&Qd;f!F8BG3_cHTej!bJTJ;9ajvO+v(v!zM1m!6>n@J2YT-fxQxY{37P!1zY7Cw2Q3'
'PB;V$VH%BjjC6MgPc?6qyb)Asg#fI88(J?Y#1$;R4zx~F81b*ONU|JM#z9a5O`<k@;3y_gszRQR1_y^6#g?YT4CvuYlO%~#tCmn|{&u0yyq8qJ'
'%;qb#u%!+DoUO#><tn*jLyyqKbC&!Ov8QxY`8-n`k35s_7OxmN`rFz)uE95=bGPCeBhN)o#P`7|Kg5$5X*bKL=C;g6EH)!uQ-;=!IFzoQ^-?;6'
'Yjq)p!@BDQ$9RKA=$Z<-?dqE-Gs+)b4C_FE96Q2jv|v-GOnGi;MY-F!GREPMm%ZRU*;k2@_<zB%F33s_C}x>rNK$2(LKi&fgUhSwHVfc<g_I9E'
'M!7m-twj~HithJ}N_CBa3y+mKIvHejQGoYbq<?t-_OFtTNnJ2#2-epc_3cv~V38$RE^+n&OrV3ZXK|T;td2r@OuF#*o@g679RX$2rX0OE5IX$3'
'00uqS<8weJYqV!@6z|T>1(xMK>MAa%QAw|<{^M#PKv|CHay1i>ih`gErgnz`M)Gtf!Xn?5!n9$wZ!)3Xv_L9ciNqtRb@vx+&|`=|TP}<Q!hBrq'
'B^Lo0$2#jJ21DmO7ZGVF2z6ng=*)wLob@CN4}Dza%15F^kio|iQ&i*cy~H@n>I#T1R@^R}i<SZqP%*p1x%n=5C>pyy&t3DZcj~J}S6a7Zg^L4J'
'kung4pstC1v(cnBXp7Cv9o`n=fz$>aG|N4{sOr=dkga>Z`3eW2zSNQh7h9V!l92ub48KHa00tNZU05v1;~2G^c1^Cz7k6^g3!Ez9(Y(mi6}7U%'
'iL7Ai7P}H@d|Sz54FJbcdaB4g2DAwfA+(_?v<lwC=}sABaH?h&0nSKKM~7GrDFTwYJtA;>h6J=%i9dV7m+k>`zo(_1JrsHN)S2BQVaE>?RTX{0'
'@znn}fFWjoE>*yh0GJ413;1#rTW92eM;n0p91V&j?+hjOk>#!TsW&sKLfG*76H)C2e@>EBwRo5*0kgj8$b=X@v#{b5>Z@NIP$MdfN&mP$FMxkQ'
'd^xLv9001FXBTVFU=gLoQ|#wD2~NNH9!3GUYk4uh8}QU;6jwe*>_1!26};n)$4td4rh!S?D!#+aTJ^3@X6T=;SP(^t5Cmjd1-vRQg#ii<HE?4!'
'Bu%)2h|5<TRr>~E5*8CCVl+VvKy5Iy@ZMqfHa{%uWteRhCZ4TrZ_746?f7cE!l*VNv071XHgr=9*vNRnVQb*cXO2>BV@9K(9e>}~=XZKZQ2}gJ'
')vG|ZoKO|_*C~Sme&lVf;S;*{HqE8n+OA>P=BuF0O2mh&Bmo9)HDAN-Vj!9#IA6mFEw@x2a|u1wCW@HWhlS3{iY@g+00-(5y9TfMmxGf*>QE+I'
'fRx38E+liULCE+*p@6QYAFR`HB*no<G8YW`-Dlz1CUTK^%j!zm=!JBvp?INi(06**ox*A@c^cuawnf*}&`=5J2cxEjo3lE>IT2bzNP%usq}jss'
'dcPwG`6@s!?Eq~NGCa|193Z;XnFeMJxCkBB0V_Q3zLfMjt3+G6A^R!J6GhsYVCa_0mO}%^tK-jqIX-@`upjFA@|90GV$TU-(dY=(5au^znk+#Y'
'*<=L=2~Y;3EN?OsIa<tXeSSy4D|7t+$y)egSi2c|I%Kq@#52itTS?Or%adO4%CK-LnAy5SIHz!4l7POB<g_`faGFZwsh>h6!;DptO?~`g(Bn#p'
'7(!=nnieGU?-e)cD%lg;12jEf<tbk5%M>|so(I{yv+`}j0F7H4Q4|QhMz^kZD{29(4_X~B8vd+g4$H{dZgh0tA{x>faK;)*#WRwpXPxMv6SaVW'
'sTIIy@){3PYj8t)rWV30jH}gQ4aAL35#_&M5?h>_R^ounetgsV<Vk#2*t4>fCl6n;r$IaX?&S2_qqo7E<I|Hrz7IYgot_R{WrN2$6!EHoZ1Eh<'
'v&og$tSb~CsDdn2)^8+*l5mw)mg)>GN^HuLn>;P)Xe@koSK%8D5LGEHyuuoe7%3qol)8*dmXO#dUWH<+El}_v?-AZW3{v#ee|BQNa2JLWxxUNv'
';<9TP$?G%Dk^6yD#PXuR5YF4t5>)C&@@nJoUyo0V6m3-z8d1_5Pm!qdl`@SDbeuWmGNWTd8Hx?j;cxM`x}1$w6oM}fhoahem_3m)Oclo`R^T>k'
'F4eO6tX*x()<-v~oiN?3HtJ)!ch;jLHMnv{dfC1AVGpqOQAfcHjg@O!vz)5@GBB#`!K@x`pFR^?8!PV4)V4|lR0!kUc*E+r5Z4@B6+aK?>?_9)'
'QB8{CDz3@gZMy*zTPz(MHdrh*?zWv7ZHr3Xx6g8;rOsmgzUyb(*Bya}B*vTR=*Jp0a|}Zck7GCP?r8YFIp$>*{%q&=)m)Q)isk<akV|8jtjb}|'
'er?5E;Xq6^htX|uR|GJcBi_9_dj03)_iyMj8+IU0D>SSF8DV_ZY(9?{aTpjIQl(N1_u^H$hru}-{W!FhvkowcL3A4Pc;H8GJWODFpORi6yjR6G'
')ypkcmdJVy??WOlnC+Lgm17Hb=w3cg;sv}A50H>;?7~{WO>ws*<4?X1qWUTj2~-$u3)|xE=jN=*M~pxS*kcd{hwW$mC;)D=$+M-w>q+fHkB;;h'
'ri4n_a$UIQVCqwOF6*jfq1q996P_f@=zb%Q)Xa#c*Bjm9D)Tb{3Btgh_=B)Pl?Pa^_v__$!cFsg{4_w-j!I~Rk7pef#{*1j_Iy_jKai<_NhX+u'
'AM<Rwo`@EpveoUO=HYShxoc2rAIJK{+=mn;3)(kM33g-tBm<O~Zi_=B@)99yPDL)o9(Mi^jZI2*vApd}{SP{ClS`D>$l~RGbB5vTzw{g`{u8(v'
'6&1jxKH^>89D$_Xsza=*kR(4P`|Ky~E0SmGbo*hf@xPdTnz`oZ595M&bHM&~zl`DC$?X1wTvnCUUZWchl~eIMUW54}u`;szGRx}*=v-VScN}Jz'
'lnC&P&3Gk|0Gco9#293i7hjNV?hGSABXV+r=A$>Yoj@Lw4t9)4=A5RrnwJ&HeEa0g?B6%>vXmeeW{+~YyZ=Is%S;%&Wijmp;oSI^^xz0B^LQoc'
'xn(T>+sG&^`KS3!)wsR(qD&8&rzA$-Op>5Fc}kX<FRgX1Qd3i!94Xqd6HkvZqV5>&6eclxnUt4<;3NFo>{>w=ZCsw<0PFOScNnw(0Y4C5gr^8q'
'6dWW9Y6aCR69FJn;l_J>m0v>-em#cr%P_orR68_7y2_%!Q*d~O;kul4U)I8b%K%(U?q&-}W3|N?&U~XRZb5Qjdt0B&m9N=VTol$;-fzsq!Erwb'
'7TWXlJdg2g;HF7PeQ(HrpO$N2L9{fb*BvDn^Lbi~0Y_T<A;VJ|ELdaAWROvS#P3et9Df)efB*UT{prbv_ovtm-GX!S3N#I1#`5Id(I1bEf@<c)'
'dR{iD=M1gcj4okI3InZ1nqSPb^LF?obRdgjY#<cER1iDy91VEY0-V^KRugi)2WzIUN|%o~q;{q(<0hP!$5fgPQc|)_rM(@aW9emFT*9<4JletY'
'7j1S0gYk;dVY}1maX2;ncA4Bx(+kY?h}5U=a-9m7s196?UAo2gLjoMPE4Mt_cTw!qb&ia?lYfMYXd*D+b;j>*873qHy@ZVT6nWw8!JN>w?hU%*'
')^=!8SgTA_s@5lyu2C)XWjF^O5;)6R4fR|_z#j_?);q)q4(a*YJ6iKyt)dt#lch=ZYU6?wdUgTTBpY5o0ga?&4H`9bRC_NO;-nbm-G1J8!^{{y'
'ri#RKJQ1m8r7<V(1~ERC0oT%*yQogoI9}woEA<C3PZ!G!Svbk#u>%<z<;l&j`UPuY{d6xv6h@lNE-G<c!ujyY6QpirC9te++fcK&N^a2=5cy}c'
'LpDRpemwsE`0eTV=;O)w&&PkYZvaeEhABm;sq{8JkMI0#oqT`v>gcZzZ*09RfH^JL#aiQW3d|pRd-(SK>BqN6r+*qBy?%Xsdb+b0?BpiBN{Qd)'
'H#WyrEn71MvYCULst_MOBn-Y!ds_vKFm$#va)U+awBlaXq>+bGvw4aZ1VdU347*u*hmX-^g-$@lDVi&0jK0nx(O}|qEdHHib~M!?(s|H!73?Y{'
'H+>v(8;49Wu+gr&JraD<3R)~Q)VS+soz9udURl2HJQpOg(vyb|qI-48oI~LNoZ69?-_fdhc)6Ohy)@Xs$?2k_7Gb8=HdMOJ>0V(~Vjm>1fcMHw'
'#Vu$YlH<PrFu~lwQVOx(hhhzsBlXun6RL1RV^KEYACg{s<1v!z@fb>6mgP!xFi-@}Hw7b7e>o_TQoX3K?&77jE+`9vdb)Aez=^@r`imNvY`B*3'
'e^a!~Xzc?02CjS{n%sIyj5}PKb1~RMt+;g4gF~~&u|N0ZlsB@pjn(&6RaXsASK}gMphgr0m%vg2i5%XCN9*!3%OyJj?8gIVzClhZj~Cd4*!3zi'
'=MYiL3K98$j}l&Oz>!>*+LsB@1R%u{HJ@6)cR>0*_mn?ae^*81bo6$0tf4&FqzW@lycb(GVRf06ncvk-fXJ!W6pspEkdP2_HI=Is)gnnkag8RY'
'!%;^9)47`xi9^wTC-{A^-+u`U*!8|{3c+S_+?4%!PN}06FS@)6JDdi3u2M+b;YZAm$^Z&YQE-N8Vl8-t)-x@TDMSPyGv9Wr+ajdjFgk;c#leN&'
'k#olRU2tzSjj2s5oofrR3^ux6UM*pAXifyRKhl};DA=~#g*a<dZBvm?Of9d5Ws?3Hi&0*mgWIN%jcWC(`te+voP|96fB?{(d^&~sHQbZCi}#T#'
'+lxmHlEDux;zJh=IagzGn(vZKS>WE3NM|8qv2krOkttSkA<_xjqM`XSsw6(~{c<EZdr4V((*oNFUEwNeOqQB_Wi0{A_!ZeQR3LK4ppmxU|8oP}'
'Saj|uqS@8b{u>c&1?V+u9r~!XA&xv$5_vSemQIM;;*GWt><5r)w`S;4D<qzjq6u)kLwC1L0Ufrt(^?F!A#A}wQXe_NK-J)XwT)Eq2uCcFjdhMD'
'CWDV=ID}c?KR+GOH>tVzy6!S<2ti@4!%mXTZh<Ct7H+{n@W^Pc7cLd3mm?cSE_Uw;(uyC4^iUgI8PVV4uJjwp7!jls++0lj=nVTNIp?cmdiSLy'
'KoK>dK5|&^alJPWrC~cr2)DV5hH$B$j)<uHC=*FHyfx{HEWm5IU4C0B`fNfIp0c|`@Ua^B&qZz?I+dS?0#)<-w<1BZQ4gX)JGEGkQDAK)PE8or'
'5)IXGi0JSH_O#T}Eb`65*K-ulx+iaf`)b53&xo0g*Fs?U?<WTCqzHmknu!9@I~QBRZwrg($;BElXajE4Z8RW8lkv8@VdEd&4c&EMh#BV0o1};B'
'h*E*Y4os<vc~GUEkAWvGu61mp7>AbBUWMPPW5c|}c80j#bQngB{QI}YBy7ZLia_T2ldNM_`%-M%JZgnvn`Y@N`}4B@=3f6hVP6dy+tKf1?uCTw'
'qHT=OtfA9<L8QHp+jM{i!4dwE`ya!JgUmvr!1G6fxEgoi)KX8kOcVk$1%>r%hdvrs&_Da~&(>Bj{M61y`n|*7Zf!;4QZ+X8{MkcWqDWWU1_Uj<'
'+&!&4y}0zZ^ZejZ4FM>n4Grmbo*T`m%sAB4twe|E-Kj66PE$WF6C7m)Ix1=pyGgaTTq2@PR?UJEKr+ipWp?o-3Y<hM)m<Y4Al>_X;y$NuF=5A9'
')Zmv?^DmP7FLk3hOUlWmjN5k;7Ri77O5$9=V38BSKYL<ppfp8QiQeK0Ej)lhK%Xp5@yPyS<Lt^bE0jxam@NlTt*$%y5}ccqQq2kEk4{gIKaby@'
'y!v$X$>FWIAGWho25Vx;rN%&6Pvl0;URmQpZ4oci8Ex#2o44)0S!f2hSTC5SVD3C+RfTCl2{PGV(m&64C+}shr>{SJI>wpyd;4gAq_`#rdQb#&'
'o}gCOedOLi8>^Ei$g35Op$t&S<kpFzM0PF|$Al6e)l<-n26}h#N>cWgR^Q^s&@gc5$xUQ7rt=h84=HDn3CVOTiK-UGqIw%qgumXH*Wx&|HW6(w'
'%-<AsP+hhvJ_Yz&;LJs}xkS^kaN~~CFrL7lw(*1*OK#5<#T7@oH`dzd#7cG!r!nS6TqRRS0<Rk+8G*%j!M+UEYqHR@%vwlBa!!A9V0F3wv@O;U'
'W44_pIxa*sgGt9b<OZC$eLP}T^1z<8p^0r%!kgEF(h|k5t@*aGWNIP7#T>XXB;<(NH8#k&W72{FCG}cUyzXul^0Uq?Sv@85^de=;*3pFT!|kB+'
'Bp9=Vp?Yd*^S3oc?5>}rfWUwUA(i>UJn9p_-zP&M>v)}w2G7piqcf_G959{hWl5lq21nHZYj|N4s!2I@)MGPwXMMsAXU{nMXI`mn;%<i_Sxrof'
'HJ`hK7D^`0esxTNPQ4V1Utc?U5*I9r$8)qYyu<YD)3Dk_54;d<@NZ)~JWtRvpG76u&@896%&j#BD|`EUha9pVT`hlT$qbeGg)b9=PNKt38a64?'
'`}rUP;L3IpdLkS{hV0bts8zjf!BM2;qBjh@##?cAGPtI_rnEdWIWhM{RR{fp>1M3654D1)Ac4lSWUMS8jqz}=rpVr+wQaSUd}Ug_#p<wC<^D4f'
'6SB@NQaQJH@U7n8R~M}Y<L2fOG^jqtx>RwuoLH-B)j#V*)jIn7CTn6{#<s$B-k~VkMs9Fiq(3%#ztzpOS|k9{1vv6YDnlDOVPaq$-_3RNqLLP+'
')npZbHdP-_-YO$1(#-pdQeY=~-$R$twy{pA4v@fS+tOTei5>{S>Fmzu2({%x0m#vLGA6T{_SPLT&nj<my-agHk%y$ssMAUfLYyVqBAFTw`{!k0'
'=L)KA6um=N;;J|B<9ZCuTCerO<jLPTixXn6dk^REs~i)L89qQ5+@xhQErv;&H;%iPA6#aJF<_`|A}d19SQv_}i<pR`dgXA6zYbnw!f406iB88J'
'<8zgf_bZNsCn0bs*_7-S3Y%!N=j7~F?)Hl5P)=a7?`(bHFh?C-3NrvU6EP1d_hBrD)9%I7DY92Poakq*_gTQ}R%P&FgU&4*{Z%0Ag>*Bxw}|4P'
'_iVOtvCS4HT}zfidMzg)m%FO!lj}at(em0FRlJ;KgjuJYAPfStD~b|Uk)X|^qgGQhcXfd%14402`1TM>HJG72$ZD6-aH_&gRLnqBk%XzTt6RL1'
'sc^@(SiL-S3M;L1Co-)=+)}nJK3|Ruy)pttqw)y~6jRB0Q-z3hUQp1;+ad16Q0>Lg+L0muF#;?)O;~{!<*N!T*Hl!!kAiF!Ptwwzqdbim#)l4|'
'qA-M<s8fj_Iy{UIYV^sYU$v_v5B?K<oh$b${mvFz@#KjfypN@>IlY*m=VRr`6NfGs*e7R4d7Y#HowX%iYT$(pj-VRUz{BTk9bP1`G$koV?!#O7'
'7XT>4U#J*iL&AYN*dYTDJm4?h2F=*@P2lCJCec`Hl~j~gPZcAjgj>r9>FTNIL>hEg$_fGNsqQ4`aO>NzodzC%rBkCZA+x3p#Dx5c6?<W`5C`Z+'
'Ed95Y3YRJHH9($FSN5gsQP41UcQgW}iK;3h4dAjI9(ZH4;X}tm(o(eZ(T((>^Liu0^u`GDZ5`c{NSYIFg#%{}S$~SdD%<R@s-h)73HJMy9qc9G'
'd5S&0Ag7$`MK*0y6H)LC1IQ_w2sC(ICT__bvsk>SN<km!E)NTl7EY;sT}ZiRhxgR^W^gZ!lcYDmG|&9Cch4=+kI-3(q?p7j@gwzxoiV*fwV7@B'
'!w*2xa{SY&9^Uh_d-K_bpKZ)%KhT|I5SsU7D*Xn*MKu^9H-mt!=nh_xLVJwps!$IPoe_s&_!nqR#hnADW`#OVyr`APnn$On0W}d|)MjE*iC3Mc'
'D~fYR@d`Mt8aIhSftRiiIw$n&(VdXazRk@J$!Yxt2GGNQ+Em{mog}D?=h^aNOpYlkD=|L@`geJ^x`Z<zpl>%%e@6qo7H7zJji{T^fI3jA6qK)Z'
'_``b{ySeRHnZ4#IiZ8jSE=i|V+ZlDa)YiOL1ulGrL}q#FpR;N=bP``m(uIp3FF_v4_hR6~mvp&ir6P~3ty^2$P{la4yi@twGH5?P#4F0@@Xt$='
'(3#nEpw#c+{lN?P=eJfNNuHvb`5l&cZj~^S??^My7yfINQkS1<IKRd59H=52Yg6=mh_N#QRqGccHLf%7VTmt__5_d0(ekc-3MgFK9L2a;Qqrk)'
')daJCF0f5=&6xqOc53K>2;3$;dVy4I2bH=rKoartUN?XU0iQwT_WkW9y1(0W!ZUr3h)tQJ1KvR;E>d}kOsl5Yh%@%X4^Xnbf6$NiU-UbQGEj;#'
'yH&c$RauZ4TEih?US}C&@HF^cAHMK_qkeSo@_}7Ya-|1$(tUg<tE@f+xkVV^iWHOEm(Owa==+`eUII<qf7y??1ke<Ijl@L?!&1U~Sj~=3)Fn_0'
'cpk$h2sXhUF*SH-Q5FRv6p^+GL{;nn<MgORLxUjvBRBD;o_a_novI2QNfs)jj7mpU75a-xP-T?jOjT8x|K!Y7CSE9G<Na$)_9aJ|4()_e8B=l='
'cf{t2UBp*(lP|*O?i4{P?x+;Pc*CdsNR?1|wS|3=r+!-isgYn@RjEXF0Rmdnv>bEAWB7_S?99PXj<(KF1uYPSr#mn1F!X$)XF&rsUr&H<p?tkM'
'C<z7ckj&wjP)037wIkA`Iz!xqsnF*(pAJ>!UX`>0PMGtMB%7*h+!!4v*yxnQ+biV<z7_GF#3i}cG8k~w&(;c~^dz!39GHG@->hOxJd!!$Kz>YD'
'?JD4}%-xe0ja&V?HfbXjk7V<xuZq{P6}Pv#XHsIJ4D6}2=3h=ND54R90@J9f+q@B|5XJQpLAP2uddM+;w5jO!WYWSS65RPrtjK=F2F09sng@jQ'
'P}ZPw2yES{B+$hrmz;;ZcxNzYc{%FtpN(jZ&VsK25Ptg{**CgY<N|a(`CIi+bDg&%UWhXq1tro|6{yDHfeRV53Thiw>!bde9#rHRQxrU?M^|?!'
'6Sr^e<*6D1E{I|9ENfmB{}+t6ipDlF(ER9>>?MPf<t#fxNaNp_i*gWLANKnFr}z|b<K3sE_<lgnk;aq1!js<}&=V9sPLU)Gf&*?To+r~+Fsvj$'
'T24<dW8et}!Ck_(@FuYteRudYpt=PO;v4dq7zFPzC`s@QjC`rT;k8o~e9Th}E&e*2!!e42QxZ;%2OriYuBrO@DxO^F?-`~*JOLRsokYR6zyao1'
'6dHj8c9lR)_=;oyUjF_S{-V+i6Hgssrd@0UXLZU1Q2K&WG({++n7$}@7v~r0@~!&vNqzY)3XVmjw^l+HP-toNCi#2(-8wi0hBi8U8U61+_50t@'
'$Njob|AVi;>+>ht2Sy72x8LXg=zH`G{yT_(!$BtJxA32@_djLXmDTl&!=Iq*L-_Bd>iV#u>kpV3!s`3^Pt~`c_REI8Z9eAb8N_onh=c8eV9w6o'
'gcsBsckO<C=op?qVhqAK+6%!zczAepBt08ne~v*AXKD}O#g@HzMj!b4`t?42A~gu#UVt$ByWK+o=GWec{!ai5&rOJYP6Kd};xXk*v=AcQ<RGRZ'
'cH<atC<F;Hs@#TdW;03-`5BY0NIwQ`cQa*aorq25k%#%33wt7O85D08zLDf5gP<SX8`?)=csAxEn_o)Mm~0cNV<@X9<wbvkBkXA1E2b|@C&0|>'
'BvcX{MRWK)^@dDL^yB7OLApu2I3m3ZN<3tMKo_2Cwux~&-c$?PRK&wwYg<)&A>_u9BrROCI;V{lE55q_ytxHh)J|<Hb+$V(^uVH&0J82$rS{ml'
'C$)FoE^flW{w^TE5#ocI2n|ma4AfI&)pSCg&AqKuRa3HjBP(9af(3#DO3{Qcqcfk<BhHjv1)wwGW6-!D<`O)Q`h!0HAGoS!@ZazK7Qvrru!jA6'
'OEa-?=V%YNIvCsMSEY01336H{{|q_Jstgh^z<#g44Ul!oZg9{$>^yLJwgZT26P_}c;I0ng@y6w;W?QX~p&2mmXHZkme6<QEjEWvo-+S(1w6Y^D'
'#$@f~M(e}?T(;`R7ugV&ycU^pdk{o7K4b9T!O^#GPCi&68>?Wnl=@}NK?$)Ht-mIDujbrPrPFFkdBVee{o&2=YjP4>msvgJz6QjqJ<@Jo9=tdl'
'V-78h?Gauk*OThkJUzdK*Otmt>GM@9QP3Wp+3lhezN#wNJnV9YJ2FI=mw7ykIa7TXOLf`fl9jBp;r`&9BQ_8FgM8AR!+DC)*AsrCg`M?aabD(0'
'G9<yLTdq0P{EtbVq275pxzozS#*_u$!hi3-JAVI~lhn`A{=o!>`;b@yiaLNd$RIYE>oJ=xRtY+I6qoBVo%e14+ArH7KMrk7&krADt*N767VGm>'
'o?*JQUcO$oN8AR^_7c^*=kapM(bc;GQ3VR5ML9*+Q={_7laI&rJjrwO`J3bK-hcb{ZRM_tLCGmcrgiaSeqbi&CftZ)DvY<s{caBgYlEBx8mmK^'
'$>bmS=ipiBWOR4$l^+rrRzqU+e-#SNn2obrGKQ!^)51{|g{K>o7IBxn8YsoA@+jKrJ#_lb_uG7j#2Bsaq#!F-StE;c6dMhTLxR-?`9^ud5!YJc'
'<ekG{M0Ut#Xj*Ebic5<vJ2tRHFZ~GHMYq~}-7KWzq^gJ2tq%>WK5lwE>f1M&JQ=;4+CFmz5K!Pn7H)9c48Ljk!Fimh8upojYuhOQn@`8bryst3'
'|HdCj#o7_7zHKLV1cEec`u+6nv!}iG^)mFb`~rRWuMd>tKGO6(H>U13C;rd02KO`<^vt(w*4*^PzZt-VZ&E{jMKS6JpFaf8&N%9NGM}PxOqfn1'
'QfHATQF2p6V*eJQ&pN$9UvWfB=qrx;E~xd~9fs@XZ@li*S8=PizqtK2C2Fwz{4K)5Hl!Ru(;i)I->F7P#vf8c12}%%C;@71toUI3tb6&7?qc=K'
'b&lUwV=gsngRch`@DAMI6AfogIBy~RorTpAbmIc_u-mE~$7G+v02pnvk;nO63CRC>pDb+-4&{$A_PPNR<`I;DFnp~rWh?k^)WE*NdBr5s6R1^D'
'u?ZKh&jBr5B6@EdGmx@&JhM*Nshw_P3JTrsGQ_ISL60>+3b$T~#W36X(%dZLtNoXT@l|c(Dx!C{#1y7*hQ?KT$bZtQxCxmWcP?DebMsPp4*Sz1'
'bB3zf%;RjWR1-#;OHp%>d{aHPF|wHzb2)A5_E*wvLm5P&#)zyTjCAH_Tq;t{4E-%A16&djue14jk$6h-3xi79vuIwJRk|XxXPYCYrwUM14#&2y'
'TXb}4WNJQtJ~b0O`l%m}Ky)c?uE$z}xGG~dn;7c!#cS0PZia6R0q_wCb#k5o0oSQ3Jm*2*NK<BS`)05aO8wr+Rnm0J9jw|KbR{&|FfZDig}Q?2'
'q1C&IuMJG;joagwOlYZ)0q%05s)$#fo=A3e<mBiII&T|e>YSm)7Q7$YV5$Q5D-r)OVhul6v>C4&wg@$Z1xSA3&6>zb>?;s5^%G|H2lKh@kcIfe'
'4FgZBSNF)OL?@U-kKKIKoTGrfK!%}MFNU2_7X!?nIa3g@(pQ%jHLI3Yw9D=xhdt7dtfU`bYROw2DTVlpDjiXS<*pnQnTbI=&e!RDI*!-VG#e9~'
'{5uei<rkmg<XVhNh+LpFNKYIEzcziJ-8K>={M1#!;^JA!nPyYAvpNN5H{()Z9eEk?K;75_MwSqZM%g~`niF$ItH#zKHB~+GhE-ejfDuUQ@??n^'
'c<Osa$N_}{n&^%Y@XQ0L0Pvc9@y@4ut<29`&_N4bW2-3zPlFa>*H*_|+9>Efdv4E;)Rqp0T3V-jMWDuwAeiwteU_0Sww>H-zK``ItJ9<+6U#x@'
'r@?(S(9Kr@xhlSbBIWBMoiyqu-+g;}@_PL7=<}a^!Yb9}<Y%PcG^4mU+lgrwoLAg4;OQ2ei<W1@Q*=AYfPKDm%GR`@{GMW@VKdga2DPK5+@3=!'
'7uG$QYpufENd(#MZ03k1o>n}Lsb0_^&d#o{p@RqgQ(*abw5ZzWaTkIo6H?v?Tx3!w(a%mrRJX0ZGgG7fV<t?acsR}w@CLxCut#g_FdvOb^d$}Q'
'REu_Dn8tMr+mMcKh&AmvkFj~N&g(QTln*<hj~XE(5GrXRt?IIfma}O!rL<a2=}HF<*(T;il26gW>^Ax52F(a&X);M}P-$|N+>u6hoZPPRw)&-r'
'FekJp?T#qk${aHW@*TO?$0iKJSIAogu{J^f<{MJyV-LVa;--|BF;eR?$ycP#1@SGI#K?1VXSRa?6f`2?S6k9<JAsf$yhr{N#$=APqY9{CbXZX4'
')sYN?%KlF;fvI%zrxMb+t{g#$23!~A!hWdl-_=53)Z2TmK*n$TXM^7JnYzs-AC6=O8%xp7_)dUg(oNeW9HS*E*N&vy(f$C#zL@6YD379BOP@lP'
'%u3_czP#E;1k9nz)VpFu-Z8SmeJEM(<Er+pGof}+&82<MUC}I_$}g;EJnlScm)-kPe(c?2Py9VJ(WkXW>8w33+hufL?sL)T-s*f4-B&eqLt5pm'
'-dOo75!}YtX|V>1#`?`RKP0<$(@RPD#LAD^iv5)8kZ*g%2;6JH#x9Vz1cZd@DD+B_RK|%0I|kdH0{$N}Os@)4!1bepQlQly>+Mbdnk1fF3Y*Wa'
'PQlG3hNHMjk`;6&`f2D<%1q&uNY*?!B7mrS!DsB5#5*{SIr*f++FrwuGOEc)lRT<=ddB8J$%(rGv+yGH^y!KaU&2gU+K___h&4!+{Gr6`53^Rb'
'+`ET$-*j8XI|2!M4c}CWNO2;!j~vhK&p4L5M~>z0XB>;13g9*1@sQuqUJUOE<TrzR@z@;zWC583rtbpeUW~A}N+#u)DB3VxPSEo@sT;0L@@!RW'
'D-uv1{y1#i<M&N#a64?Z)0=L3vxg4=FZ7;2MZcL&r!}axdWVHCyXmD>_>Kxshx>y<b$MG2U(HXMGRWEI*b?NBPOPPUKUm!yZt-bJ3-S!dD-ZT%'
'GF}!pDJRh;Zy|CnGhdS&K(7}-v@gx~?0+KCVHGqI&3MN~s%@cxg^B@QxqWa}E+>a-q^8j%-wUDL;%2fX?xB>}?fznD3q;u)gEX-etSYV4wFr-J'
'XrWoaDFEy>HFza%%<ul<&o=uI`val&HTwf!8aHXsNaChiQJf}pR%buCh{MVXMSdPV+!v<P<bqR8MR)RhbllnBF&_g+IQuwiOyYEj7KbXN3C=_}'
'yUl3Jlm$t!<ZCnvl<LZy729iNs5bZp-TIJ-E6~8Pe&#aG%C%flnsXPrX#vln{Oy>!W=rME@5lEFo=N4qSRGcOPIniFLZMRQ6Jnw;b7Y*ksR`D$'
'gtbLX;<l8W#8zO^zP9P`_34(xh%81ZLS8++toANAijm4qPFA_{e5&jMxfSX}x7yK;+Sy_eFQ-n7)XHXuk^|H%cjno}9C+oqj$KVpnF$*++jToG'
'FEO4SnWr3t%_}Tnj`o%945#2CsfyQ3f&se6iVDy*0J_6fJpFcOI7)HPhgb&%TgExBl^YmL76)D9My>9w6}9H^^(=+IUHE6#nq>1VA5Mn-+x`Cj'
'`N0Iqq>i@27YK@KI*%0*&8rGXN`cni*AY+Ghjs?N7qd;vxCr;?%*^qJ-J&2*Vz&cn?b&R@#z5Xe&FHVHnaV{8zwMZjNR7RMvFsaU51#@lu@%!6'
'4P@tdS0~Mpic%H9qOS7dprrV+=M9l&tW6sirvzYKjOSUlqRH%AQ=+<LG-%sucC@vis_3g{^lHsn>pSLZo_d)&u5zTddZ>1-Q)3U>H?JX(QEtAR'
'Xi#@}F%RolQ2VZe4p$MQ=@xB_Q6$<^kdVwU)84AJn{3v3K57cJMjIv*JoPP`U>^Kkbu%KZg{i)<#(_B|P*L&hjKejm##(1PND9f9W?yx)rknWM'
'P`7be!tjo>pmE14dQ?=dYt<K7V0S#1uF|n<rB6`}v2PA)s390+ru@7`kqxQ3=bDh4HMps36~1rDa#y>vrytQ18cqFPGcKPxZGEk2*;C-m9(G09'
'`eU-B9;SiD?xZzNU=35)Jr%<#DCSxD74V{{|5~yrPL7Ojq&H(bCow6Vi?O+(#9(^gI8Y#E*iAtz_#2nhi1WTTYZ;&*ci@}Wwwe<E-52rQ{E*BB'
'(>_<H0t2$>XXDA`dU=&h8N~XLy+4t@kc*cW>-Zvxs_vVEhSqv~L_kpA-U6pE=b(e<iP^a1bC%@JYJYq|Lrrh?hmOwJ4u#B7<{@q@Vv`m8I%?7P'
'*4a}yo)|BVAHp|!<ha)ujXK@_I;gIxNtIN_i&YUw;9v|Lh@n9*;B?XJC|ED&>6KP49i4u1!aFKh6rP@7*udWF6^2+F-nT5lsWmX8E3{2!VAzi;'
'hGD4H^JlK>`Qun=v~FARg4oTRXVW{47j+qz7q&gOdUC(UQ5bo!gx9Me+4rr*x=et}6P-_MW4m~%&m$RobFnQ3Uv%%)Y@Qk*V5L;K8Qf@CQc$Ca'
'Qk{)zxZn3PR2BJ9)PwkAhw54WN;?>+Gc&oL+`_1ZL^?cj`J_6*x7j5B5FgJsk&P9Kp_0BUE*JvBj(>VE&>4A`AoFiB$=Y1k;(^0>xt`C9d@`I|'
'qR2GI)Cyz;1kzEG4T>1$BUJPy!^4;Te&6&ODDXy>2s8^e#w2#l%;C`7mXv)XZ8%A44Vf{lNR_0JH+VZ!tcl#Gs)`}faFc{TSg@PN=P*!z&s2=}'
'ep@lQT_9zCF=h=O-NqYIU>cc%V8#Auq+nW`1&<DWQVj!DAv_*5kw*P}+c?^(8z-2mAY8e6X$RF?S0~8xZn;v4JX_|*n(!}K1w0-A@(p7aet|N3'
'nus!c6;JCm+ZigUJ9`K44K>jCjRos-K)zQ|rDrzyUWVW5Vt;g>ZU)z9_ttnf5ypOr^Wi>zS(U}`<LA@v>F1+QpW(~s8nzdzotd_eH9$4G@Rfg*'
'I-xddjVN+P1cS+3kVF)hdAiUPba+1{c|VGM6Nx<oW7CCQNUnJ;<1-ME7*p}sqL{OhI!LWh0~V`XtvKLgd=aOB9S?Eju=zj1tZESI8NP-2Xe#w?'
'pFO7A8eTGbbIVuPA3oG%T#n!g@%Wn?^+y90Ow-bT0b<(MN`*<h9N*$43ECzbdIqb)*XAjwd@u0Y9ZCm*x7LweFP_iahBbg@@F*xxpNiD(f2br2'
'qNXfe>z;e6RN3}WU~Hz|+&H2z@!F}iJVcC*Q)SJ))MYtfoLWz#vtN@t#E*%(;=Rc5hM5d7&?XU9LESi~OJE!D2hxF70l=9Ta%F5rR{wSIEi>lG'
'$<G0nu!?a5jM<ql(;^_}zx*0Q!9&42DY7R#zS*9Z1~i)~j@RNOjb=QPhy;9?KoBDb{$rwSSA#N6=f?QdzLL5Doy>X9%bsCy!;jPkjnXrZ!qR-q'
'cO_lCTk*?Me43qBpYAubYrLph_DFTyDlr5OCr|@GJ9QF*I!3JMQ*B~|^|H7q6j0lgPPr=~2q(g@XZ2nK*@ooUY4CeG35|W*W6)qeYmKhq5n?8f'
'f`G>6AkNl3)!mdxrHe9KjZqRDPUfgiq~8VtRil9oCR}=Nc3+E<Z;w&+XjXY-{tK%eUjq`uN-qc3u+Rec5wD7*3O%RctM3Q9z5{fg-DYiE1zg*K'
'4uhe3CMo>1xtUZ`R6f2ft#7Bb25N^!)WX?@aNMv7G+Na-hxav%Uj1ggz8QES13T*OGkgphd*UE>!+Y5Zh9-0VUTAXn*R0?JCEn(zwSDH*@S_;('
'm2bcC<0bbtTa6!^URk4m40oeqh}N+O$kQ_n!#)8H4Yt7|p1>Io_XqK?Kf1<>XKo5(dvKvD)yQsJqcBP7&QbJX6#JvNHVT_WhpL2EuKn4U<Y%Al'
'&h>6w4$T}rNS$S<vVuyN)x4Cm@ZRWoBOHBygiTC$>{-hw)zWGP<9b5wxYPw0oL1*z!nzuB@FKr6ZWk&$4Lb%7U-)4}AYfj#0V_WhD=&-6I}MY4'
'M(vKF?y;h3N)B2x8kAF)EeZ|683%R1;qgfl4xUI;^sCdssPt74Y5$4U9Ts}Hy!{h(xI;|;kAAonk$u5BbPu*m{01nI>`x7EA`w&bi}N_50gAx7'
'NN<yT7{^1dsM4XZDCh`?Hdt^GpRkx+D~fEM41hB@j$>p4_}d*R^4y8ucDze&W;UvYM=~L~Dz*mrq|^)6Sy1u4xK1rij8XrVOeLe>j{c%HtjybL'
'7M;ladFMGl4~P_wAKkiPaZDqp_3cFWs*BCj%O~yqE*dgdCI+3o`|8j)O{vzBW%WGYI$$^M_~VA${l76}H`{l$m%E32yLo!^(+v6cF+;9p7JuBB'
'yZ_oT6SuepC=8I@KX_8W@jH0(@#Kj+z5}x}PglF5_8SNg9T~9e>X)`m2ApW%BfOxfegkE=squYvoNAXR5uqHWOA!BnnL%0tGi+Gl$f2dY;)8g;'
'x<oF>k14l|;H<*8PmRUg`}_*Zpt@uFum<l9ZfmqZfCbx}xh=37=5Ba9Y~3>~Zd!vo_;IIxB-v~nPlz~Rz@`D)3(&IgRsXPco|HEU>eV7+xY2zR'
'f5M;vf*m~a&BSIBQT=QdR2-Iy4QZ+xDRXW;ekE6cfv@M&@%cKZE6RY<d)wY@h8htA(6y~=PG&So1C<oT_&m<5zO4kM!1^`SRu`e{MeNpU@3y4y'
'zNFsRR?;(!#syKm`okTzsj?F#Q8(&_2kg|8n9g#WW?8U$${82PEBx#@*@D77GY8Os^8yXbXDxo#+Ij-a1F(T^0IXHt8vs99%k2V+E~MZNm?@Eh'
'Q0rxg0-|7<t|wPx<!HOb3<@(unpRYDOYAecBDp{Y$@k+03=^2Ul=YOTB*Dk{l+<3LeN*jJl7`h{wBOlj%4Nzi<EC3^-D+~%+18&c+2{if#f<Z<'
'3QQ9?_kky^R$OvMt-c$cRb51h8|m)*5@Y1Piyw8f!++)K#Jy3mSmjtK2S(Nd9LeN4Qf0cmmqHw-!+!6j_ZmiYk&nOo`I5QN62lzu&*s17@AfHr'
'#z%Jk8tPR%HwU10OhD;iW6F^=JYa~1!=?bk1aVOXIz-*n89J3)4;m~V!CfnDX$tgk)pdh8Vtq<|RL;gFqy8C>+|!PuK-8^rg<2<0acx#r;S67@'
'S$mec?9*0@bwF+#ou^G|%@~G0WSbe*4AradZw5=0Rp;rC39?rT(lEmrpvzG_pHro#o%Sq4t;()l6y30HW~0BMX%>?BA)4_68Jkasz5c=T0jn5Z'
'4&t%|v442bA7FZ2IK#t(=lu}^RntBjkZBe(##ljuPq#03f>oi~t>6wQb#TEvTTgKVh9|@B{^8*BNBGlwK6v}-Xt=*WiUE0@Re=B=qJlI>hN{6D'
'3>XCrojF|+4yJjwf-VhZ`>-E<;gIEOID@T>_|JGY=k#K=Xuf)FsT^Sd2|#M%fQR_U;D^JNTER71MsSVqF_7xVPsgXnpIcE2?=&%>smIh-+R+Ai'
'I`}D6E7q$dm+!iXsD*z3!}1sSgT@c*1(u&@H!WA3!jI3k^kjJr2hs|jWe(cCX0q0H)RJ(mt%(0dt*$u1z%UL?XAQF=dzVb|St~+LP&2vtOkGJj'
'wrG}{4Kj*f2>=RQVp5>)B1StrgOjTsh5{)Rxpr_JMhaA(7{kJsD}dM8f&$Lot*;Lc`fS>li`5>Q8F3pinZ3mngSmT7KjHA{NIP^0&(jjQq0JQ)'
'ipgb&&EK|K)Ge}W8V~+>id(BPNZ|<h+D9$Z7^8a0T2VyA_V4<_GHDI(6O4EXr}G8yP1xhnNnrSXDz~&i`HyWJheYns3aKTJwnx+Q`T2<lJIGRi'
'4$+u7leVh(QWCffikv57vgT1!RwZiAvoi2-pypy{WOOnkOKX;cEylv2P?DIw$~QT)wDv(91k9H5phwk<vnY6Gb)ZritCIZt%q*g_8>=$%z$!$*'
'YLsDE=ciUd@$sD{F{~C%%gHGn*2HKiQ+wvvErfoXI;iP?;Z_$@HTGXJt0SebE<@ZT)S>Uv5i0r~*9X0-)qAj<zRQWP5{e%$zO&@+4p-aKp_-{S'
'a1K7>F}$n7&gcs{w`Fz}vYOHjQ79~IVjRi;eYViGCM$oU#+EBd;0xMkAE&0K4bk~g%|3ZzZN2eU1rFXA-u>$pg?nPWVB{;%M!(}!MYAzI(3{=C'
'y<wdR)r|q6pw($y@Na|vLid65gYjiD$6%LI2^F#M5U_uIk>qAX=yPB<i7P>rr^3{TBXaoa==Gmpy*>J`<MFF+U;p{|^Z5PIyJL?1h3Gx>1pnps'
'{PBO4M={M!LSnNWrtI99o|3gNEZ7?LwsG1OPl&;NwTzoVjSA7^(d!!RFjxc4ZNEA|4&|GZ?>BoAecJGm#*eHD)UlVz^uUrwJ=tT5V$RwQo^Vw4'
'jqw)|-R><Y0{*^ZEZMNsAHRL|_T)7M1AldNii>Q-V5`5RXsJ2JUqfYBJx1w$NRvg}hkv08%uEjwLvp(V#;9Zzs{0<tx!I$k?Wr@R7LVMOX(EKz'
'P#q}Sx;ye12(Gw@;&IpxUmXOzpehdCAovY(GZ=5oGa*_0DjRDRo9tP?e<o)!Mti=10$8k<R`iN3fhPZ+o!6pQ1nlE82wsEih}d}4>-G4Zo5A!;'
'eB`;Hj@Axi<tMd?$2W0m7N~quVbp4rvR*(cuEu7Fe}5#uzkK-g=kc49PY6kjP@t&-z$n<GMTK9Z5dMLqrju^yrp0j%9Yp~tlHVL|S>27!6I8=6'
'1mtP)=rJ|TMKTQ8N&`lvwCEVtScT~pE6}0Tj3?x|CSfY8MbYC_aS3sLRV{>$H0za_IN`Sq4l<HHREO${IbBw9H}P3@u~TK4!ulp_7j(&9LPx6O'
'+)^kCyCo~9kOV_#a?^-kYbA|f7Dk?_9lZ2uG6((uSP5Cb8z6kyO>h~TK{MQV8NSU?o_}<>?LFE*P)k=?LVD(D`NK*0TEi(W(ZNZ#1a&bW$fl3V'
'*{x}x>rR`h^T<flhORCAqvpF=gRC!!$o|<}#9+PDVWz(lhmyTAB{S(6Af0mzjI}P@IE;rz97f+p3HuBrYkpUxg}qe=FF9-m^E5-`sI8kJw#eSH'
'j>#=@@8wv<j8bc8EuLPgC^&p6>#I5d$mU34z<t#meCWX_#5>6FjGTK>a$?g&>s?1B4n`S@>;H36ToNRL=o=wdPZcWN3qtjhayW#Z3TCRSYBpH9'
'H(f7Qg_B~#k+Sf^h!`^+)7U%(<f!lgMo~?tQnphi!@q7~3#wmjyNFU=!9o{BX^{OaJq7ZF^f=3sLbVAnAosvf;i+M4MxIgd@Ijygx{_Zlu2Y)~'
't3z64cVO<EP)Q09eBeAzmIa8i<G7flDMtu5vv?Ta-LzwNP(|xtRh8t3)hA;#Y@D`dhmix@#<{D<d#p7(e9#JQ!3996Svs__z#+!2^bbcG<BF6}'
'{Vl3S-pFrthSZrOUq1YI3H-alcU#bhYPZKV&4VaIDjfDf1tfJlkz&6NUeDvjiUZVlwaN(a>;>R4i6IDG&y|meICcO}l+3eLBF-HXY$Ta#S8ky`'
'hC6pt=I(m@EL%HFUOtBNq?sSme5cc{*VCxi%`By>v1tY;SOA#}M#gOSQ|rT&7F0VGf08H-q20w(PivF_)0(T<+}HLX-6=Witw>~}|Gd?h#Wxt8'
'BH51NXbfqX{2j_-gj{|^Upofy+B=69=+3|_CFj<?mQVqTayy~FJS1n9mgR#%E>9M5y5w|WCPylER+=S6&4&weT@Rpb2jNcV@qk{9&~Ge!4*PMn'
'4fLF0dq5Y!Q1{XddA{j>?HiG^0cK`-U$BR;8XFSbs>h9+z`k0F?r_)8w!oq@KFVF$5Uo$#Je;6?KLH?c^K|tyro;d`NHhKr33@AT<a7cm;g|<G'
'3Rdh_7*U}@XI4AWqGT@c<p9<{+?5hREi$?`x09Q!fYhlT=_%OoSXJv<qw>O`xw^Z7P?$cWRn7Y8UN;twvv&YP5E&hs8*C(dZrn~!(&e9iBdd(X'
'KjBuYMjySIX2q^Ewk;c~R_ei9YPGC4)#%;cR%5s94mHiv>?5hMtMc{yBW{DEGq;gqy;{u^L<(_!C(#zXO`%1Ji7xD=sikZ>g?6)?5@B+qIyWJw'
'WCC28U1JKTJ7VP$%DIm5p*6RxMri7S{cvBORh59LX;JLNT=iv{-8NRO62k0Ie5rw!d5R&oZD#%XVI&6|vm>TSLF~F`f(kfPe9=X`OlQP=aJdJt'
'=Sp<ut*gb}bX6cOk8by~MJ{2rGOBT{slIp_Dd}B-&ro0b>G|1tx0)41D&#4Sy)gB=^(yLaLRFE(hYzcSC{kUIW2}4>J=*)V7%&{N^Qmp+s5;yp'
'K-T3h%=pPB#5s=y5S#<(k?AD_t^Lrah;V|_`V(a0NMr@cy@x%pnr2g5f#@N#)lkCs!Ic;*pyD5y!N$t~d)bwHTUA%0*3}s^ibYJx6mi_l`%893'
'LViq;OvunzR)Zk)HF$51suK9eoHlgh)alegh`<~nw5>u1pAltnmbXcnP;XEUL9NOA5Zgihjd#RWtDAbWyhCV?PhT9u!He~^WxeLS1ay1giwfwr'
'KdP6(4|^K;=&*L((DBthJ}Jys6c{j~&8|D41Uq#Kx}n|lR@Z~H)(h6$@y*2<2G9Nl^Cu6VCRes)woJmGb=rpCMiaGiQGy$=-oY87s@{#|9*kh^'
'C4T^-trX+lhTPIZNE>?fVa$%ZfE6Ns*eWV?&9$ALKXMK_-k6bTG<D{xYwLJNbJ-7CY~xzaISQ)5;cm)dSvL*Uz*P!n+xDq3+q0VOKY%w)%R5<6'
'zMe3b_@6|qBN@?`G^}SPT$W>uvUS_48{f*nf}X39neIS#YD(3lDK0;T*(0su!9nmuqqaSg^cr}lj*|F_q5Q3GL{<4bnWamTQpO-|sbD={TPBC@'
'ReV*aN}i5SOjKeT&Dc@=D=u3G6$Qp1YCLznX+L*2^q&jZs3G3_+fN09^B*tP8#W`qq3@It@*4V37%S<A-kTnc-iA$xjt*`kt2*=|QnkWEz0~%5'
'2Phhl=+J{r4Zw9p8P6VS`|)1~pDK`9#CO4Ee2s!CIiF1>B$cLL0^2>PMB%yRMq-krt5UUB<qk*K!~!?zk`#!<)ybtr0n919%Q2DJQqqr8fUFWj'
'2u$;MrW>ScVh)x7>E#ez^SD3=1UVb&I9;yxJhD==K=l7hJ^urxp8qKVg1E6L+>U&%vXr!!K~ZAL?xtfL6(a?X-C+NWiM&Iz>+TdN4qDB57;CpX'
'k35K<hw)cd|J!&i^TW1Wo>^|^!OYKt`7!h3heGqy%%_>3hx2T9?>7I|yv^HqIrHNYO**^5U48r;;9(0bfbpOE-7KKfh#*~xY4bB=8&Y<v+`%qf'
'<_yEo6Yj!LRyP)xs}oE3Pp)+lp~#`Sc<|N`S<-%t*s0nPg{Nw>et=Di@+T=OhY~dtvy^Y9x<~tQWmL}4PaFaaSlg?9wCRYCm$KL2!%%Hwhgf~P'
'jotb}3l&wBaexCo52$PJ;n4nRP9QJSSv*0nLi^Z-mbI3$9X6lUeHL6yWo|V6t~pNdwQ7?gmoi}LSo`a>H#4Mzv6M4S&&;Z(<(Xk`rfM~<ZDl^?'
'v6#ZF=8kpL7G1tdCTdD|`*=NvzESJKD&Vr}48uDIYfgB`u7K5pPNeghPRDUM951t*c1JENrfE(Us+$X#$^Q#bO9KQH000080001DU1Alut?do~'
'01h|+01yBG0AXogV{dJ6Z*yf|X>u-bdBq$3Z`;Q8cLV+(t{etQEyY+$++{+UH4ECIXjgP?&|eILIguxcHAS-gu&rhJ-}l~q@$N`Ua@Jr*Ym>)!'
'@9y6B$Hnu!Us+YBMa~kRR4h+e#j1JE+NPozVM*GMvMRO}-IGmGk=NgmH^sgzY8G5xTwc80F_N(@jSuAAJMkfH0NGC&&ELHPe0!SaHE95BUnDG>'
'Z)lw6TM`#J;E$Vtkhi<EzPy01ZN^}HU#1zW2`roDO+jc**khTcv9XUP-Lovs8L11s<C-J|$%}^U=sml<fSKEt)iv1_4*=FyG2_#G3yDuMdVsk;'
'iRqqI6!wUl_hHq9&^Ad65(AdYi;88m0SpBJI>FZh4padFJkPtl*i^-yMA4>g+KNRHNr4Jg15f})5nqjz;!w?@#M4WN*ZILGKco$;oN18qkkGtI'
'V+HhgTC;y5>3s5S$}(UQCuDR_voxViQR$g9jbg!?s4ReD2ZiEiRu|blOTGbS7F9~KpZMqgv^w4a!}RvPlbrsU{SH)UeBv<J*O-4mGU7%qFXD^>'
'*?-|Af0L!<R3Biry1XFpbC>4HN}c=06WsB9;^nleIKjMI5)$=6Oyb;#wjfO_vdIeC00gJUG+`UsW=*&Vu6(j(;T8VPShx(XW@4^D6x9u_8h5ss'
'U;{GTp99tyU{Zw6K&T*3VXRBWdJy)(5PZQ8_Cimq7_AE+NZnMuSzgUD7B?)3)`uubD|jv{pz@wRMj6Yu%`UvYQi#}nny?&>Ua?KRB3W8DH-Plk'
'#TtS70-8gJ5Eib*a)#VTWM4Z1I>q<UqZ?qjCl>^KL;|qr0R*CWsHd>w&5X=n5w8;;PYMnGq#J@DLB}TYRcH+fj&w6Pk!V^o@+-|+_Dxm6c06PL'
'zO5UwW@HPbgzbT$0)t~BQyVtdH@K=dV9Ug*9hevfw_u?&_fRpgGC5z<)VDX>etpjl8<@rVI)~Qoxah|;>p{4xwUDZ{2%cZdd5>druH>BgRj|C~'
'v+zOAIy`xnd(^k!J?{GFJgt(aRPj#IP^p1U*-j(&X|848(u9{15&DvQZq7y&R04p8vVFOubz0l%`CO}K-L8KF{S|orLw-aj<f6ktM<cI3UL`A<'
'Z<(Qx2GNH!(o)5<G(b9B3<B=RxF(({{=cSmOEd0;ekobpmkR)5O_U~_grCZa_jJMVMhgwqll?)chcsz+!>c)vJC<&D`T$3zV{rbzBQU=V_73qD'
'%v)Ybisk9vl3p(d9MT&XC4_O!s201fcyX@B+VjvSs@oDHkWcQdl=%FkqUA1)-?Rgk(}`AgMRA9*N&)e4fe%^nz^bU(rB#BjzqbXbPfW!d7}X3Q'
'`C~)*pB;<ua6kA0qR0QLfEJ(EG}$um_Kz}W4SQ^Ca*rCWj=*I`W432`V;sCGO57fB29BGtjW`s*u&N`28ebPpQ|v_;Ct-S0r7?54f=L^i@V_<K'
'AQrV{_YBcNj&G&dL&feH%>;HnNfGOSE==Q<Qh+mPjJv?62U_J2f@@YO)^KEM#u>7zBC051af<2FnC<@mhE}c}?<D<?G2>f-57VUo-HN})82=n4'
'lmS`-sZ0TB9`X<tPGfwyFVdJ5_d+V`&7)E!dR&i^wvuvrwjGr-KQ1T9=$d6ubLh*fv#hAdltczNB$OS!!JX$Kn;UVO$4TZeB^GurNo$@Ha`Hc<'
'rQi;2;+-@Ht<o=-<YTd5=S8m1@**;NXIr#j(Yk2!gjOd=fvGX3w0$B#ivI=@{yYU+jxLf_xmy>BA;oX^EFtTIQIEhH!oOYzyhNa4Y)r4Cdr@yO'
'<#~6tct+DXV*Pq4vtN7cILYi}QV<}9>`57%E}uFhfc^4?L?mLgh{tGlILrgVCzvpe@}383ktn-Jluq#WF=(Y3QW!fc_arl`B6p!dSox9olwo=T'
'w!eO<9I#e0h0RvGk2!8kOO6mCQo7Eta<DFn3=qAoT4p%I2qbiqqEnP1T%0kQM@-d$q*UEgctK|`B0$H4iOf%E;BscofY=%2k!=qr77)g0MUph;'
'&d<lv*Z9E^Ogl5-T_Z5`l6*Fjr!ZLSyl|1?2$DooWm$NsYt=7)*(Vt(srh|^!Vq`8O<QJ6>>J&GXI*|4&XB@c!5n~A!oI<ys$iHX8gql9>y!y2'
'5QwRjOe>S>v!O3vta@&@$0T1KE078=s-Qh=AtVO_*pfp|_Yky~Z65C+m*7r7jCn&io)6%V0Y3z0k+cw-qIwWP^!VQ7?2V^*`CdMJ7@elLADOmo'
'+7GsUH@fy!QlAKIlJ8yWG|YO(TPa~Mtf+MVD@`iS+b|T_0nm^X6%VcEVuR!>vJ3|JfJpi{6jN$5+$$Nq1GRzR<V*4enS$q7m^^w#7KCLG=owiC'
'ybd8)Tqy+CCosq<Al}nOuzczB0aYnUld=%DM;sAw8YiBS&jHc(Qd$Hacmp#5dX6i<#FaT8$e+nlue*3@35i)X>JGy3hy1tSjl$X?{BMltqEL8a'
'N5oFu4HG$uPE^!H1o$Xom;-mx{%H(~APNc63bE+3KLHmWq}$8VLPMUB<p8Q%<GHJ1g+h+yH^Y+3o((rBQHep+nYI)_SG}Hs3XhEd!Rj)o%{>%V'
'QV+X?C)sL<+fwSfCi(620i{N{S8A-HMk7=dwBQXd*%!lO%3c;HRR(6>A8%?|0i!hisy`J8z3%z0N;PQ2k$nYlZa{c*dv?I5K+g$lG}^YR3k(lH'
'&cq0I61mEPXo@%;UP13#T&rPIq*<`0wGQL!nPGlZ6pZ$SKBD=#)_Bb3z}y`a=A=V}6agegv&z;d%y2@51793{>zYy;svB}(%`JpO`x=I4#tyn6'
'2i7WOhJxH9Ac#-$;|onl2UaM$Ue?lk^<$Ma!kdkocD?89{ot9l#Gt$>k^wDC*wz$Z3=mNo4n=azD-<0c(6>!=yKo|mzU|W`6-fWErZV|T45h|&'
'awJY|cYio$rzQf!QvCuEQ3Wa}x8b<FfHN*2H3Zdo;5zVtV$Us^c|ubvz6f4`$A7q)0EH&E@YolIdGMmYl(lMtmSp-6Y*{m%;BpgxGMUYS3Wh=G'
'XIS`wT)o*%r;}jflgVfJ>u>mL1#q{0^srlbn1N&B4}7Z`YN!OeLC-IVJm=X&K9!$l^Tp5-S_T5`eQy@G0oqAGtJx|RlJuDmk?<Jm&b<N7Nx7@g'
'=BSdvY6AF?a%v4uy4%^!3Q`GY$Cl|-63yRi5fTBTGgr~d5v0z>j!T&k{RwTGV$u^#g*cP|1H95}bCcVx81V!WW6$YCm2@VEQ^wV}3k=t>B7##6'
'e3cj+7}j@VsgjepZ6>ob6^0Ay#BeMV)D8z0Um<)+q79>X-$K+`jDzBe$IjWE#s)IhRqX3yaVloRl0`>Mu=?R%MeJt;LYb-%l_pp~w5&#+J@?6g'
'cf8?ZH3#)D%)%L9!XZ9;J~O1{Gf6}pNsnw&dWrFRTYs7(38p8O>Ky8msu;+s7@_6tCb?Zd$2rTEU&yBh>h-AW6re-y*z}(Qs_%v$<(Efi{2<2<'
'&N%P!$#4AG`7Szp5<t8-5VCg{E-?=ip3EVF>bEE=sCNfWHuNNvd>al61V}F^FqmmXcO2M`?F1(!_^o2|(F&dlBs=pX^7%;3LZS7F&Bn}f|3rRd'
'4@C88YOv7!S_Q@}!;!c~CR_vgp0UzIXT<c_1$@R^fMkQkC$K{BA6dgZtXkA_L=SWZvqm;F%kYCw)@@_rlNZ*K7T3q?Wt8wbIo=S{n~Gb!Xs0Hr'
';RtOAYQy`qVS5o0ajL@`Qk$#3I?o4(Gea<r#T03{4zGQlw@V$J6A^Ipm4yLu^crLD(3rxja8s4q8?yn>oeL~hU8sQu6#n4V!8BKOPW7V>GH$z#'
'Chrlutm!#o^@N?M(8x_?qW|1Y!wot}NS=g`!eNhAER0VhK)NR@Aoc4Ac&GRrd`$1VV}Z91B}?J)xXtHqgK2?2x|JRXW}8D#at^5WD6FV;5t4}e'
'e9?;MDwO7qW~@GqaLDfeG07PBmnaRUPAr1I92E%r$S0*jH0VzWgVj(!CJMS`e%$#NKu-(<#|U$psoaVhToo59hS@4&Jz<7kVg2$`@mb#PIL2sg'
'8nszm1}~0Tf}=5E$MKFHr&%tqS)As&xLR?P-~4{cac0L^Q6sZQK>NHF14r9whHK}QIG&?=J6cD4nm6M-mtJ3nn&M`*l#ut{z7^LvD#l6UeK4c~'
'9Uiapq`6%N>*q=^+|?EY>_iA0e(CBuAbGCb*{4*}r>`>znd)C!z(fIYTQTUVl)##SVg3LLekUPx_U&svmXJ2F>K<ZtgPxx9#7HuBFH3S<B<sy%'
'#79iBoj&j8QJj%a4XE0dc)c&OR$RbevrLY0x2tw3hGZ4S*qXlIp|__;p0qvlo(ODGzXm1=yHR_;-8d@h&1&)T>b9e=z8lo7p-x~#uy|o*C~iHu'
'b9L2)$}19`Xh_e(x32PL4jrlOt1XIAc}ZlZRaLcHokZ)sr*Ao})V+KuHF)C&?*qYy0VU;Hi7Vo~O4i|t?mWsV-UokdEvtQJE@sv6j73xniYsn`'
'#fj$hD!uk&J#;Pr%DCZ$>{h{9Yx5LSg7XbX9*|8&xAl?r-FCXUde;LmR<c{g>ddkopcm4~mF$7(OD_FVx{Onm1X-b~X*Waj#elKOU!nD8Kv}tb'
'XuTOw%8Q5U!5>nNuzNJV8&#apc(!(G-<=5Y>Vzn%xSvcO_C8t^GA~<%N?y!ClL&%ap{h9dzGyKBHSaqZfjrb{FkS~t+I=Z+0{(^qVs6d6KoM&7'
'9%$kTmVt%kyq)T1-b#y&&pn}WTljb@+w;6{(VbFa-Sv)(3JgJ6<O$}vyelUEeHQoA>x1ryiGLR2Ty5yKKwgEsimZ1Un6f!L)*+XDpW&G|-4eH!'
'`s85h4b^c1NO3y!P~r)LSB>9Nh+gPH*R?1W3*Q5wWZ$RH$XC3p?Ur{EW47pdxdB3drYRr%mZ6r0{{m1;0|XQR000O8003fLZ^8)2K@0!@A0Yq$'
'6aWAKVQF7uZ*6dIb7fy^VP|J-WiD`e%~@@4<HiyGZovP6y&zb!N744BX%M+A;Mg+a#&<SsC28v#1VyeaMij|%mpZ$y|9hX=7ilHYSNB_eNW@*v'
'&dkpH4Bx$bcUAGhZ}r5hYRM;cJ@@jqSn}5EL{U9>a>hOWSO`grvd+@dJ5Pl?xxVnyrYVaol|@}e`+NI)H(e#XqViI2+Hx`Tnzqik5TV!dG#6MF'
'%T#hNtE*}8(6#CyK<fpaC`ulBX_b2wC*lCEq}bn^^CuB`V=y6jOD|fUaj*wMQUKMWDLHWRYFV^(wcwTXCVX1A+$$DM-72or>d9Nwc~^1)KHt52'
'x4);TCdssuUCWci00R<2P<(;3+51h@)oM`-bGAw4tSlz>;5C*jAi=Yi%QXg3XLBys%Wh&Mv)+1IKg(jlSBtJKK_%jCTesVrE&pf7#TxIwiiV^}'
'n%LjV`P4H23EjY-qyhpawnGm>rN^hTPV@LvUGZSJzvtmEPi5-G>dH~xEgIn)s3781R_ELgykcsPdW9gP@d$&*OTpEX#$;#%*(SHqjs8{qyw~gy'
')tLSldNWuGZ$<pf{yKn(4SpK^jSY!ic3M{w+8vCa8qS8yvAqLgm%Rj<uxy{qW$5p7A!pkhU|Bc3^36IVf!CdkzdsDTRDg5QK>ysqqAvtfskCV5'
'^1g*Lbopb(n@xp9KaR)OBYGmd0B}&|T&6`CV>2S}@PjDlO2F!*h*`^<GR-&(?uJL-9v<GWkG5$6QN!7YafRNWG9u+Y6FS23qTTA5<oh!8o;9@>'
'7U+F|t*s<&vl$#JX*-Cdu4U*!7)>XIH3~S)=uuhNw5ZZjC@*dKvf!T&V8}~erB%i~tw5kIyPUkvb;)H~Le!CRJwgfp$l<^7-MzIIICUdqqJO~1'
't7}n^ZWHq+QaaYa#ni$_g-FQc{lJmzREb+~Nug+^F5%lVB<@7=N9hMa^uVRh6m+skt6~a!VBy_;u%&#%Zv?C3RqnHYR4n?fE-GKh)>p?Brg`9j'
'I3qFRcMb+N(a@hISL7^YvXCXmZz^S5Ogg$TRxoT1!&*Njuqv!X-L;us1c5O-UyJPI_I5N*PH#qM7vtMVK1voupW8xmeN!;JV%gk9v-+uq#w2f{'
'S1mMwu+Q@d(1w6952c!@Nn5D%KoO0qbiv=Vfwq<i(Z?Q9PG_P~^!K7UClwhyA^Xh2)w_%9<ZN_)c`_cIDc{q$wv<V9NSFTWh7mHDV~!BT?ONMs'
'GDG63+b6rbD<()40lyRLb2>=arS>wt+_nnxE^V^J;^sRQCFQKwLhYZCoYIr8LLuP^HgAu8(vKEE-lkKT{Qf{)YEq|dZW#4euMnr&g2Uk;A#z#+'
'uuI+ktxl$;G6dL+wuDzuZQsN&IU^1Qpm?s$<Dsl4zTrOTyZ(-dboxg^vD@NlQ5My_S9tvb)JI!hA~PVS$y)IeY{?Yh3u4y<x<0j1LiY*oj;W$2'
'p<Q+#{1+{&E2u5)Dquy_<q~gV?Ln*co9sbha*V<xA>&Ky7`{l!u~l8A7`52-R_sJUbuSlkBOV(6t=oC*0Nyd$N%EqF0FKq`Xfemq*OwIWxb1kT'
's>Gz8tCc~eL)IoT6UK}{BmUSxXoxzkAFRZKxLRb=#pI-uxE7IFBHJ}j{u)H7!6m07ynVxT)H|q9rx3S5W`h<~X`%Yl3MI}QC33a2MW0?JI=QVO'
'9ZsXJL0aVg^L)6}IU!t<;ccAsE!`j@gA=;OTDsNM%dQYss3`@_gCYGa1DD8_0PH~)KEJFeS;Q!@Vi&nO^buk5BsslAltwL<P%RcBxkaR$##f>='
'rEVnaIj<-)iFC^SZ$7-v?=I4q>N~xCd3AbnncQCed~-TV&Mt0Zx|OyU<4}PU$@H0%%gd`@lb=6bUVOYj9!YLS`uuIoB+Q3}Cl|@-)yM0r+fnlC'
')y+>Uyky5NItNbkaSqf!oSgpDgGduS8;wUdA1^*#+>S3!LHP9Qm(k7NVm3vU#I0d{<^50~DAZN?<jEQ>RNYL{Y<>*odQCS$iuYt!RlE$TVamHq'
'wM7}k7LtUsvhH#ZNvY$JlU{XC5$PXaB~N;G43?JB1f+eR9ewyNI>f&t79Iw0VKU!X0YK68OEJa|?v6}7M#XtNWiu(8;laV`^zfMuzL+|!taS*9'
'NstFj_h$$*59{}_g^5(f%~C5A(*VWmUMRX&Aa`hm5E6LS$#jC2kPQ|vx^7uybxR33EfQS?@I=RDlw?Ev=VmxI7|d!a<lK8>lbB~Ash*w!+9KA`'
'ZxATEcTi-Xa~(wz<(Wwm82bxh(ypffT~5sdE=rH6-rYOhj?b=s9-BLxY=ffr*h9aaP(D!I*D+ecjNqqT8K0Bw^tCcM;xw`$k5xgks1Y0Ls>uA{'
'{r87X!!rZB-Zsde?BU^V{xYmKby-s6SqGh-4As+muuC-)pn1R)JqFLo$Wc?Z@)$u#G(3|Ydrv~Qwh(M3(ameFG>|swq$pABaS`u!vg}}iGB!O('
'i-EC+fqR052r9uY)(NuRd~@40Za&<aE)=@A>&wzgwxZ;`@sAt|)c<a`j7nN#-CpiUOUbBf6uz8{<1O-n!q|eRqC*issM1ysQ3eRZYG}cf@+kH@'
've3bN%2S$kh`#owPp!9l+(K0Y*aNdAO>S~N^j2g&OBMwQ_qr-w!T`u?nZvOiGvvtUs}jI$!sVP|zzOz2*i~h*K$6N6DhX)MR2i=nverV1JZ7-n'
'jF;sAV%$hiGydoweQ?DIH+++ADRT<#EM((6QN+(3)6eYwc;(wFF6!;orYW5IY9;#n3ZAE|gSs??mqPX1kS?N+^y;Y^$Jy}<HM*@VrYTIul!Vb5'
'm2XKNY&JC;jp6R3Cq%}e8f!yNs`$}W4^1}~T&8MNWC|e|iYO>wT_drj5ByUt&+D$rM|x~!u^2EBKfk=z;i0d|X4DwBX(jM#p@Z`T4J(So$&(}l'
'Mt()<)(#Ns;4dvUnx{O4url*5ZHk>Xg`&)ggFB*lADgAW)pjX&)={DFLOW>Kq7XDIW&Oh6Xdzz_{xhy1X8;&d{<HFgpD6Nz4K2wn`)3QdWQq5v'
'Ft_Ws`Faci$vVZ$kLSt9tFzH1HN@-PUoLJhu0Fl;ay-5@+ri6AuZgIEt<U~aPlhbiMjx_&n~5<j8oSE5re!B)+h+fIZX={_A!~+p2|T;5au~yR'
'9}W*U`9SAicfw@W66Rpq*ZBV=O`>^k8lJy3J?i<y+fpFpE)LjxBmUhKh@u~oNZY4)ZJ%cTAr}ohUcG<COf{^df9Hf5So%Adwc&3vf;7R)*x=Ke'
'#q$F)N9gXgd<t!lMFS=LZ|gOb@__Up*tX1|Fh${au{8$@mfPwcIL7hk_z&OqXwq1<;w}DKc!~>pu%z+KzE-sQEqli-73{HVmfLLYTxCm2cxFLN'
'T?Um~>}~ev;!q=I(+ZFB0x{Q=ZrfUVowoh((nsCax$aT3r;3l>Nf3Ec|2XCgmtv>(`wxA;(!&u>BP8`XNo-$-{+vpiG<Dol5IP0kH~)KqX9P<5'
'gHz>Xzjje!tO1WbHF5Lo$dVXO3t}9BTE_zGL#|!hL99Ekw{W2y**ooDw^)#)Kik4&XoIrV#n3#UsnRH!8X}lQK(kOaV%6Jgw|;dB8#rklDs{v5'
'9MVks(lUT5T}7HUQYDN5KnLb~p*#BDJ@slnr4Xq`<wG4T&6g!)%B@dJFG#Z_o>hb|@|W#+;?T=xNV)!D{YRvotw60Me#gLkvOb{)Pxq?2UGF+)'
'eRFZ#RY*Hd|6;XZD19jpThE9*n!XEUUm6nYF95q^&AzE<&>zqY{&U?<Kr(PhLhrC=5ZoQ!+b9k`+hpB=JVU@=#D1b}5NPdZgB@t~Evj@o$K5^>'
'Mrh4;<6EVjGp~a|Pt19vpY8HtC=&sHO+gXucZt3((^@0y+(kh{`R%-5Z@hy1oEFlz-!hNB?EyD6tuvpnq3f|J5TITOycT#~Q0a5rPjA~9P|ZYH'
'iJFA?rhkw~l2E;zB*-60LL=)WvH4?DcWRtXg>c$FEboqN$}s$o)DQKaJ6~0``AlsPHkH=UdzMcPN7I!Bk!6BDi(=!2^&6rJ?fZNG0Z>Z=1QY-O'
'00;m80AgLfdEwr(0ssI%1pojS0001CX<uV+ZE$aMWnXM>V_|GxVQ_FRaCvQ1+iu!G5Pe7DA4c@SrXp_AK2)j{sWcarLK>8`s;a887JCd+mR)Cd'
'4dL%QYvW)=m<K$UGjrzV!C){|%caIMYxEveS!&9ZCu?A3ohP#dze2!gYJ~9wGP^0QLttBZ<fPy0_Z}((JMt^A24QD;jXZ*dtxOKy!o~ZK$+`S4'
'5}32itE>($sghDMiVt}QPZd@;>i05L*5*388-eqd)e<4oghu>=zspvbR~p#?qgP}CN8%&P9kKigwZUM}?-gt}!rFLz_L>$z!k;}Tw>Q~Ay-`Mz'
'aKCr3S$BYy%C9<@xrm|Q+Eys%RY|SwR>bjq@)AQ)dT7<b|ENf{&6G|<ZP>!UTaOEC*Wc{oG3Y~@e(&>1mbDUcEI?C%u2d$Lb`C%Fdms-LU~H%%'
'S)xa?jibAp>&ZC1noZ`ZFifX6zoQXE(J*nIsf=4-cvz9lRtD;6gE!2UPr+eRIS&gAZ7pb7G3Fu_p&aQ6`>KqFO-x}S6o{?)6OAY7)&1T4{_!TA'
'-CvDo>Fszn`!c?oMt|WFBF_~@4XW;s=w3G^4g-igm32b?x__9a*OP}(+6Hq{=|8)0(QpXwATozQXF->^Crutdv>fYf$wu)yC22HsReH?hgSBMb'
'lC7N3IZiNf&5VM>Yppoe5gV)LeW$&9LY+5aK}KoaHNJtLTgogOaI4Dac4wMChiPh5TLiD%+QE*gUUO9lS!i-H)|V1)Cpkv!t~%MHzKJ6|<zV6t'
'|8$|U^^(KornS%Th>_{mI0*L^Y#rup%p_ALLmhWuvW=&_)lSh1v1BP@MULOro}+Np6`K^FGpxS`?(&E1MknZrmbYVS?ncf*bcd6SJ$a3n(c{<i'
'3ure1WJLrj+tS|9W&1L$@d=qsS2?E5hpyYEMk8+j0Z>Z=1QY-O00;m80AgM2o^c*71ONcx2><{Y0001CX<uV+ZE$aMWnXM>V_|Gxb98TVWiD`e'
'rB-2Y+cpsW9w7hW;K7hIIKqZuUlJHuma3~;$IfhJYgQBjC7o>65~-5ZlA!2+-;t7C#deDg;|E!!j>miNo}N4y43=E1N<4dzYJ&=dG?+@cff|+O'
'+Mvkr!ma`Xg>}g?tWX#zq=H(>yvYog0zNj_pbiIv!Rbk%WCcl5G^SCQB*3d$D%vk73$xMs^yKuUTm7J==(c217<@D(Pq$mTm5K?rLbcnYKbG9!'
'PrJF{6?QplX4bs~w=}hfot|VR(;8+Sy4NyY8maL7^aN;7*>4bQ0+=DMjRuw}N#PpR-^7VRZJ46vMiW;c1;X<8DU|>@Gc08qEn7S4S{B;OITl0('
'7u+O?udysfuEd}_*bciI+C><<dVLpg<T+gwQ6E;DoGZVb=y9wX905L3uM)X&)86I>PI+C1gvb<_89xA~Ay0c1xI)tqA2DkT`rh>-x{4!+uYR3I'
'FnI~Hc?{9p$ueHT2bpT{c@A;(Hip~9<mPH|4{xIT5u`GIYL{*N>~1;@ykN`2FZ-?Bk+Ut!7TKpIPl=DN(_ufzaWQgDmyH?!_;U}gQBQlg903Pt'
'?Rer?=Je>xlY$FYmirlqnXHr={h;5}x-kin(e;$>5}uD>!v$SQEvC`7(@&*j`FKW(8CexIrRgT9lvM7%k6FnF0-bY0YBP=d>Bjq6yWd^n_Gg>g'
'T}OwgX*4y-7W0Ieu_ZCoMiutP_lEap!|H69$HQ0W!<+Nr@-MF?I$TvQA_)GkwkES>w1{Cci|4JF{<xanMN9B6M{pUyd<NI^*~{tVIwq9`@L~>k'
'w=b+3mr?8_Hpa)SY;w#)JM)f1w8P&v#ac|5H&v~DBKR2uh2&Af^4{<53J-hRZ_~d9-u1Hwe;G}q>lO&ST+DA;4*q-<Eh6Cg_>%bE)*S?4fhJp%'
'M*PE)9Vpv>2kPAq?*rh4I}l6qBde@FB<Ifw3isOq`<|(l?&JCn;x)EbVtr20>qWNKc^|WOpP|-grOa3fhUpEMH3hBKz$u`hcPC}pChXP9IMuRf'
'gcVO@73q6h3vQ9CMgSL-hm+g757G7*FF3598uQhrbiC{XubA49{}K?@)TH_YpCPlpB8;-GF^Au7@3!Yd<~~^N?W&NRg$dnyZuxX!KZWkgvt|Zt'
'hcvCWzW<*^V!xzYsXAQ{A4ep!d%Tm4*9nAp_k$w_*Q&246uUh*jB9oC&T9?Edv6Q)<WXk?HG~A|Ir4Mw^N};Ub9$S{;1!IQ6#tcIXHyhL+d|t='
'uU+v1AR<0_EfwB*$I0h!WeHm=^E-#!9_%c0g=4SR_iq?QZ?xX|MWfIE0APLrx=AZ;{|pIyuqA?#^jPE+#a4_9?+C#=k~*Z8r1tiBY`^=R`?;58'
'XMvadj`7^#cCQE3@()l;0|XQR000O8003fLF^PUL!wdib6(Rrt6aWAKVQF7uZ*6dIb7fy`Z**yIZZ2?nty%wa<F*n0-!py(l$nOo3D!}vB>U7T'
'GtT)u$uzxW?%ezs+oORfNaC3ySrD?VsCtIJP~WMq&~E{fqA1$EbVC0S2?C4V#qPHsAjfg8lvv6W9W3RPJxE{2ip`Zs7J;8K5lz{B5J?^Q%om9c'
';wa@z$!IDSx`(kZ!f?$dYZe9*k*4kAqvNCZF`KCPF_nt5xs0R|8hz!H^jBI;LdpDiDmlYEGc$D>{D$|zQp_bw<v&(3@@3j)7h!mO<j1j^1`#Ac'
'86{a1tEC9d039>EkdPasI*8`1<DGj?oi|;NU3~n=rZNrY5tNP|1gYgX&hgPq#Y;9G&sKV+<amrZlUQk38O2%{<4FIkRmtL3%P0K^pJWmep%<_>'
'AO0gAY25MARL+=8;V9XlA<g0O5ySNb`+KC-DkV2)C8E?<L84zT;f4^-NY-qLmn@PGQn4WVghenm#CTT0lpMBPVkyWDlBXo}k65aeou;j15(-~7'
'oZDN6Gv`15{+Cl~+)!Ji+_I&XX1V1yq9!!%4DqqL_l*Y;u4HaV#71l>o*EuvAYMgN`<A_e@4aSoPq~@1iP-Dc>~!Ga+79g#J+MFHW9W7Bwoa+3'
'jix_S3q3eHU6vvlKOz#+hWXDCC-bF@bee0XWV#A9XZ}j12&@six#P@hd-m-NP1y+ew-}s_r7M^rR1pJkY4(;g7m+`8@tMXe<x988MkD4%LMam~'
'ZfQ7O+0?C3F28NZjgc)Um8Mh+r7OeZ2w}XEyH?Vg%q5kL*_8<eXA@XD3uB>;Uv3;D*STx*y;dlIw*A2{Sg@}TbTtA@+6(qojUg2k=t3)nMG!S6'
'yk<o6a0KS2L0>abBW|~vZILAC*O&&rwFKfxy^-~FOVoOm*Nd#VAUhZJ+!8fQ+W(*Hn<p!fLQWfboZidzC=|=dRItY(d)%$uF6JLqpyhZOQ>=}F'
'tqp>mp+OshVBlm5LCD{TGdT_P*xs5*$<qR;_rZWQfY9WP@O?@GC4!KiBA_mzQljEHRI`AeNOfZ3PA-#$NCS)NB<_Q;B}J^RkUq?SAwY}ng|Anp'
'7RY6wArdM=QB15CA~IUCP)2jT$U*FFR!+>6MZsB!2Py)IprS!w>EGKP@+k=eKhP|m%~<*vXn!GdQ3l)T1`{qNf^p-JLt+((&>1r4%WpmJ^6bhR'
'3<%^VaX8IePrtdmy1JrP#4mhNbiXo}v%Se2-L?YTW`;XETZqD#64W?19}S|j@1j|JN7C%`^*WJ5_Rwp4UW;0P48{6`On(`gT2*TK6)h-S)bcYt'
'@lmfs4D349SaxoH>b018otk;Bs2u7cdyhKG*%crpD(<MPa`vNwS!(?@4iTB0eU!kNX{q6dl_op#$8X^9;vSCT>^qHm>vNPagb{Ux5UqlHiIKPk'
'0g%6BpTFWW5k87BXBW{tlq5r8n9;bH3%wR%8b6{_h#GCd_HRPXgXsJG=0kq-6KC%t5C#Z6bVuf>OF0viRhROU_-VyHqQde)m+#iGdfPESxP9Z<'
'i+8wZemb80Gxz)y|NH#hqu)*!e>E%e&cy%TYJ4x&#C<**ID-09&N2_!b$ZFx*$ma6vuo6KCyTQz+eXX)VV)}EoP~{0jWcdGw#i5>=>jaI^qgn-'
'cp_AJY%|Yo!(FI*Bed#UG$oYeOp7_-qReUN+V$A+0t=hm)Pn+*+S8S-nWl4duk6}NV1`wyfyuAD2+W*IWOF=UVZAp-!!6%fiejfM9%+Ux&toE6'
'SH9V7$cXKByCE~S&Ys~4PH&VAe&TEmEaePX>mwiF4~$Y`<Y1xA^rL>+70-HZ22LiTv)kJwP8*viXt~}tZ*M2a3-9(8t3)3ie`6-uqFcvlnVAmi'
'*vwfn4?iK>JZ|wx|2GSBw=m`uh04SWN=hjNoI5I9o!sVNmRFE=PU}&|h@4Mt37|!*eN(~1Jn&MAZ}z7VbAZoI$uUb>KN}wv3TNiPrUBJ5Cv&@D'
'bMVUqoy=!O=Wg!mxsjb>c))M$SW>!y($g@HP2xN~+IvtF2ygpSj7TLa{dnV&ME9;~rITno27_3)_Y$wjv?<E?i5Ylv(z(mn7bLXz0%qUPJj(5S'
'e<_*0Kf<)<ooUAXTr#hRHA5;B?EBG9mRnn8nP0H0;0Y_3tu&hm3Tr=<LZKF$Em*orXd^_Mu!kT84KSVBwQSWo#=9$Am925fSNr2^WQ|@7yvou?'
'07_GtvDcc*(&9JCNxM)@77F3#Fd3$jL8pB&<eGrolF6%CJj<3x;L3&EQB|(BF4!6T9D|<J3o%-tVQ0ljXHRmg^OEc&kk%K~P|8jG#EV3d(Q0XS'
'>)Axluyato;DMIp(32+FK9RZsDZaDj9)vEw&w6}fU!g<I#$cy#9<RXt&Bg$vVP1f5$824!3dRkNdB3ahz(Ur_x8Wu$^So{&&4Z-TteM&3ch}no'
'Z-d=^i{1Sm>VLoE0la7Fpli3w7Co%Wbz*ZU-mlLUFs4=Tv5lq`C@N-N=1XfpnMl;yaCuXVhb)Ydbacz;e8N)YbfJRi9^h#Y6w&~pqruFzT17sc'
'q?9t#3?1d(f);I4<o8g8T9p)OOsT*&ZV+h|k9YfE?oL!Noy)zZc(vbYdjKn!FxEG_4;^T8Dzt>314|P?@E54FYK~37I*E-p)>#3<Dhg18X;ol$'
'c(sflq}hvsy0X~!S6H5u1z=F|W0ZO0gz63}v}0))>k3{iDQRG8!+3@Ec=Tzx#w$=nud>`JW+zdj8V?QKw?DkUe)r3Dov()O^|$XBI=p}P_WHXY'
'-n-B5U=O@%$6jCj_1(qK7r(mOo84Mf(%mqe!UCSzlyP37x7dGhm$In)w)0KLbGc<Xo#toBIU9KVgeY<kf87qAOup8cGINoy;>&N}T_TgzZy0rZ'
't3e$%%XG_a)1c|{Zu1#9wxB+!L#JnyGo&6mwSrft1AY!`yU(jvR`K$jlHGsmdHX7&iB7x@m3%>g{+Y+$U?-8yj2UKP+AzC{E{W<_L|H@{@Iz^C'
'iBXc|-Wv-_ryl)J`Wd@)TgHqWIkWi&`FvTtR%Rs>0ht=*C!THkXHFRrA$)?+x*rqfSSmNVRx4Sn_t^`2Q-T`B2XE|VYtW-GJndRD&d&>aysxox'
'Z{NEXaWgfmkek7+d$31xbHv^6T|IFN#RMgVie&=D^KxNAj~XcY$QA802?O15|LGb+QSN(pfSh!QXmQhDNdG<zQcY)ybly%I9CvF_jjFqyy-CHf'
'Z;#-@uu;5del_Zy8dc0AiOdF65@8MfyIn7z-yHDfC!ZTv#|c0PgiWe(C}$e;n<`3(wl8M^IcFY9X-T(N=iX-pJW_1U2~idqrg)u;&cNkaUUhrf'
'+EQZnu(pSXlZET(%(ZaV@5uR1pRZee+P~Az^qenFkS3Mn%ctjqI<P_LJn_QWJ91R-cDe*yR;U8c(I(i4dDZUnx;oPC)sO~bktl)3DJ<!ATUHXG'
'*MNRM(-gY$oZJbWnPlu`Akyz+TiWXPiz&7y6MNnLO!V?hv;q0Y=3aHpCjw>CEt12~5=xDpH~icX>;hLP2ubmJZ~Z>(Dlx%shPk~#;UddH?a@+o'
'Q7WYBkMLIpfrz*otY+OaY`1FBnt1sQb=uoaM>;#hKN}pZ1U#W#ixPt_HwQhZAj-sz2xnfK7QCQkDuB6J#^yyPyiWvs(9bX>@5^4XhS8t}6e<r}'
'o1<JHmNR6XvpyL24!6+M#S$}z&a<V$;~BhNJ)H~V$6fyptp}a|Y8@}lbI;qh=h8+`fs>phRkR@G*PL}u&%KjHlgK7$s8ra3l()quehhX9rUBlD'
'sP6<lf!0D6X^p5X9g(Ej491|BC?{Z~Uv?0k-(#$^A8yV;QrUC<AS3Wj8U|B2UL~}LvG-F@W2Z{dr!v&S7J&mfw5Kb?*w{EZtJDVM3Vl80WEG#%'
'1}L+5-EA4kTfS@Aa!antx|7i1a}x7B%!a>cv)N5k5LFH&$ayU*ju_o}-+Zp=^N{S>cf|TiQ&pgc<!G^YeCX^7pkua_brrCDe)m75zA;iQo*gVb'
'(yJsX$1_$Z&7Ms5H*h79!*oMAxXs|qX_0$4Q_k&s=UL8`Z*LDU%qe47>98?7yeS_>4=aXyrEjxJ=f;#8oxz#S9i(XH*Pgi!y!Y8cYPSJ-MC7cW'
'Qu0uo-2`rVr?exe_&@hg{oy;DRQQO#+iTlz{0N)P7xLEa<`lfyvft3l?5mKGv(mGZ^W5&rF+*jFXAdF1X5QX+l(ysX(SHF@O9KQH000080001D'
'UEA-QN$U#$0Hq}W02BZK0AXogV{dJ6Z*yf|aBpmBV|gxcd8JxgZ``;QemCI%!0KgZP<gPxzB!!$X_}yUNH?3LXn`b9I1)9ZSR-ml$@bLpfA4pA'
'k*Ldf)6~1MN95tT@!U006ulQ(mbZd&wilI9yksIPnxfjXT9!q6V!4t>)*J+@+Z`6zi#P10Jl0YRro`{9(9P!Z;_~7nAWE6yd{!utHc~NtkZqZ<'
'N;d4kL86hYsB)p0xE1QeWDRmnAuF6wZ7D1zS&7SwlB+#E3vrc8b!0Vm>Sn{($3vmnQDy|*w5rmEGsj2Fv}{!>fWOE@m5P}0D!aVc@$}}7tL!QT'
'7Es*bu&jx)21DSAGkxS`88dNTL&T!-vdi5!hyohCVJWZJ4!pHmWNde0b;+v=dWu25%7ii>d7Bl|kY&HUfL@{~y1X!QB}v|bW054Rfc8`aAZW%Y'
'K$8O9suKN2somM;RMUmH<rme7;LT2b%6Qe_R@~lSxfX9_CLqh}LX=q#$crSkx0dMiAdbAp_Qu`jd#gk`ae0xJTx;fRBYOplkgDM2JEN9WPi1|5'
'dBN}lF<<35R8`=bo$kDz=z^eSq3E9Pq`JvVd1p+9PMQnF|4?oZl*+16JjH#I>2OwwJ8yi@bS(i0Q+Q>JC?_d_z+%&axGes|>54VX&0iaXt?5EH'
'V5z7KOW))Ux=uw=H|DyO@C<W{tK?t@%Xv|Wqo{B*vaPaBFR+(@xW#sYOA^;iH;Qdp<a{ZzAkL!_lCaW6bL!#JQt-+_rEQZ~-SykTE8c-v`%a9N'
'^+2@}ko#6Rm|Iz-LP9w`fXs7$zOC!hNW;}Bxh-_dOT7Shlw_94Uh(=+qyvN&rM+a&GCxAh#DWh{WYwuh)#@T$VkN&{T5V3HNZ{^yQHrsalBqHj'
'++5rxC89~$gIbw9wpxhvkQfD3%{ALeSpw)agcc(Rm}Lh4XbA|H1el*k`&!_ZGw1=f%gy%Y=l=FVnd1<4OIX+^Jbob}WSFZ=<h(7LB<B<nPg{am'
'JN8}d6aT|G`VU(yX~ccAa$)a!=EE`hS>_qaiqv3_DRO-IvSwG$Nn6f!!F;7tMQ#A~v1s!R{<XsXs%XS<vlq=OB5Y9%qhGHX;xVHg@*_jViz6r='
'vpjn6Sz)INkw3DBfx=_7f$s~o3PtfMhSBKNAMxF^u-{K~k1<_TY%k=CljRn^pFxaI_K|)Jt23U1{hXbZb%ULvUabdc3KWK7Z~!3*@Gyuq0~z!8'
'`3wlssb&IZ(SEUizy}38Bf<4R&}r3JEKbLR;TepO^@!X09%>^c7&49l@K0#?{S3{5Wap4LK1)%p>@s`4WxvG1LTpF?No*)ySph;rFttGvhSR4k'
'oIJpI{luQ#5kW$EOY9?ui?5rINU#6l<e%UPylrGOzlk2mQ9~i3L1{GrLj5TUmnzA`Q9^$z`n+4&rB2JdK_wzOUf127*dZ5)E(J?BUD_%wRhMyn'
'*hv&CU5W(TAqC-rdD*qa`tvSY#~~er$*fHqPs2TSyY|#+#FegA$OK(3IMp4U4VPV8?c{xjh$~dLgJ))QyqQc}mm@rGWNC^+U9i=0l^&1^ItbJi'
')h)DA>`hGTeErom#Zi>VpK}^b>teYlAn#Bgc7C~oifi*rcL@Hf!f&mCdfx(dM|B!v?i{VjHoekiweQSISDv5?-R+<YC>E@#rb9_U#U&?3Gy1)t'
'VVR{|4;r?Qou~^<w*ySbv)T%Mkl5Liam?|U(uUOUG4%_bF(*2YAc@e}Asq4>p%yg8%Nr^p4uWSMjay^1=1-Dbq$!v6uyB->EG1YAkvhjEhpv#t'
'V}{C@>M!zaW7XHAAgRPg3%c=@gR>q;0hsUg7Ta&u!HK|Q2>U2V1jW(B_xt+jDaW7HTR##2W)<206$OT48>4l^WmAek$OKIgV(}b>-Hx6@5c=T)'
'f|Lr=pwe1K;-Ui8RL3PzqU?kl3Xy~gCTn;I++b>jfn`4~u^5!M(=~th$~GCv^o~}e>s+t|S6|{Bzh>iNfpzR3Xe40)^v6LyQCrO5jZK|te}<mt'
'y%mX0X2OPQAV-_xTW2^I`byVI_}+&y-(Kjx1$yP1&Qk0-PC1GD7WEEyK=r(Bo)s+9Rfif~(3{F4sgX>Jv_<0_v#(!s(V(P43nv=AHO0f%ucmnU'
'`W1Ee1`V{>{)8Ox6D=!&_FQFPOV;|Cs|wK}^?iJ5I#t7iLiDB0<Gc?t>V*dFUmW}7N*jt3SRd7_@u+qj*Xef8mgDiH0w+B4f^j|X#0c<wL1BZJ'
'JJ!d5;mj-$jl<6R`y<VhwZ7NT*(QPj=8<VLbc)8sr$H**9@@i1+>>#_cQtw|q`L%gIx-dTNdHX0Ugo_gt)3&Y#M&a~VFSC!`xqc-ZWs83jZr7C'
'TA3I?j}MP)Te=>&wm7I|vGOQ^njsa-(oD!MGd!1Q(>L43r%cMAW;VdwLj*=LJ{5#bZzdbmhp3WQk(sL5urMI9fq~s|-4CqF6RK*OdSsi(*Y)K{'
'^TY9_ytcq)cxF*&;hCU?cN&`X-s93BF!)~7ze&>pnM-oBNOG+}Z2CoUHl=kp>chpG(#D8AYNfog!&pc*;@8oR*wv)uCzMR=FKFun0J{98afvb8'
'$t9lwTu{q7&b~uIF9?v*DEeyjR5Ufq3T-ak&)GcY!3M@weum9Ika&ErXLT~QsV5Z!yFj3gX7vW?=gFGm(Ydpswish3!>dK!W-u%|2qg_aLq0H!'
'*g7p93Z*zKy*xY9$=F2mb9!g_U5@zSplvzmk0Azje^jK^_$H&etzfN1^+d`T4gTn7*$k(wo)L2)-1_3h#;}anM&qG{=l4OJ_27#EkwBM1;RuC='
'TXrS`H*oR8reo$~g{k!tS(T@u5;r7>#(obyYeU7(s%K@s8<E<+7>+e(^bDD^(Wa-#KD_?7Nj90ht6-%W;tf@{6O!Wy9GEvX;LA>EcUV*wJ4H=S'
'c-3YzX;n!fS!zn_qPhiNZ<3e4zJ2%WhgZqle|~uLGI{sn<KOMT&d=+dUWeE_bw)>b!lUPYG=m<SLbx;B(incpq$U_2rmp2;-Ss%u1rzUha$*z`'
'zyH|Ktz8EXPtoyX8UqjgqLnlyN@Cl%DSSBkx*k0yIl5UrJK$$n*F&e;JIe%g)DDjxlfkK`hwMNI;OcN{4nXUIO9=RAYQ``P#q)O#p>dzBZRQ2s'
'ztEpt(HJaqr>rNrD(2a=jG?67C}AoeZeMQJsyISMR&hn>Z$+Qo&t_wrKy8MNsq+23MK3q*&=*ThSJ2ng$~HQ9H*m!f{3Z3CL{Td9g4mP0JBW&L'
'Y7?^}1MBo^QK4GXj9xh^aiEwetB~(;3b6e4`6URLPqX~eU_o-dwQuO4JXZyi@(B5@(WE%mXb1}JUMOl}mO>t<%w^tN_PouA<k3rO7|hfsZdzdo'
'JcK6m_$-M2BdC7)F$j@v8glLp>nJw&G`1rf9bjhDtZn}(fErq}?eH<!4_lpW_z=MK)%dpm6k=Gi#_i&2aB5QjY$wm51fkVzM^Eu2yl>6#khlDP'
'i+}N%sJtCL1yJEz#_f;_!sQU>C^Bp(gN+&7FzDC@GmjaV8-8rf8|c+|F=oGnbK)SCZ-dYAf*Wo7Ps2#q59O?JB=&n3c7$@@to+;QGgdnp#70p>'
'6FZ(#Mc=0m%{0&5cR|v!&CH0=eF5W&inY>z+as7QId2KvA`iZB&<APe8FYTKeOP$k14oTQJcH9boEr|~2+c|cbiMmbky+>QaG?_ca&R6G(63DK'
'3{kdsqaNv!J5A5`&g4%`oQfne?lZ^~deao0b*UZ~JpqJVs^==drxkLdVJ&p2$Ia&|z!}-EZ&b3nLUcyR<gUB5vrmv{m@6$ArJ|jvL|)jhYU~8-'
'(@}`he3OUX?!a6Gd`I3OFQIz0I$}j8_=de2sA)6%v@eaRBDRTagmK7;QuSXeVn==>OmaT5wyH#mIvIB{7Vyt3!n1vwZgPlo#cL#lMjRib(Yw^a'
'PQAyjd!PI-P)h>@6aWAK2mk;8VqK?d?E4l9005yT000#L003cWUt@1=aBp*EUvYI|Y-x0PE^v9xTHTM^xDkI3u>S*3FDG4X*Ly*V))xhBbGafw'
'Tcl~;jD^6JD4U2Z`ABMaUDyA6XZR)R!@HY1=TJYaEpj-V`Eh1AGukMMeq@S?9k01mJSRW5wBqVO@-53hOHww1s4XLc)ddqw%xl&vK`X)vuE<_A'
'8$oxAD2lFLl%m;@EGt{p3YKMr@Ai#Qgw}PVsNzj6uU=feu%mL@DqeMu?Je-inUt2#o~mudZyl2l@Mve$VbAN0oA^t8n31=%s_1RSkiS`dC}@q^'
'SgOBL$^Ow4teTOZII9XzNO_hwKqFbkg`<84s(q+vJ*JU>%8vTI-@i5EAKA|>lM1<By~rynCHa?u@x7ESi~aV=>gol7k7RdD!1RhpB}hu5dn#(s'
'Dl*eKRczA;Gi`@pdGNRcr06CbVRyV>b<Q%uO1UBxoaP!PZ-Bd2IWAa9+e&36%@y!Yks=92{s=7C{YQqnO^B>j&p>VoN9hHZ+vWA|DW40y_|Ml?'
'gXZ&!+%`?+);0oUEvtsAp*-pJVmTulmM-zTV)&exWgNkCx`co}sgy=c8}N!0Jl9Cf&{oc`L^8WlFgWiW8hnw#EL!dt0t!=v*K*Htm7$|#doZWi'
'9+Z1tc|)7IeYF{_MuuhQzw>mmsqR3yiXNB%<7{hK#B0QM2{Z|rzv-Q(Jwou(w+ISRe8(zYGgxy+1qWA=<d%V}>yhR&`WN*sZ@l{%N3y=L$X9$X'
';n{1tkWDMVH1&jKR1ydahSZAJ@n6vW9yX7APUL(hjhGOPh-ThR(#R=<gJ862R5w|;CyZN<Nuy5?%}nqD$$-386pjj{A!db<X%QvX8rTCKBcN@A'
'9fG`AnmLS)98h|g=~eKXu)5tb0g}a<PZBCu5_&83_`)*Fq@oZ>=A&H_@&>p0L$Lgk__*yibRS?-M*r?0AV;m)i4+Z!`ml_w;KFzz?9t<!R=PWY'
'<gJhm7|X05iqdr%*`ttb174*v%Q(7m5L<9GFqRh7Bd^omkly|);en1&@xmF+JxJKxKZ(qiO1R#LUF__~Eba{Gj1l$B0HpVRfi^Otp9kJlv_jvf'
'6E<=wJlq!9j>{cY`8M<*3|quebmpPyv~85Uh72KaCvO}{;vxyUj*eg2X}oTCT!PRhvdO;h=&=wnG?4z_1x>^VGBOYoawqKL?^>6mkq53*rZQfP'
'OvD}Ml#O^5AIR_QuX|poEu1pkiOXx^5L+C8AT~d@jBmCIr|{l%q`w3Lmx4S3f>FL{l>`URN>GG72Ltw?U|&h>LTw^-BX%Ihe^@~@pP^~ks0yhj'
'+*iG@+Sy7@aVXO?!$7MC7bar8CUbBTjh`7t+j%qLPyj0uUdDiobRN4<SP!8!haQM;HKEnqMVgAUK7P)SoalX1LIgjvgGLt7GitjLIaILp#0!iB'
'09Xj$oH+FQRBr+_CxN>-cy51>pB7*WJ+J(+C-Z9YvOLKFnjz3<%n+d7Z?T-1Hbgr;v_U&Own8`Yo7kZf&Vy+O9yP+hS%veK{ZbpW8J^QsK@<Qa'
'1mDNWS5l1a_Ue$?;V0F>$P7szgg6eRBhm5&r|)lIu#Z}DT?N3cy`vAYu2X88-+_BvC;sm4&N5s^;hb^B<ZME|Nhf#v(rbMxfCub6Ox>6%9j5O1'
'(p>U7MMj{G=fyvfP*rrAV~5VkWYa%*1<7$r)l+9W{orp2&N*Hv$0+{WnFSr*0Kvpp_r50%;4vTk)T#f4E;wP1`=;XgVX<K<j%2&vR}52eDh`>s'
'<7JeP6zX}bLAhX&w%Msd6RDP$Fio8wbfDOcq2ktg1K6ZAD|&^SQG+;Qf~r7)iv_FIgWR?X{aUlJ8Rk7$u+ycpwDb6wB7n99Z!9cO?MNo>EG`(='
'x}};f%UgC4k2SkGR19w;T-MftX+dgmpEJlfL8VK-H3tof>J`2={F!MGz8Cw@jNNeq#=BUnvv87YNQ|u;f5ln!QKNMdbbVaN#O4m9rVuUTko$Mw'
'YTv)YAR3?9#$hzMkzT)imH5VuYDL{|WOYZ<d`^3cnP!Qb)zYQ{rixhqo{^WyY<Qrl`3lBOn<+i^qeT>SAIyHyD!4Re1M0Bq>5-o)j7AK4+2u5Y'
'ezrVh;F5yrwLZZaE*st@qm^NV9xnEdb7pPlSfiyb-ocu203Q<Dz8b`q0&>7w>R`MgYdgMxfVyv0ri-l^8H}0&>n7lN3Vu(t3e&Z*V(Xn#c8j5B'
'VCvs^k8)d0v=U86&YCoK$XB{CJ{x2#4hP$DeP(wNu991}^1bx+8d=}W2L02UVK1_+vx1rKG&X#}-#>|#Ig!S_PV!&!&!)z>fL~rVR4gkQ#dine'
'rtNUelO0aqy@c|TSJ;zLTR2eLxyS25N5gu9UC@}`H5}?wO7cTqvE+*zo=O~o(7yw%Da>j>V>~9`NzB<^k$3tFuXPB)Ft!TK>9GKY9|&SU!7axI'
'Hg2L9M%k&XC0##1Ut#&S=6O)`Yc;Zx0*v9M1BVlX@X31hYPq~gLI<(x`g|%ib{K15%oXeXH^H22*|q%3ZrN*lb6n5HrmX}2pT18n(E1uF?0a8w'
'3;6d^vX9QFyc|dGY^Jp2aPi_YhOf=Q+9&ef{)%2c5+J0bQKXB5sW^qO1=J==O_H0buL6{hrqd-vhn6QK6jFj?C-}EonQxQRLpW$ehS9KYm_h-T'
'O=^4Up{kpc0ajGH-0t!To6TD@H|ldV4~7x6pMQZfF?@|0G#^YT%+9{WI-bT^ELtBbi&3&)4h;e<5)O!dg=JXg_$mdG3l*gFy-m;M^zJ<q=x9=F'
'p#vYehm2#Zlhqo-xEEa!*aT#&4g@5HlBgK;+95m+6!XP}=K=tZn^nH0kh$CA*_n`D-b=C0^LR89(~N?wDP)5wvI(VJvEA@JfXOsqqTf+}Bm+D3'
'ANw|k5c#SMHXmU)gA7g|gr4jj^^h?LR^mI;Sy#D%j3bT<lwOYdL-;W}flRzs@nARY)|UU$Tqwzy1HM@?OkVq)CpOZg7xjZOI%?rg85nnWoD-2!'
'SkuRFXJkDQyY`1c`DQhcCN82br$O!CWoEi&HUDt2&`uRcWx3n4O)mf%+Cmq6?w>Ihz+{kstfv_v>h1vyV}6H!9YoQb!(Yr8bT)(oB@WkZRq5&I'
'W|+^Y0^(#ENb=$Rhj(Kl*x?BxZ2Kasm&;*pq|JPkP4zpN7L<ia%ygRDXHqZ)nGt~ID2-4H^B701D(8R1ccKV|L>qzQ%UP7H=hw^Sa&>c=F4OpK'
'*868>+rZY0yuVSYKdkg$!q~4V^Z)Ov{GbdqJbek;XRC=#Mz@+s@`jW)n_Ep~cq3Y$&6ol#p3Zc}9Ro0ryuSs$wugFKPmJgXF&9?$uZxQo$UzPK'
'KTt~p1QY-O00;m80AgJm;VKzr4gdgrHvj+_0001CX<uV+ZE$aMWnXe-W@&C^ZDnqBE^v9Z8{3ZCHugP0{(-A~h}6}nPP4(bZXF;^rf7f(8ce$j'
'EL=mNC3>umEV-gI7Yyd#_Z(g%lA>%+n$<%_CaJ@7zmW-o;K#bFa+Z_4p_}%sVK)W46@Pe9SEO#TE#qxNTl}@D8}jx8$?Dy{<^>KV&!0Vi_S=>b'
'c31GWsIJJ4<pupZqkHj1Q+F-nq+vT+RHUogx&wfDLVnv8ob2knD;a^e0w6Xm%b!21tMZ<#?@0?lD|QR}oyq+exviV)O<CU(z?3zlXgS#w4R1-#'
';P<Ym5QPjj1A-dBEC>QbZ`0H}lBS!k?HZP*YA-^os%`~c99QX4eqUup-T(fB*H!<ctgo(s#{PHB`&Yj0+M*nOcI$mpXN+^T|DLwnvRLcQKfzBm'
')86lqZ))Ogbsv*=v@GemWIgt@NEH)O4*c6uy*zyWjKK5u!#g>B$^Ps>Ua=bHoVE7QkCd}t+0~9!dgR?8%}*t*)aWJSb$P?`3wqBQ7>mZ3unL&L'
'9844*=S@>L3e-iN(ei!Ku&nLJ(r#bYG*3Z2_MOIL17xphcAf4AJb;U9mI|d(BT~Gnp@jW}KslvIHTUCOG#XzVDN3LsFIutTc=5h2i|k&1puyt@'
'byLt%&5ytPv!i9v_8%4%-vjN#PH{gZX+bfgYJpBKw2mcLti2FpVPJr6*amox(hDN|?D?~tZAh9c?!vliQ(S?b=V2q|J|8JZM9$ujbzPTp%{MTN'
'N{lEW;!|@(fmJtQ@b*Lc?w60Be);u0{rRW#<1g>eFM^l^L6m@JDE47weAN)OZ&H@Kv};Oh&JXnB;$!;n%R?}Y?=CLhetet$dVcwz^UJBPOx7r2'
'q+mALwW1G=aV~67sSdkCaj-niJL&4?1RM=*d(E0)B`O(mGcU4s2`I#9zbnO8a6gw_<v1t!7`-ftod}dTSu^mfOmBj|fD-M16XC-UTERI?&q)bz'
'aH|#Q%5sHd;%wX8&j*ePbAfroctWH(($r>b-;#6j2fZ}qIA#D)D)BIk2!iVCU`~FbC1<fSBe8?o<tC83ZYWq+J|_==YY5Pz<^0ubwt5U!#<!0e'
'f6aG5S};BoNTblhK+gpcs=_RpDKd)9@YxS!NAJQ}@-iOFl6(apefLdd&?nJN=zb592six}Dc^#{&L4U;NM3CozkV322aZb%RG5Sg_a*ALMc!@!'
'@4qd2vydJy6*HEO8=q}iakXu|-^nSZ=93iJ(gFGka2cpieEu-(^Fko;B6_?t2jvbh`hDPn(R`Rc_^1IeUSuB)I3~*={8~g)p3*Ka>R_6#U<bd!'
'Li9onZL`_!*%c^{48zIow#c?&AcuoNU*v!RK4)2t+0Vj=*h$k>4hNU=LvS@`H|TH}hBtgUE8t($aA96Ju)*P~D@!qLX*)i=&4D=!W4BKqKAn#j'
'!45|jwIEXj1y<$E2*0v$?bPt7XgQMbL|Qt&&ut*!w!ngcF?yv$Dv>|Kpldg0-vW(-6?_pi3_@GRf@pd6ayFaIS2nnj#R)K}{2B~S>}HtM*aJ!L'
'gGwV|8Ke#FXFSz=k2xlzu(uwF$*ag*C~bA_F<*0)GR?WqT&;DzHqT{b^=rEI4RR#2LEjWPt5Z3l*FGB2t*V<H#5YV<Ah8IKhbx@NrSplrPwYK+'
'0-BnG{5#hX$X>A0fzSauM8bC^-R*T?1A~LqpOpd=($*r`r&$Z3n((@7GWNCfAOu1_E9*TQ;vjD+<Wvg+5ml!+i5N>cs8p5nScQUFEQwhTDzGAF'
'7(ZS|IuKU5f;3tg_l_49Y0$l!Hdpp4%Vk=vIb>woI#%N<tf7jQ?H~sf<R@#2S-t+&GE!WZNQR_{>kkV>zJj6W<lvhH66BaM;0-B7$25`#qG``c'
';Foxh1ut+X$)|X{OB9Xa3~Yoq$$oo_e|Vt5(kL>q0(wn=Z)`LjvNXT^0S_O8{hnC?!ys#6Z5AK=0QOrLm`*Hi7=_YtAxM_*Q&2O={^UCG>RA@q'
'Lv9fDD336#g<Vl;b5V8?OM4#6lVcSE8XS(*a|pOai+R97O_xPK=Ce{c%=A*47VsLpTyMArCfzP{mH*76@}@EMIL1a1AaM08vh8fZ6f6u?)zP*!'
'2=LlxL4)Qy*m^7ntf1c~npArqm2Z4FNeN8rzW^h=EQF8!ec0`D+Oow%Fdzw`#@PCCtYzlUfk$VHV;E~zjTH!WQXq~lOb^P%DrYn*0n`G6QOEUR'
'SbW&aItp?F5j0nA1Etbt=j5RG7GI@Gp?&UK`R&=@v$eB!2tL7a%gs!;ZGVG7&mIx6O@~95!zviY240arVU!yW+U`J#ds1DGK74p)VeeIGQx2b_'
'*a#VP+SmtEb)n}y9v&?>bk&w75No1jD$7+v_fTW-L#m<I{9lwpkEl&7rI5MLQzajbfu1ICgE;fUGY7W=BG;VKLO&{E3vw@HWA{f*jwOj+pn5>9'
'k$t|!WQ*@$<OH(iD!^Fu5Qg0rP_%3|ffJV#kMayac(>n|qQgg<`xKI8NQ$RHjTCka@TM*@R^MQAaRNxcVgjSh-Q=e*m$Dn>*@wU{YB$+b?MoHi'
'($d&|fnr7V_9n^|QSPNxw_CFYD#NPp*10<=qR_b59U75Ybq;7FI9)|5uvYcfH4Vl@L>QI{_DCJhyWKv-*F}KG33j0<&Ruku624P!X03-W>!tuE'
'F3djDCg}SuV)pU(>i5d%wFLjIJ3>Wg5lO(lVl|x8h{}?nrmN!=DvyBl@d+^@eFDp1%?*Mo*$u5)$t++gS$FuO!TvM0Bmi_wwsn0S>+lYLOPCa1'
'!-OeRmx!i;IE^4}Ln|p_sNvKB61-9HC8Ao!PJ|EzC%0P$bak9<y3)q<QtBQR{r!@)h;qLNk==L9nJL+W>i`qSW^lH(fxNF}pP{-D_=^lf^jpB+'
'?DibpDF^n6vQ-xA*5^MXas?ODN-JbSj->2Zwxv~(!OoiBc;q&>L<gcpO-WF5c&HRJhVqW%5f<fHQK<e7a2E)nE%so-7x9ajyqLd;BHeTA!`t5p'
'p3-3!q%DqYDF-&Zp3P=wfKmhXH)w(Ik1|M<Ak8)*gC8b&&A0}K3<A{Ua>N6w_@=8e_<=20@Ek>pCwaQL-3bjumgvKZ#PSwem3?kss5drJ@b7Yt'
'D6bwP*SCDJ9>`(0^7$PSA=!rR*x@CUeyi<3qn9qu!->GEKhUu~Ex3)b_w7TPv<dBYtlc9^Le@E6=mLxq!F@A8#)?`cDRHz)l)S_l-VKEy3$f!w'
'nf+0(C-^Q+)H>2RB&I{cFY&-r3&=VJJzdCnPz~{-ZY_*%X@m8Xd4{7U>w`5n%c6<>jG+7ZNz9PGDtTZA3%>Y@g)d|G{2>GNz*6?5*`)7JHevhG'
'xvG=GH<n2CW1RTvM+(9jlzQcSE68&TALGPWKZ?AC`s*)s5R+8;WT0krb{iU}xIIG^%kxy=EP1pjSrtkeB0EV&&uQ$E)Ync@+)!|4)=9dj4B9}b'
'ldV({!v*){nVaduO{qSpm&Z8?eDEGjq=w8T#YhxJkIpv+O$x50UUZMdi$%0X;thE@Dgm6Ymt*9_oEaMYtu!4l{ImE35PEm|JYWJ!zji}&=HZn2'
'0A&)l9<HpW_-)Z{!+?VzSP&hC!61&iP^Q0_0QPvaYlsi3gV+g8+?5hOE3<x=URs=v(OX*dmt&H)PTOt6s9T;-Ud_;t^wTGp>d9Wh)}9l*Mem#a'
'fZWx}XFJ1#6Y%;F#fRRDoW?!-Qi!u7#*t79WEqqgi&H#w3|7$;a!FsVpTykJO`8$Gg^zB7;0>dA$BHjBqVf>SnGGhIOapcL&W2=Quo6+tVChs{'
'HHMW)m*8;PzJ26z15e;KK6n5O$;mQMoK1_|ztFE_k(V7R@avw4#>PD%Rm<EIOI{Q*O;M5g)d_c6aDF=3k}RF~x73zNlKhoy?29DL{*>#lC3-t>'
'#~Z<;0~;UA&yHt)GN+fzCh(lZ@4$}7rW6W&#ehja7x_+K86?A9v$TNHP+pZZ&z{s**vK5;9g-=Ma?3p{bt@8vhRaJTCpvVCE&7}dY%2>sNFqcR'
'LsLfhbpiI3L!Fw@4&rcC6JhOrl}hmZ4Evc-&<}VeMqaQQEV5^qN8(1VZnlc>nwDsknG@bjN>6H1MYmVJ+6xp1$ij3E!jABIo>AY~iN2JkliJJB'
'<@KVF=v;y8b)5A$s$7B3=L_s2&yw%{72{>BnLHByhJ0n$o1?JOGgK^FO!VHIwQb$fQm*B8s2Szq4$F=w;=wy{Z&+=o`b^4b2_}62NnCSKsgvmO'
'1lHE$3u^9Bmq73-$6Z>u`uxFEFyGOaVuya!7giT8KbEtV4B?KD#|4e<nb<fDfKN+-@Ud@fdV;73&SON6hv^aHJ!|~AktIL-uBH4)RAQ<u#D6Qu'
'RkROy8udmi>tyusF-7i~w%f|)<IDK)@g>jr$YAWlZQ3{^Q~QryKE+2I%8$YjUh>V{s~}?QQz~YGyb%R@53kDRqLL(#2a^(iwJc#7cUxbJyRDug'
'jELLeu*g&)f%7<8OdooF2HxNMtNd%m_PP_h#wem2<;quCH+H}I=a?#|tQ_5q@X^KYTBOUh3<H!C68#t3hAn(nU|Heu64TXh%f^p~J$>>9Dv4e2'
'V{z|W5%*rnxcBl4#yyD#@B{GRqUmHgfBS$inxfZEJmOKV5vzTk?9)d7DdV^t4|p<P@gAL|dN%q`l(o_8#;|@&*53(Pzy9*FeqxR@&Icu_6YclI'
'Qh|E#bDKE_1nTJBVGC7b-u8{BQR7z1;g~ZHMw8f#%wdwr@j-1gucrMEP)h>@6aWAK2mk;8VqJArNf}uU003n%000yK003cWUt@1=aBp*EUvpz<'
'Wo=<DaCxm-ZExE+68>JG|AFWI(g1GONs}VmUJlzU+C_oA-C|oT4p<ZlEm1ZvlBklD6Cdu!-^`GdXua4@+6B6?Epjv*&OGzXkh9tB^R^ZlOI2kx'
'Ppx6O(#);}D^$u0_T~fI$V{l`sVZw_q?M{-cS_&pg}OU^`sC@8&oH)BnJAbvj2G}Th_i~-TEPT?vf%$}`5yt&=WN+n60b#-2`w~JRk39}O+{_Z'
'(<d8V$c$SMAuD5fl?rppSoe-fY+#NkC1}bk_WpfYixp^Jmb}VXAu9n&&1SQwPjanFmLz#&8!eKANzh+gm{cjtaTx}+r%`rW%W4%yys5Tx_N%l)'
'^P(M6Z!=z5nTA3C<VO4pHJY>cQWV+T>6du@mByJ*pQHshhP^RH*iWj_sd%fZT&|wA(@$PKeZt^BvUXC)rRIA3iY=8Y*fsmC8{wYU;$9SH?18-J'
'OFlNGsZ0&8U$>9m^TPONxfa5xrpm@1-pN~8icIp|hm8He>SkGh5lUe>tdg@gA3wZi6>T=UiDtKA>x7j2K1pFuXg5t#3!Qvz1g=e17Db}fo@FBE'
'O<}LE=4>Ud&*rQU*XN6q*m!>`nF<BF{(DWkHnbgD{#Dqqzh)}WXLB}ND|MTobG2VUn?yBMEBqyIiemN!6ru4>kQMj#6*E?||FPdxB?fBZk8@gD'
'A*z*KU!PwrP+Oe5K|LQf`$aUlXiAt3QN@SWIXhjDeHQo^Ep)nTqEGq#1C0Z_2NN~|KSSJCF0f5&FTPr7VGNES8@1z)QFn2Ac{!r*h@0;i3;)Ec'
'dY4IvBgu=83Vd!mTpEP=FQUSA$^Qaih%nxPnJNr;rY`+ciek>p8l$%{;Hpb4MNSZg!afsBwKKUE3|=LdAl?c@s0_q|1ptlS-AiK;Fs!Pd7XU~F'
'ySb^sUc%no%$>NKn@#ZBn;VAMbaP|!d-$BAtqkB_$z{b%QzM>;>@}fcO-qG`MinsC$pUm+0<al~M!QCQF5#nqzo4Sj0IP_}0G8UC)7CO!UAQ7q'
'*#s~y@&osbhVsTzKv-Vf!lIJ3;2D0eRhDvN@gqd6&>Jc4Xk6p0Jex;CObUQfC7CEm7QjK)q%=2EDY!}n+L2>z3bn$&C8)($Vn;~X7@3myRk9?A'
'4v(D=qN46QZZ+zPilOgrBM%SjDJ4oN@MBsCUXe1}ikI#k8>#(6FfG0<o%P?6Ha2k^G9sS#TW>@nyf-99LRhCFfoTRuwEj36P~B?`C5D7InN)jT'
'grBtlL{I5s+1R0|*5^j#JQNmV4Z%qZ;de3FNYil4MN_W;jdP)$zF!;g(1E~G;qE3Y&Fi&H_lPp0K>v2DUklK2ut~gxnTeA)BFI?UTU4v5(lxk@'
'i9KPU8T#O~J|4O7<BFgMB6n!6{C--(DB4D7yw6eWwh#%#Rt}!#z_Sm%;cejjA<3NWq@1(!0L5BhL??t_k-DWw*ARcHb6dC9&W>6_M3YYb*CAL>'
'W+4EiH=r5C8T06Xs1@+`6hO6xNRyU<otU|kHU&TqC<&_q+?KeNfWQ&DiHa`^kqwE8sx1(W*DORQOA879ixMqL1hAEm0>qkcqzX98N&STE9GRqE'
'Ygi9?pj@pB_(#Zg%QaWGBJ3?$rW@cH#afiHs2xmLsv?V#iX?O+JipV593okAJx2euN=4Ry@PyoMFyMHH?nDiv0^G@MdQ$MM&}N*)d6q&s|Lpvv'
'o7)BY*tD}~+SN($b#%AH0|-LTDETdb&Lj5T$tqKKi4ovX+0<*!oqjLen_jl$3&W!)X+p&LMZ{BIw&cxO#s`#Fd6ZcbeKu=^+q#KX6b-4k*1U&h'
'fwcLpla&W+`p{vw`wFxsTM7SbkKne#lVt;gY-g|u;(6|w=lN+cT1(|RARF;6$bp2i(h!j$0J)R3%Um2{d*EWhY)LLSR&1aXn9Bauif(fo3Cr(Y'
'67-B=CkymomcZIOAkE#$W>n}n5e~d&U%vG1kmOJqE4{r&jdtMlYKMov`S6yiWbp?qMi@JagCAW9rQHx;`2s;*ZrR7*KYeCDl14vr=}Yb%ylbjL'
'LhTD?LW+pIVbqB{rKq`GkCN<YOrwj*AwYZuSguSk;UqGf9OxG_9CDxlx+IivGKh}vX0?tRw)yR?2KNwH`pXKLFlB@}!(0sh40a`g{~RB=#)TvP'
'$gM%Ls#a+8QW2n>#_Ti7m@nd=BV`whg@1Aqz!y1x<o-k|6H=k&oj?!)T!3Wi5jyNsAtv=wh2+}F@Y}%k5wUu4>0|ZM$LiVDqKj544-Sb~bam!Q'
'$gf#{(%!vY*=RCA>Z$>fe~eFb)<5s@6)(&|W|FF=vis7A!{!%$)(`BT*yVum$~4dIbBEFN#}NP6!X81jm%|2<ljKr^gSh6@X>jfwO9U*UXrBos'
'Ch|j+$u3R?(_K;<3s}FpJDFj`j;?NT<)QfUrCZ|F#bVbsKxu$^Oicgjd~$kTM+_WO+&Pr*m?d8>&?8?gJU!C+#HzjJz2G303YugoEwbe91!w11'
'b9VL;#Dh-wu9x@{_}o^k9*C1Zx$E+NE+B;@9NTJRw16HYEj|v~Z<F{w7`fB7aANm0FS@aipbm~iH}^94kgBUOQq{7=wm|K5OD31t8@D6x)h>-c'
'Aj6Mp0X!-``#rl!N&SHVt&qxgU9HLK-H<@#kPz(6X>P=!2G-(gr-60m8*a3}MLF_<Ly4{>n>d|X#gA<jqJVuI)3eb>zr#uSx!2BqhToyQM4+5`'
'O+XL}{rbdHfM-y#=O^s>KiI6#hGwt2v?`&-YszP+{WB2Jh!Z+@ppOs*jKrKqG-l4?U%>wUhdxC)IbnYTG>KkC7|bP<g@2;-cUr@0W*=R{8tOxE'
'rDDs_Kq_ak{J!Wp0}UgUf*HbkP@O1W;&Di&tg}5f00#s9e(e{vlba7|9!mgcwot^f6|o}BV7%H*^<zpp+QP1{hYY1t6hiP6_nd}BO^Jj0^+W=k'
'<B1ElTv;cEeR}`zST(518(q<xV?vGm`IA^-@*Hw$zZJCHekAXZ7(Y<pX-98ZOjp2$28)@|zKUx^0q<UJ6W4TA$md2w+@wm20bS}K=vJa8EopS>'
'AscMVGsgt|gE*V6z@`s9C3svo19uZqFh@JlkK-sV!Ni?c$J=wrX`=b#J{<`=!fz;#PZnYR94g-DN&ea4Zg1D9xS*=L#6GsC?)Y0oxc9o{A%CP+'
'FB&|0E462_yj8X9AX%$Q>f?x<N)8>mQ#CM6Joto#`Hxil)`j#<ypaL!z4s4Hi#TVy=Pc-AVvZ-fQ}eDaLOF5Xn;3ZR(e+4QKzqIrS+;`l)Bxw1'
'vlIjZB!as8oT<C2`xXrDzO`m2om}gCBSI&wmDPLw;d<=6HTb1p!)4(7HBaf_;~*GZ_EvuY=Sn)~PW-{t6vi!~;?Cs*PfNS$DYEMfk4uiIMSJ>-'
'jhhLSBM)67(Bc7Io31wrSd&0@g0hl$>tkhoCy<R{YYb&~;Poh6rSgJJ!|IWE3OxJWW;6-+Q1Nc9lv%sphfHtXIb!W#5**e;l<`fx&=KsKb+a!%'
'MD|1HqtBoB`o3q*0#`a4WyXU$2_Fky4UN*>BN)jqXHyplMl;5AYv$Xo4Ez?T#wT}vJld-KP{B@NZ?YvOuNfW?(9sRh+Y+n^dBY#Bn{9=6edx5E'
'*vj1B)>(l+8<E9*%y5+`a-FuT?Hbs0M&Wwc4>oGo+Nan9m|eL%iuLvWd=V<`%b&h4+H{ux#kHcxJqORp+)bkAj%DIqDp>fBzJblsroB2vEo1Ci'
'g{xUg$0qJYCLZ;6Hbusu?!Yv5Ej1lNeQgvVQp(-86fbhQ9)4<Fdwea@TU=W>RHeX`rNBE^v~4^KJSfcmUQFl#Ti>OaG5X|b6Qnj+ddT3Z!0Qd2'
'Ny6`*T)7pwG2eqecsz{x0Zod|cq6&qyXg2QO&u`J?zqOYs&?DLHj%I};|RbDg-nu(l=`7#sP|oi;Jc*BD1yFEmJGr5&3W=|cS&+Qz72#?$rC_D'
'v+F-*y*P``e11Oj%yICuj|gceuK>LH;1K~3A{<4G?ke5>c+{P@uJ5eXH%VB6fdV`S$BIhLFUMegfO_hr`ZiRM6*H>G5JYU_5+4Wx0~30Hu=9&H'
'V|&&RgzeCX2S^O38{KiPaalYF?q~N$$nH4V;6B7Lhr1(fkM0rx=@>=vV~lykDM1Tw!0^T6V5@f*^$&M1FZ&hy^~30PF6a+I<I{llAU1AD#iPIv'
'fNM=Qt+};YE*qC~CdPDKjidS%!`$4@#k<{r-8z3*!S5~2qm^{I(}9=EoCgbIEQKcuU!b}peK(x47|Vgf5_j|n|J(5UxfTW-YXZi*r!7?hS-s>D'
'bax)t9ubif@A%<<cMi=KM;VTef*nu=d&wLS)KAW2N)qOHRsO(jiQ#!wxW2Y4Wl{(Zi-aW3Ru7;26-qmB8-GjZF%lqBB>rq3I{E)lO9KQH00008'
'0001DT_WO{l^zKI0NNS=02TlM0AXogV{dJ6Z*yf|b7f<6X>V?GE^v93Sy^w~$Ps=Qp#RX(K9Cbr^jN?q!PpG4ag1PrtZl@?=0O<{J!EmFS0tP1'
'CS^?&|Myh&!E30)6d;j()O~#2y12Obxz?asRsq+Yf}GV{Dc%ZMGhND7v#+8($PO3CJ5leMR7~E1`l>`rpIUY*1C|S3%6$j%`Puo|^??QyWUW+z'
'X050|x4ddJQ&50ntz<1O(NN+N<hYahF=p2Xp&8G4)8Z1iZ+M++T34N)$vT6GiK+tR(?WS<J1C?Aq*v#lP>z`W6EYA@OFUXGYLrv#Ma@euVJ4-@'
'ai9f_LVlE}PY&hf9%?{!QHqAQ2huK+RfW76N(1+hbvTC`E-o(4&k7|gmZnA5qNUOlC2Ay^j@Pwpjmb2rZ&u~J<ypzK2JM&iL+591<iPcz6ubVr'
'LJ{14k_M;z-p^*neSGD2&@<v9MTr}!A=MdLi~B8q<r==gzJj`q*xwaz4kG(p@;YL_N|6CMkv@sg`PpAa!kItpvx0_WJbYGiuXwdNKV$ebxMRcQ'
'?l<(D9Vz({l-{sXXg9*UTu3{>{>MJb8tiz*n+9Fwm(ouT1#2v$R3k|;VW{9g9R^Hfm_?33#FUin#uy?(3#b}N>6H@;z4PT*W3XuPUton$wB1EC'
'%R3v-NCQ{Y7$7K#2|AV`BgU2N$loRpbZk6LL`A`zC^5zz#&SHQMv|~V7DYe-gU&ai4%59ZPCR-rEIU=t%^FU1Bx6PFX1lWc<_?A$LmIq;Q{Aha'
'-g%cIk@t4>2c@JUBNHW?$_ab;<|w1wqJ;{zeu3;eG7wisBoMC2F0?g9CZacns9Pli2Bij8LT&VlcbuZq#-Lse^v2I3HLN#1wa`rS0?UG@#a^*D'
'Pg9#N$5bXKk##N2ir<HCqQ?Y%{rYOP8i~6o;!cTt4*{c6LB!YW-C(D8kCEO?#a3LPG-HQ_!B5PRW;Fm-Kt532LHiXjfZMOqS2a&Kn2OtP5w42A'
'fbTU@9o2pp1uQewjma$Qnxn1Tyn>#?n6cq><7c)QEo3$yf-cYVQ55bv)U(Bbb-W(p(i4dK2-O(wkcWnTOoKXHjZNtiLh8{_j%E?<(FqZ^a8F?q'
'{CgeXA75kQK5y)s>25kkp`gA)SNxp5|Mbh}PhUQy*VkXtUq1cq!#{$E1;IEvGLB^z*-(9AuUXLBFu*TY;bVQ^S3mwC^isu8XV`vFi}(Qdx!9vD'
';p%2{wcgGM$JH&i(vU2NZceUMhXn%ev4W=Z)_l%IAp={PP?%o7biIzTZv>MtA;;Q1vCwHDN|b08DY^m34oK1jv+0>wj+$fGHP|vBc^FveV8d=E'
'y(J{8C%ye6b|=fOf@I11G_l%#n^g&@`K~l7`8^{Ma)4%Tg3+cCdD?_Df}$(SV7hn!0Ldd2HFWeMXN<_&Y}AFO6rEvv!kg1lwUmD!hum{gXjX9a'
'N&eO-FDQ5BX}QC6ycJCY`8W{?vKf>R%LeO^ZdGXAyu#pByI{q*run(AA(I5U5}KnVs0(aSfXRlT<c7=$sTEEuj^F!^#cy6vY?hR@9TJs<s=>rH'
'Fk-CL5A0tDSz>!PLSaC5R2}t!Qh%oafq?;}4bR{$dVnDuMfjEb5cV)29w_=D)R>@Gj0my}L$As!7GM3iV!!W0uV^$`V)Qv`k+zh*N_wKhoUeS7'
'(rDV63ndIKziZWF4@dFOkAjfYu1H7nuGeGZFBrZUS-$1BXvPOiWfNZ&-}GB+PV6x{QwB{VRxd<r>VB}?a@|t|4=)tdvKsXoJ58>(94z=Z89A>y'
'^OW~Xr)@SG#)mXUYByT~R}(ZE**GIdsC-y6tjjuu?FT0eV4SrofjsW1Ln&VGA$+9i6I_|Jt76#bV`G*xHaHpsc1Ka#N0QmUot3>sTg0d*o|3}R'
'aUitdA;m+PYLj*3zy!j}Jn_t;Q)k~I(`3}%oaxd%9=C`Jrl6K}f`KqWOW)FA%W7V!Pu=9@eY%lY(XDTuZ!0sdMr_X~v_|%%9-|SBdg+N8Kd|?B'
'Y;|>6DGQSbw3i4ZTq@B47fh599QAc8$I7>_5kzMA8BeJqP$*5;VnO_y6m{6^AM2(R1~B|-i@*MOHH!oj%U;3J(J!<vJV<#PJP+d?3O@~mXCZr#'
'@v_8$VD2c2F#G=Mi9j>XHrksrU<MSlNs!@hG(Lu1Eh@7&33Q2&D(xD>=~zM%@Ql_43uIpqpD5uSVKhtz1%<W|HyN6{-jxw=?1z9%9BjsE5_nF*'
'X2KXJ>><!5Vz6;mda}&2eiIO%xiJH>o#SINC9pY4K^|-f2+-SV4zpa5TkKwawR-8H#-F-e^`5L<EL@`k=hnqTMPirpL1V4&DSdwbVjp>-KP_)`'
'=2!@{=z?hd8sni;jTRgnP;>D#={>R|h625fJiHt<Plm<!xSJl+>L#;x-dxuR$Pf}*O(~(r?x%YgDovQ87&Mrzcf4f0b$2Tx{7OB?+#{C!h)@9V'
'4Wo<S(k{t<{WRRd&_k{uY?q8~ljxxqoRy-XM(S^uPOvyFxUp{Bvfp8)a;{u}<p@t&QFkzF7ZKYTgm~|@wj@VB+aj>8utJk}$Ft;&?dkDgcgYdg'
'#p|mj89JqG_Jo$u)Zg1z{PL8&XF6ZITUfhr4}&6k6OXS-=xT%Sbm&C7S#K}hvYR*C6+!(e-W)%J%R;|ry^TDjH%n4b>v)9PvVe~)rspW2%>AC{'
'By>_uO5XGAW-LY-g1x#8c{@&w?~l}Eh*aZ<Q%ACQ3FX$ilYm<~aN66)#VA2|iexSqI6QNd_=#Ez-J!Av-Xfw$TVd1J*ewkJGF5~w|4UKBDY}pM'
'7MK{&jYJkN7hI9!QD95qx0N*|^7J>Cp3D<%leX|^XSUe#Mf_|s(;HPWGK0ZQGbsryrRMf183qm5@uNdc>gRQFpJGj`+b5OdsF@8GT7dzxpxuZP'
'MTl8vZW{CBcE*(c+0dpAz}lk>K~r?S5?~5(i4X{Vaxl#et1%q{=Z+dqN06uXUYbamx9@S0*+6PI8mB(6B&`?hyYYh+;F0TcuWU4xus_AQ+`?e{'
'KTt~p1QY-O00;m80AgJbyOhHo0{{SU1^@sP0001CX<uV+ZE$aMWnXh|b#7!XaCwDQO>g5i5WNS;e;BAJK&rAVD^4~@>%(HvLxJ7w2liGJ1}%+Z'
'!V)QvRGhTvfA0(>d((}I%7$c;!}sRBp~gv)ytkEV!9Sa@N3Z|H8iG^C>!7WHYCsRl)L?raQ=sa08chISS%*Kv&JL!gJzw5_$de?wyl9;5V7qOG'
'FgV<9L3h1%fiR2>iZT4<#WdHcus;uov$oxN=z|-oa2_7jBTm>o?P~Sz1pFz2@TbP+g$%)QcX2s`gEsXYV`fkfE~aq@O>0$<cb6AH7v=uWp$Z7r'
'=?Be9KfU_^YN)jZZNM5_c&sd0QSAqF@HyODsO`}Z3LD^6*SF|L6Bv>C^IkbJEAI}qc48d(%~v;I2Kc0X*xExJ%5e{+Mk~%dzy`{7Q54zaOa3K3'
'#OloydDLMaUCC{aMv$~9SZ0tMcZme$!49Dx!laL{aU%B^C==CrAyyfb*~Cg*$^FOsUlV##t)Q}9@6da!6Dhxe1S(@o>1h+h$xBEh#*3)NI<QSc'
'RBRq;ctAk}aq~zhUnIs?bd_*-&jFQl>RHHt2QZ}hhBl&S&}kzk*op9`5&wIFm^={}SjK?Yt7smvB|9mIQOTUJ%!^WzylH$^ya<w(Zsltb-Ne%d'
'n++u0;B}SE<iXbu@OjXzz_<3;K9d2YOH=84Rp;<^uRYLLLGNsZ%-g~a>kPOy4``4Rc|?dD@b<TSp2b-kJP8u3t0KQ839D6+)BVjlzv0pPCcnL!'
'k#X8fTO-8pT<@@4OZX95I&g7P{xQGtY-GBD{EF@4cfpf~@kg&|N(4e$OF~T3avtVUwF4K!51lw8l<9<#atf(SgN#SR@zLNKt@84k@7Gc;uV#we'
'o+?D~e<f9`s{<eT-uC>UuA-`t{QVzAJEb&}j<_j`G%T+9PD#me0iSxY2x(c&c#N&b|KXulkPk+az)mb!oSbnp4a?KHWHTuuU%!TpFBs($(9~jV'
'uhgEZgGhealX9+ar=FN)JnS)r^i)d8#AQh>Ny_h}Ol{@pbijuC3VIQ_PCX6jpTNX+czN*;P)h>@6aWAK2mk;8VqHBF@1Z^f0089(000yK003cW'
'Ut@1=aBp*EUv_D8X>V>WaCwbY-%sN>5PnDEf3W)E6onk1WpC9}9)N|tN-S`@z)MwGZZc^t65D4x1s3tY-`Gyt!2Q^Z5KSiInQy-NW-=O$PE{*Q'
'D2b5!+Nc7oC4W4EJj#yu?=R6>fdN@SN@~Tup9RegcThi*%BY686_8RGD4a6cXf)d031-Hs*0ly0BcjpDIKrh=jys{G-QC&U>8#6QbE|@Vcc+Ak'
'lwb=Z7QiaQ8?cFx)|s~Or&AJb<H_z0!3%RvrPT<4w3;|YKrB~|7~pm1`5Qjez%7t3H+SSQytiZll`=paJUOCkL?`4j^Z9+yI}4WF8ZIq5HDD|y'
'x?R+w7?AVJN$b>2<V+cIax=9Vxl%-PQ4)})RzkWo#xu|;2c%FX5O{*3jg_oL2?~q#c0*QBYcMInU{zb!`v$D4gD0YtK8JO@c*r(&$D58m<iA)&'
'OX$GO^s;|7ZTJ!fuiT)p-sep*DLEC+p2+g_*?~P%vJ%TQmd=jZ{pE!+Xe@=t#_KO2!GzBC^fg#d&5(v1yPAPq3!~(Wo=n-<_1By0@8|4(`h9wR'
'#eTnFU#~x(U($Jo4Hh~XcQsW6)e>vUinipLu#B&{sQIFXr0;#8mJ96cw^lmQz_~HXB=k}h*!{Jzp1S~sNPyB3PR{N=pLiY6x0#|+eiHVo?^W6P'
'lMN(-bozuX<~6Vu@YpBN8-?U=NzQyPm^Ogz5E9CyXDua-kW7mwsBM0D^l2~rp_M=N)}!N((b`wxJ8V-Fm_(^KOf{RQWcEVcvj#e$Ugb26Ka;^Q'
'z0NyXc+L5eaN@h#W|L3z0UwXTt)h@Jr9mdaX+|}7D>@$&ZX<@li7l>-YdlgW;VDCmYcMXM-@csE@pyZn1D_#C@BJ_6tI;iXguHYBe0%78k{wmA'
'cJ!K(&7qKtzRt%3Ng^*o>K3OTh1t69?n=pSI-GeP6wn{}x$gg?l_iRy0M>AwQ;zMY>05?^#98S<=TYtzZd!NvF(uqN$#6VtXpokWD5tj6{C$i+'
'TaGgQ5j<ECrZxZj04IKfGgQ`xg>d<;X>qA9<RUz>5P}61R8}9}XZT;&CRm{wq!#|Es$suTWY){WtM!E%Rr|KJ77X^f(*m7zH@#2;WqxJn+rjek'
's-Ic0;#dJ7U-)@S`rs)t`~Xt?fc%2j7Hj5Z$(EbeVb>6C<YRzsBa=;*#HY<?{y&aZ6n5UvoyMT}{peUl9kfQWxU_9fwWH(Asi=x9Dq{R@v?5EJ'
'?=ADbZ}yzOwwjHhx-uZy+lzqXl92GoLe)0^_22+CsF>wmN0EX->WFke<$^me$FSv+F5PUB#^^H|eo@!fFYFc9Fxx+zPmmA$AT;w|VZrM@cc653'
'(61PQBU2CO%THIAA|)IAzOZzQsHb{+Y+2w6scZb;Cxc1`p9++KLpktb1>WN?P)h>@6aWAK2mk;8VqM#UiHE5J0009B000vJ003=eWNBevbaG*C'
'b7OL8Vr4FHd4*MLZ{j!<{f@+cxa%*ItU_igwbBSFE1{B|BDDi*VMZ%dS%w(!ZemBa17&CX-}l-HO<*5I2;%FT<8$uIgpjG^1G7^WHq2PA2Zx<7'
'CD(9J`VX#Sm{<dnS+VC(@IvV$Xl^Sl4Oj(s$6@*eN&<foHYSAhdmF6^pmbBA7pD|NQ7UbLNvSO2C~5k=ey_3qZj^jmDD(Dcj;28>ncd}Lb%L4U'
'(!JmBWqboxGilaZta!kxOsJvXk057qTd^%4g0UL@g0D*Q2!4)$<+(agUX=DYgq2b`B=9TC4G&@P3*^GsWhU14dD!m(emsLZF!U6dhRFAN`JghM'
'J37w2a0^82?gjOh`zRA{x#e;nkcV_U8PVJO$&7x#n>@|ZNBS@wr}rcRVud`CFh=I03_?)4F-0v>#`j{)lm7=Cr{5;GsY6_s)dirlbnw=yC@agU'
'J(lmG-cBE8i+M`tzs)?hkoNNhxHom#s)NiG%cxe$z6U?Ly-VqM^60QX+M@h;UY8d~{hLQuS6NZJ)0|}{DcDPJ6#<e9-k~sC(D~hCIvPJY5?3S)'
'!w#mb3R*XZNv;}uSpONmEx5EM0*Y1Ot0C@`Xp4ao>Vgg1T1`$5PFP*AE6xgP#Gf1wat)bLD++EbD@v2hb;Uch_88xd$T1M_MEKii?5hl4$6#VR'
'aE-wf5`NS+FByN1+_*m_$?I9A$V|8{ruZ#2)}g8lwfw~<zK<fX*j-#R*Zs+-_|tiWoCt%lS+<Bjh%*x3s=*<43!mCH)5rofYl}V`tRstFe(g56'
'%sOUVbyEeT)M~|6A{X}9aqC!7`3BA`AaLN}Jaj1lj*|S-Ym2%;c1@;T*iK3J_Cv=K*J9DSd~V;y<Ni}94#^A}ac4AYJF4qiYs>~3VV%7`(z&};'
'ciOcKG!rQ5v68NsQmvH^$i?ulM?7~Z9unHRBTXi2m-@hVAQw!?z|T_0aQMb8=Q6;V9y7h&hn|}&xa;WV)rBB9-%b*^@fuF1<@GcCl)#6<z$+Sf'
'MIXX{XnH!LiXz3UC?J*b^S?aW9bQj^0&lNOFxs{wj27d`6sC(g&NH|lef_c+eMw<?cF6NduJ6_eRAq5cwoQwr{6_L>c-<t!T|z<a8pa1L++@WL'
'3EXWLXB9><=!^yLeK`%<A<#);au%l~xxv8%T&`6n<TfFd-3&fE!v2EY;3X_s;jUl|D52ElhEmc*t<F@x_cu^W0|XQR000O8003fLeKcxq;|c%('
';U5415&!@IZDnL>VPA4(b8mHWV`XzLaCx;^ZBOGm7XFUJf3UJ&(z$JE87?ym(~+2&G9bawEiiK>gd#U_TMtQ`jU8Up<-hMaw)5H)Sgmd))W)%o'
'&+Fso7{_tGh%64s@o|)}1&LTB(kCKfA|o1xWGSSQha_QX#1&zWNyvTpvE++oFJ)3>sn5u7F_XANB#H$+9-CEt$8ol|=BbE?=gl*frOfjPj}nn8'
'LgQE{s<?>d_SW`RJ}RGLpNrzz7qMcG>Yk>u7!&2s-%`ZIgIs2chvj27OH$!8Sx#3hO6ELd<p7VE)i9xI8S+`K-UJ>|Fa2&#rJ@PX-<@1tO~#Kt'
'OHg~9<ei*;KOv*4w*9Z>_fML-PQ(IEJ#?9uyG%M=60kYVLNz+3p=2Etw7msmq(0GOF`b1ha66>;naqR;54N`md{PD;jfvj8zhKI(h`6UA52#{|'
'O<gPNK+Dkre~6e_fqYKvdx%Z|enzP*zIUDT@wd|>@96C8;+J>*<LvbO^lJRgn~pCpuBTtemtc!CM<bk0KNJrv1zjP!1QNk@hWJ*c8N=@<FuY^8'
'(>72pS>2)T-rzmVJgFsEBmzdbBmtL3K}a^WgaOIYdzLyC(`7;*V$Zl-QF-V3A{E_+Q>_E6CN^%32>K#<^6UWmwlcGyX0dA=FPUlz@6_UBQ@Zb`'
'3<FY8$qF0U?}`mxDy27jQV_9(L$r9Tbg<yts;ttdiW?ueT0-%$*qu8&nDsj=SogdGX{$0!H5U6K;PGPQQ0a5-bU-a46{r+d#NcNxQWEl*Vencf'
'*i6AWeUP6jiMzYb&uDsic~FaP^<%vO7Pw|9013GDW_a5n@5pelyJyxhod?9dxE!acNW0`G4Kt>HS4Afj>$)O%Q1PC>NJW--2}DE!=$Oh;SrW1v'
'*uRTP+>-xPQy-9OYUE%rC@7ecOwt9@l@M~oK1jJU9%qllh$r{E1blTrG%&XwlvD^zDMS+2*^y6jXD)ZlTBlP^i|8YdvIu6ndc_WL3<me%4HOiF'
'An;5Mg1=LqxRu2ah+_Q+7Q%{#|E+t_*78Lrp@c>{7|l)nEKOOg>Kj*f#FUb-HsF3elC{ifO2t)8TW}f9mo@oTU$Dd-bd91tt*G8u<DA1|>MMTF'
'JOp!um4VagbaY%@>yFO<luh-(y%M|~V^cMT^*FY<Ri}{0V^h@|z1xWC=Mfo_PoK#Cprxu>Bl}8pbxyxcjlVn4Zw<Brnh<)MLT)`(ihX0m*BqFG'
'N6P-4aUd$j9&ZVTfQb=mv|tAaiCyvz_T@3|>qRq-&furvhyCV(3czp_L|jTZ)`-7k{vDYy@E)KF32BPx5{AOpEa;HL3{EnFkPAGB=w<Q3Vh^Bo'
'7VDoZDDj4G<_ZVRiZahRB!a%{C|L+CX2AhlF~`1L;|0_4zphWGWAE$t>BaS=jSN7dtk#LZb2A>A3{0~aX_<}~9A_)-_~(NhbuB+NZpmVLPkD$o'
'AA+6pOuH<1N6$KpJc#tpS;H9fj7itE;zot#`O%{31&m{czTY(nvu3}s8|lz^`g81^e?bF>yB{$__1U)f@y`$YMx@OfD{_BtgUIvKAKvBF(e%nY'
'86SO{UR<0Tv5<alnR=a;4fN5B#`N-f%B}d6Z(ebGG{elTuHO)d21zKx!eg1u=g<?tp^*p2Cb?z@7!aqq+BL;IT4@z`c0iA3a?ZclK#W$#gzAem'
'8P=>|w8{;vPPb(oc=MFOKDpb=DV>VIHsXBG113C^NG>)92l_D3#ec`+KtlmI&(RZ|meQZH-^|Ax&1433{$xu&U_iydY91$<BJ-3+Oy(XlJ=o4K'
'V7Ab3VflqLivc<Q&!QJIOk&d@^&ghpheK^X*W*rE_DI?F*cw5vr9!?E)yv(i&jii63C$7DDR{7iE(iB7Ix@|l%S~>yJcx1c9!rPeHu$3_SqLul'
'3=c&f>)am<oNgr*V9ruJfD3>b>88ptlSdCcfM_ilxJqeRVvb+fG1@x9pmuI6!lVjBrbZPtC#RD!P7Xk&>iD<u&mY%kXI<Su;+pqxd})9;&72il'
'y4LuhZ=!<nf5pXLMHU8SKy0$;RD%OKVlg^c<{q~G0T-irfVwo0uHDiY)iGyj>ByVgdfe@sSJb*R-I(Rb5>{HBR7(8<VvXyG@x3J?R{qW4_Quig'
'om)%Ts&8z#a^0Y}()a;YW?}K@FYomH==<24oc=XFyL3=}tVYd#c?6K;5C#FLE%-6HzRDL^6fU?LqVTW@!ggOsm)*gK_xl6QH41lW^~t9Z86f|+'
'R_2f9<-2_NbH3Hfj%{MwsSfuFEqf*_E%W(tWn?~Uio*hFE@*mq)sllsFX4|YT*u0yChgg<vRXYKVo~CVf*3AY<5~wrx5K2TvKYYC7f}Qal54*Y'
'K)h5p5Yb!H6IbpTiZ?~it(QOu@J%kE^fI6fVxf~?BxojXXF;xX`Y{0K8sOz(&G<ja)e_DIG7srOk|>i{PRwV4Ehvd*V=|{Av@T3_!v`c&!OulX'
'lci=4ES3El{8n~G=))b%bw^Jvk^E{X>KQp+%Mj=Z?dW6is>w}JC`XAYjTg)fp|#Il1V{R?t;+_@a{usKEC{U>no|suKgdwO0y4ItPjAjerfqvE'
'sH@7NeB$sxPrGq?kW`)~v=(6Z_eyfo>9%jDKW&a0zT~aqeQn^jb|Fw?Lc#2_{95Rxw6%z?Sv}AZI%A<e>U9!Vj)1RPC195mGf<xUmm1KBok&8l'
'(0LYcd8eZxl}Lc#{}!|S*yoUmx{R0j3&>u<L;I*a%7sKOBu+0cO;`;h*5{5S6d4yqrGU?3(NMC;D5Q@9upmpC29}-~ff@)7VG`ta448qt`V{Yc'
'co-7l`&kO=sRzblD^?=D0qO(blgX1>iZ~}K{la$h0}BK@jOxu~z2@8T!+=KpU}_ZGruIw+DkmIxd6osRad~q5Iyt?ZjHmiQ0RJJI)+l4|3BgU$'
'PdU&bthrpz>Pd2Zd_Ea}udXjQ%ex0WO#ZpwJWWb<@Id)<zpX9!8#)brCz#)O_pcb<UDNdG7@k~x{RdgzeQ^Sip6ZR8&Au9so9Ii&B7BxXX~~u~'
'!C(FMC{_A)0_&_F2?()>x$oAxGYTvQrqpjX*6(P5$c9B#^b-+=*lMn|tQCB0ZVFJEYxKFO{7v_lRx9Xzbh72=FsxnKrOz@K;<O8yft0&e&012u'
'w!3zG^;$T$ng^ZsCCEBq)#oxpH^_d?x-#4vd2D65E0hlXluf#|rs|Eh<L3>x5-rvhtFTn_2cQOEa`6O5Q_%pwV19cWywOSS3uhT26(1thC*VSh'
'^YtqX_PC@#NsyJj;5H+$Xqc;Zw)D+a!(Zs0*Ee{%n>Jhw({GfA{q3S@!p+J``Wj?|<aGhK+MDv`XG0?3XG3b~YL_8ni96^I*Kb)(JX_KtFD`}u'
'^o*%$&BAp&YA%1ET)9Pu*doH-RaWry%FG@=CfHTKBL9vf;=MtK#+(L`du_=aTo8s$V|4+(5qV!zc+<|{iN0j)&K*<+;Xtv2U9#IYZf}$ESuw2{'
'5a#{&ANT72W>~+JXi~cJTq~|EivGQ;l|%CBcFbPpcm01etr^zR|5;@DgE)tHYo}W=ixP9bfO^&pk_NxAg~6BDlVF>#J5cP>*8hWIEZpAuFHlPZ'
'1QY-O00;m80AgKhQGqV11^@sb8vp<j0001OWn^h#Uuth+UvqSCa%C=XdDU56Z`(E$eK+9$!15(lXFG0@0fRRUOIvqK*VKXI02_ipkZmeaBa0eE'
'wJ|*Z_g#KSq9n?7+MvMn5ZUDAC3()_y_ZhF=o)!mknxNW&qLulp^PIxjw$zfNaM8I>2`E0O^c5_T$64m5Tp4f3FF&*<Y~MyMlkYEMp7=FbUWwM'
'HxuN7o8c|tB>vRH=aV;Qr{25iEAMhXolQJpf%^zEvZkD1XP9ypCOu~u(a(hSoNnj!^uv5MeeYeoKm9QAUY%Y}ysO#!s#m*>@N0Nv>H|K;;I+Pv'
'(~C3j&&l8W7psk`e!HFd+4*F8HTNzju#PvE;A8DS^j;1}eH290=e<2HYh#7$pRnVA*V$w~JDcR*Ui8sAjC&`26vlfU>S4msOX0{-Oe`ZmP0=}7'
'h5kFbxa5?PakqouS&;yF5ncL`mx|_IN}|Bgo#K(4hjGCCVM16MrZDl6<n+ri$WM{sq8!Tz$Skh*8Ph7(y(Nu<@K#NX2EQVnB@wv>eEZ;@-56^L'
'0|YuncD@ZeIkz9C1kJKoNO!^*Wo<%1VJh9Kb-=k;35nF^Y!Sl&leTGX@2na*`-1yztEr(Y&tk81DiH3MU)O}+(bZUhY_ujmzpJ-BH$Hz%Y3wcN'
'YBNTwaLE-<5Cv|~zvu&v2`E;)j1hhWr`bChos7|f(g+OR`cax&NXqlLF`SV!fgnV*GXvg*4@1yB*}!<Qkzq|f26!10lLUAx#_YCYM>|}0-w}TW'
'N(Pf{l|B5|Kmt1eU<cx)kHvibn$hWUNfHiy*iVuuT*|O{EMUQg>8_%n0<@g<+UAU9UE|j@hBL+o^G!nV4J@-cQBJgY;AnG?2(69L<Uy<r%&KG<'
'r<pS3b57O?heHi|jc-X$2MwpTAkH?m7V)s0A-80$DK+y^I_ujzfvc7tjfbn>Emi^!w}7|BNSsERakmz3E!Ix63Fs>n42g18AOI=A=UEB@Tmc#`'
'5(;mb--=}ZtO)nRNaCp-i~T^#f*{|+sux=pCLSUQFL^&?nyMmdTh&&ejV->4?_>Ho1_B{(ppSGR(n{V4jAukP?)FTo<5>HYmWbKz4WH*L(TRt|'
'tAs*I)bo%P5QMUrJS1ev;hPLPxZNSdhyrv@9{Bb6#b`9T*|iL@tHY{?U(k=EQ427|0J(V&&ZMV?_F7ohDCXmuMr8~Deh&aMXdqbt_{V&Hfws0i'
'?OLIH1dO#EYW*7VUw$a#7?c)ZV*#?VWiT;AqIkry)q(B@^aA+MQO@E@pfyY*LXzHSc!Hjzo-tCDPz4iBx!SOln?M1aBmxO`TdsA79WzNM8TVjZ'
'dya-qFKp!K`!H^_DWaO$d;>Ko7(<3`W@aEQi+D??PATBen%QOEuB9=+T-E%Lvt-eSYiO&)ZI+wPu!%u+^Oh(r_HSuc5Lh}i3a)-zEEE#1dRh7<'
'7_R(V>#Ye|oueis&)njc^W~bcIe!3gH_Z}JbHf?6U8CHs`1_3Qenv(3JCaBrAvhyPYVsr8SbbG{LrdK#9(-{%$F;2o_!wd@VX+?_%N#KP#?s@h'
';%!$qFS4||ZEg??PDR0Grxq)2@wlC2tl0&dcAl|f7;IXp#*$^Qu(7=}zE@5%<Z7v6d4A1fE6<6wQhv<A(HX#Ye)5^soX-DKdPY^bSB1Y%zEy=q'
'E$7ZzMyhlg$Q28Mg56TATk9HG;5W*+N-&3RVWmntQ9L#8e8LJA5T;=aT8F!~V74zM<RAhUCp~MaJz0u*B$n@{0{ce3C=)_ek`|SYZGj<VaN&w*'
'whAI2KB^g=19-$2)62Q2AFX6bKgW6pb;hiZ5lpb!E1X!JFsi}C5*T(()N7<&OLA_SC=24g?u<VLh*JJ-idP_(y01z7&^=ewzejJ&$Pd>V3%C#v'
'rWJM<-rjjjKk*k4sowa%8%zgiO|JgYN|XBp9mzLLVD1%~&97h9SS1M5(S=(t;|t<j)K-O?(|Zz^DinBs6`DGD*-le^*K$huF5kN+BBa<b*p-lr'
'KTROTCVN&i{V4oIuwwx<2;(rlI~>h6X_$3(Q9xUQ>31jMkgBOs|2!3Y3f*s_PPq_2gId`0mkKzzT*?)EA<LDNN-oMf%u_Q`O*9pLo(fvM^+CS%'
'M+PlB4ORw}oC@@R6QT*1>gT(XkUfy?xe`9A2-P=g^%IkxF144Q%BTEYj^7dVx>%F=VVyG?4U))*yKBPe!v^brhp211Tm6TKVEjK&O928D02BZK'
'00;m80AgLRRV3QI4gdgjG5`P#00000000000001_fdBvi0AXogV{dJ6Z*yfXaCuNm0Rj{N6aWAK2mk;8VqLP&f)Q^90090D000#L0000000000'
'005+c=nen?VQF7uZ*6dIb7fy+Z*6UFZgVbhc~DCM0u%rg000080001DUF)m7gZd-@0Dyo102BZK00000000000HlG66#xKXX<uV+ZE$aMWnX1v'
'X>@OLE^v8JO928D02BZK00;m80AgLp)1a*rYybduo&f+900000000000001_fwnpT0AXogV{dJ6Z*yf|Wo~C_Ze=cTc~DCM0u%rg000080001D'
'U1Alut?do~01h|+01yBG00000000000HlHYrT_q8X<uV+ZE$aMWnXD>E^v8JO928D02BZK00;m80AgKl!U)Gf3;+NhApigr00000000000001_'
'fflv^0AXogV{dJ6Z*yf|Yhh<+Y-KKRc~DCM0u%rg000080001DUA}qY-m?M#06zr)02crN00000000000HlG5!T<naX<uV+ZE$aMWnXM>V_|Gx'
'VQ_FRaCuNm0Rj{N6aWAK2mk;8VqNT>aUL!N007|$000>P0000000000005+cam4@tVQF7uZ*6dIb7fy_Z)0I>UvqSCa%C=Xc~DCM0u%rg00008'
'0001DT``G%F~bZ102Lwt02BZK00000000000HlG|$p8RhX<uV+ZE$aMWnXP?bZKvHE^v8JO928D02BZK00;m80AgL+@0&^M3jhG6B>(^v00000'
'000000001_fy>qa0AXogV{dJ6Z*yf|aBpmBV|gxcc~DCM0u%rg000080001DU8id7`xXiS0HG!T02KfL00000000000HlHF;s5|)X<uV+ZE$aM'
'WnXc1VQgu1c`k5yP)h*<6aW+e000O8003fL8{sM$WexxUd^Z397ytkO0000000000q=7N*003cWUt@1=aBp*EUvgz;X>Mh0Wo~pXaCuNm0Rj{N'
'6aWAK2mk;8VqJArNf}uU003n%000yK0000000000005+c%=-WUVQF7uZ*6dIb7fz1V`yb<VJ>iaP)h*<6aW+e000O8003fLBI22q9ti*d+8O`='
'761SM0000000000q=8on0RUlXUt@1=aBp*EUvp(+bZKvHb1rasP)h*<6aW+e000O8003fL5WAGa9s>XXa0UPX6951J0000000000q=Bgt0RUlX'
'Ut@1=aBp*EUvqDDZe%WSc~DCM0u%rg000080001DT|E--p*{ov0Obe(02BZK00000000000HlHV6#)QYX<uV+ZE$aMWnXq_b7^mGE^v8JO928D'
'02BZK00;m80AgL+f{BNz0{{R62mk;R00000000000001_foK{50BvPtX<=V<a$#<BV{&O?WiD`eP)h*<6aW+e000O8003fLeKcxq;|c%(;U541'
'5&!@I0000000000q=7;n0RU}fWNBeva%FRGb#h~6b1rasP)h*<6aW+e000O8003fLYf*tNss;c69~%Gw5&!@I0000000000q=8i_0RU}fWNBev'
'YHwm+b98TVWiD`eP)h{{00000695wc<pcl#FE9ZB000'
)
BUNDLE_SHA256 = '5ed4298c1edc6653b637461794b76c3890f1c82d5f1fbffbc8231445678cadc7'
raw = base64.b85decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256, 'Damaged code bundle'
RUNTIME = Path('/kaggle/working') / ('sh01-onecell-' + BUNDLE_SHA256[:12])
CODE = RUNTIME / 'code'
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        target = CODE / name
        if not target.resolve().is_relative_to(CODE.resolve()) or not name.endswith('.py'):
            raise RuntimeError('Unexpected file in source bundle')
        data = archive.read(name)
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError(f'Bundled code was locally edited: {target}; preserve it before rerunning')
        target.write_bytes(data)
source = Path(SOURCE_PATH).expanduser()
if not source.is_file():
    raise FileNotFoundError(f'Mount the video and set SOURCE_PATH to the exact filename: {source}')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this cell')
subprocess.run(['nvidia-smi'], check=True)

"""Repair a private editor venv while retaining Kaggle's CUDA package paths.

Uses only the standard library until pip is working. This source is inlined by
build_one_cell.py so the notebook does not need a checkout or a second download.
"""
import importlib.util
import json
import os
from pathlib import Path
import site
import subprocess
import sys
import venv


def prepare_editor_environment(runtime, *, host_paths=None):
    """Create or repair the owned venv; never install into the host interpreter.

    Debian/Kaggle can expose dist-packages in the kernel but omit those paths
    from a venv. A local .pth file adds the actual host paths after local packages.
    The host's pip can install into a pip-less venv using its --python option.
    """
    runtime = Path(runtime).expanduser().resolve()
    runtime.mkdir(parents=True, exist_ok=True)
    root = runtime / 'venv'
    python = root / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    log_path = runtime / 'setup.log'
    child_env = os.environ.copy()
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PIP_TARGET', 'PIP_PREFIX', 'PIP_USER'):
        child_env.pop(key, None)

    def execute(args, *, check=True):
        """Keep subprocess startup diagnostics, including sitecustomize errors."""
        result = subprocess.run([str(x) for x in args], env=child_env,
                                capture_output=True, text=True)
        with log_path.open('a', encoding='utf-8') as stream:
            stream.write(result.stdout + result.stderr)
        if check and result.returncode:
            raise RuntimeError(
                f'Editor environment setup failed. Log: {log_path}\n'
                + (result.stdout + result.stderr)[-3000:])
        return result

    if not python.is_file() or not (root / 'pyvenv.cfg').is_file():
        # Do not rely on distro ensurepip, and do not clear an existing runtime.
        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(root)

    probe = execute([python, '-I', '-c',
        'import sys,sysconfig,site,json; print(json.dumps({'
        '"prefix":sys.prefix,"version":list(sys.version_info[:2]),'
        '"sites":site.getsitepackages(),'
        '"purelib":sysconfig.get_path("purelib"),'
        '"platlib":sysconfig.get_path("platlib")}))'])
    info = json.loads(probe.stdout.strip().splitlines()[-1])
    if Path(info['prefix']).resolve() != root.resolve():
        raise RuntimeError(f'Refusing to install outside the editor venv: {info["prefix"]}')
    if info['version'] != list(sys.version_info[:2]):
        raise RuntimeError('The editor venv uses a different Python version. Use a new runtime directory.')

    local_sites = []
    for value in [info['purelib'], info['platlib'], *info['sites']]:
        path = Path(value).resolve()
        if path.is_relative_to(root) and path not in local_sites:
            path.mkdir(parents=True, exist_ok=True)
            local_sites.append(path)
    if not local_sites:
        raise RuntimeError('No package directory inside the editor venv was found.')

    if host_paths is None:
        host_paths = [*site.getsitepackages(), *sys.path]
    shared = []
    for value in host_paths:
        if not value:
            continue
        path = Path(value).expanduser().resolve()
        if (path.name not in ('site-packages', 'dist-packages') or not path.is_dir()
                or path.is_relative_to(root) or path in shared):
            continue
        if '\n' in str(path) or '\r' in str(path):
            raise RuntimeError('A package directory contains an invalid newline.')
        shared.append(path)
    # Also bridge distro purelib/site differences inside the venv itself.
    for local in local_sites:
        paths = [p for p in local_sites if p != local] + shared
        (local / 'sh01-kaggle-packages.pth').write_text(
            ''.join(str(p) + '\n' for p in paths), encoding='utf-8')

    health_code = (
        'import json,sys,pip,wrapt; '
        'print(json.dumps({"prefix":sys.prefix,"pip":pip.__file__,"wrapt":wrapt.__file__}))'
    )

    def healthy(result):
        if result.returncode or 'Error in sitecustomize' in result.stderr:
            return False
        try:
            data = json.loads(result.stdout.strip().splitlines()[-1])
            return (Path(data['prefix']).resolve() == root and
                    all(Path(data[key]).resolve().is_relative_to(root) for key in ('pip', 'wrapt')))
        except (ValueError, IndexError, KeyError):
            return False

    health = execute([python, '-I', '-c', health_code], check=False)
    if not healthy(health):
        (runtime / 'dependencies-ready').unlink(missing_ok=True)
        spec = importlib.util.find_spec('pip')
        if spec is None or not spec.origin:
            raise RuntimeError('The notebook Python has no pip. Install pip for that interpreter first.')
        # pip's directory __main__ prepends its entire site-packages directory,
        # allowing legacy dataclasses/typing backports to shadow the stdlib.
        # Its cross-interpreter runner redirects only the pip package instead.
        runner = Path(spec.origin).parent / '__pip-runner__.py'
        if not runner.is_file():
            raise RuntimeError('Host pip needs its cross-interpreter runner (pip >=22.3).')
        # --ignore-installed makes both bootstrap packages local without removing
        # any globally installed version. --no-deps prevents unrelated changes.
        print('Repairing editor pip and startup dependency…', flush=True)
        execute([sys.executable, '-I', runner, '--python', python,
                 'install', '--disable-pip-version-check', '--no-input',
                 '--ignore-installed', '--no-deps', 'pip>=23,<27', 'wrapt>=1.16,<3'])
        health = execute([python, '-I', '-c', health_code], check=False)
        if not healthy(health):
            raise RuntimeError(f'Editor startup check still fails; see {log_path}\n'
                               + (health.stdout + health.stderr)[-3000:])
    return python, child_env

PYTHON, setup_environment = prepare_editor_environment(RUNTIME)
# Use a private environment; preserve Kaggle's CUDA-enabled Torch installation.
import importlib.metadata
constraints = RUNTIME / 'constraints.txt'
constraints.write_text('torch==' + importlib.metadata.version('torch') + '\n')
setup_log = RUNTIME / 'setup.log'

def run_setup(command, *, env=None):
    """Record setup failures without putting secrets or huge pip logs in output."""
    with setup_log.open('a') as stream:
        result = subprocess.run([str(x) for x in command], stdout=stream, stderr=subprocess.STDOUT,
                                env=setup_environment if env is None else env)
    if result.returncode:
        print(setup_log.read_text(errors='replace')[-4500:])
        raise RuntimeError(f'Setup failed; full log: {setup_log}')

stamp = RUNTIME / 'dependencies-ready'
setup_version = 'bootstrap-v3-av18;torch=' + importlib.metadata.version('torch')
if not stamp.exists() or stamp.read_text() != setup_version:
    print('Installing isolated editor dependencies…', flush=True)
    run_setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-q','-c',constraints,
        'fastapi>=0.115,<1','uvicorn>=0.30,<1','pydantic>=2.9,<3','requests>=2.32,<3',
        'httpx>=0.27,<1','python-dotenv>=1,<2','Pillow>=10','numpy>=1.26',
        'transformers>=4.51,<5','accelerate>=1,<2','bitsandbytes>=0.45,<1',
        'json-repair>=0.30,<1','faster-whisper>=1.1,<2','av==18.1.0','sentencepiece'])
check = subprocess.check_output([str(PYTHON),'-c',
    'import torch, bitsandbytes, transformers, json, pathlib, site; '
    'assert torch.cuda.is_available(), "CUDA unavailable in fresh subprocess"; '
    'import sys; roots=[pathlib.Path(p) for p in sys.path if p]; '
    'libs=[str(p) for root in roots for p in root.glob("nvidia/*/lib")]; '
    'print(json.dumps({"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0),"libraries":libs}))'],
    text=True,env=setup_environment)
gpu = json.loads(check.strip().splitlines()[-1])
# Test the actual decoder in the isolated interpreter before loading any models.
run_setup([PYTHON, '-c', """
import io, wave
from faster_whisper.audio import decode_audio
buffer = io.BytesIO()
with wave.open(buffer, 'wb') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(16000)
    wav.writeframes(b'\\x00\\x00' * 16000)
buffer.seek(0)
assert len(decode_audio(buffer, sampling_rate=16000)) == 16000
"""])
stamp.write_text(setup_version)
print('GPU ready:', gpu['gpu'], '| Torch', gpu['torch'], flush=True)
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run_setup(['apt-get','update','-qq'])
    run_setup(['apt-get','install','-y','-qq','ffmpeg','fonts-dejavu-core'])

MODELS = Path('/kaggle/working/sh01-editor-models'); MODELS.mkdir(exist_ok=True)
def download(url, path):
    """Atomic public model/binary download; retry transient network errors."""
    if path.is_file() and path.stat().st_size:
        return
    partial=path.with_suffix(path.suffix+'.part')
    for attempt in range(3):
        try:
            request=urllib.request.Request(url,headers={'User-Agent':'SH01LocalEditor/2.0'})
            with urllib.request.urlopen(request,timeout=120) as response, partial.open('wb') as out:
                shutil.copyfileobj(response,out,1024*1024)
            partial.replace(path);return
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt==2: raise
            time.sleep(2**attempt)

# The standalone Piper distribution avoids Python-version wheel mismatches.
PIPER = MODELS / 'piper' / 'piper'
if NARRATION != 'off' or 'voiceover' in REQUIRED_FEATURES:
    if not PIPER.is_file():
        print('Installing local narration engine…',flush=True)
        packed=MODELS/'piper_amd64.tar.gz'
        download('https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz',packed)
        with tarfile.open(packed) as archive:
            archive.extractall(MODELS,filter='data')
        PIPER.chmod(0o755)
    locale, speaker, quality = VOICE_NAME.split('-')
    language = locale.split('_')[0]
    voice_url=f'https://huggingface.co/rhasspy/piper-voices/resolve/main/{language}/{locale}/{speaker}/{quality}/'
    voice=MODELS/(VOICE_NAME+'.onnx')
    for name in (VOICE_NAME+'.onnx',VOICE_NAME+'.onnx.json'):
        download(voice_url+name,MODELS/name)
    download(voice_url+'MODEL_CARD',MODELS/(VOICE_NAME+'-MODEL_CARD.txt'))
    run_setup([PIPER,'--help'])
else:
    voice=MODELS/(VOICE_NAME+'.onnx')

WORK = Path('/kaggle/working/ai-compose-onecell-jobs') / time.strftime('%Y%m%d-%H%M%S')
WORK.mkdir(parents=True,exist_ok=True)
environment = {
    'PATH':str(PIPER.parent)+os.pathsep+os.environ.get('PATH',''),
    'HF_HOME':str(MODELS/'huggingface'),
    'AI_COMPOSE_HF_MODEL':TEXT_MODEL,'AI_COMPOSE_HF_LOAD_4BIT':'true',
    'AI_COMPOSE_HF_MAX_NEW_TOKENS':'3000','AI_COMPOSE_VISION_HF_MODEL':VISION_MODEL,
    'MEDIA_CLIP_WHISPER_MODEL':'small','MEDIA_CLIP_WHISPER_DEVICE':ASR_DEVICE,
    'MEDIA_CLIP_WHISPER_COMPUTE_TYPE':'float16' if ASR_DEVICE=='cuda' else 'int8',
    'MEDIA_CLIP_WHISPER_CPU_THREADS':'4','MEDIA_CLIP_WHISPER_CACHE_DIR':str(MODELS/'whisper'),
    'AI_COMPOSE_TTS_MODEL':str(voice),'AI_COMPOSE_MUSIC_PATH':MUSIC_PATH,
    'LD_LIBRARY_PATH':os.pathsep.join(gpu['libraries']+[os.environ.get('LD_LIBRARY_PATH','')]),
    'AI_COMPOSE_USE_NVENC':'false','AI_COMPOSE_VIDEO_CODEC':'libx264',
    'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'4',
}
payload = {
    'brief':BRIEF,'preset':'viral_short','target_seconds':TARGET_SECONDS,
    'output_width':OUTPUT_SIZE[0],'output_height':OUTPUT_SIZE[1], 'output_count':1,
    'captions_enabled':True,'fetch_assets':True,'llm_required':True,
    'refinement_iterations':REFINEMENT_ROUNDS,'local_asset_library':LOCAL_ASSET_LIBRARY or None,
    'asset_sources':{'local_library':True,'wikimedia':True,'pexels':True,'pixabay':True},
    'editorial_policy':{'source_audio':'preserve','supplementary_visuals':'require',
        'camera_motion':'dynamic','motion_graphics':'hero'},
    'editor':{'enabled':True,'narration':NARRATION,'narration_budget_seconds':12,
        'caption_style':'highlight','theme':'editorial','progress_bar':True,
        'music':'library' if MUSIC_PATH else 'procedural','procedural_sfx':True,
        'transition':'dissolve','min_graphics':2,'min_visual_layers':1,
        'required_features':REQUIRED_FEATURES,'visual_review_required':True,
        'strict_requirements':True,'publish_outputs':False},
    'sfx':{'mode':'adaptive','volume':0.20}, 'music_volume':0.12,
}
config={'source_path':str(source),'env_file':PRIVATE_ENV_FILE,'environment':environment,
    'work_dir':str(WORK),'payload':payload,'timeout_seconds':TIMEOUT_SECONDS,'keep_api':KEEP_LOCAL_API}
config_path=WORK/'private-config.json';config_path.write_text(json.dumps(config,indent=2));config_path.chmod(0o600)
# Run outside the notebook kernel: no stale imports and no foreground Uvicorn cell.
from IPython.display import Audio, FileLink, Video, display
import numpy as np

def beep():
    """Notify at readiness, completion, and failure; browser autoplay rules apply."""
    t=np.linspace(0,.5,4000,endpoint=False)
    display(Audio(np.sin(2*np.pi*440*t),rate=8000,autoplay=True))

beep()
print('Running the integrated editor. Evidence, drafts, and logs are saved under:',WORK,flush=True)
runner_log=WORK/'driver.log'
with runner_log.open('w') as stream:
    process=subprocess.Popen([str(PYTHON),'-u',str(CODE/'ai_compose_kaggle.py'),str(config_path)],
        cwd=CODE,stdout=stream,stderr=subprocess.STDOUT,env=setup_environment)
    offset=0
    try:
        while process.poll() is None:
            time.sleep(2)
            with runner_log.open() as current:
                current.seek(offset);message=current.read();offset=current.tell()
            if message: print(message,end='',flush=True)
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=25)
        except subprocess.TimeoutExpired: process.kill();process.wait()
        beep()
        raise
with runner_log.open() as current:
    current.seek(offset);print(current.read(),end='')
beep()
if process.returncode:
    display(FileLink(str(runner_log)))
    display(FileLink(str(WORK/'server.log')))
    for diagnostic in WORK.glob('*-output.zip'): display(FileLink(str(diagnostic)))
    raise RuntimeError('No passing final render was produced. Review the saved diagnostics; requirements were not silently disabled.')
result=json.loads((WORK/'latest-output.json').read_text())
for output in result['outputs']:
    display(Video(output['local_path'],embed=True,width=360))
    display(FileLink(output['local_path']))
display(FileLink(result['archive']))
display(FileLink(result['receipt']))
if 'api_connection_file' in result:
    print('Loopback API remains available. Connection details:',result['api_connection_file'])
print('Save a Kaggle notebook version with outputs to retain these files after the session ends.')
